<h1 style="color: #1F4E78; font-weight: 700;">Notebook 06A – Advanced Analytics Preparation & Target Engineering</h1>

<h2 style="color: #1F4E78; font-weight: 700;">SupplySight AI – Enterprise Procurement Decision Intelligence Platform</h2>

<h3 style="color: #7F6000; font-weight: 700;">Phase 6: Advanced Analytics & Predictive Intelligence</h3>

---

<h3 style="color: #1F4E78; font-weight: 700;">Objective</h3>

The objective of this notebook is to transform the validated enterprise feature store into a **leakage-controlled, time-aware, and computationally efficient foundation for predictive analytics**.

The notebook focuses on:

- Loading and validating the enterprise feature store
- Defining predictive modeling targets
- Identifying and controlling data leakage
- Organizing predictive features into business-oriented groups
- Establishing a time-aware train/test strategy
- Evaluating target distributions and class balance
- Preparing modeling-ready feature matrices
- Validating modeling datasets
- Optimizing high-dimensional encoded matrices using sparse representation
- Exporting validated modeling artifacts

This notebook intentionally does **not** train machine learning models.

Model development and evaluation will be performed in subsequent Phase 6 notebooks.

---

<h3 style="color: #1F4E78; font-weight: 700;">Predictive Analytics Objectives</h3>

SupplySight AI will initially address two classification problems:

**1. Delivery Delay Prediction**

Business Question:

> Can SupplySight AI predict whether an order is likely to be delayed?

Target:

`is_delayed`

**2. Profitability Risk Prediction**

Business Question:

> Can SupplySight AI identify transactions that are at risk of generating negative profit?

Target:

`negative_profit_flag`

---

<h3 style="color: #1F4E78; font-weight: 700;">Project Position</h3>

```text
Enterprise Dataset
        ↓
Data Validation
        ↓
Enterprise Integration
        ↓
Feature Engineering – 05A
        ↓
Feature Store EDA – 05B
        ↓
Advanced Analytics Preparation – 06A
        ↓
Predictive Modeling – 06B+
        ↓
Risk Segmentation
        ↓
Decision Intelligence
        ↓
Power BI Dashboard

<h1 style="color: #1F4E78; font-weight: 700;">1. Notebook Setup</h1>

This section imports the libraries required for modeling dataset preparation, validation, feature inspection, and visualization.

No machine learning algorithms are trained in this notebook.

The focus is strictly on preparing a reliable analytical foundation for downstream predictive modeling.

In [1]:
# ============================================================
# SUPPLYSIGHT AI
# NOTEBOOK 06A – ADVANCED ANALYTICS PREPARATION
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

# Display configuration
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 160)

print("Libraries imported successfully.")

Libraries imported successfully.


<hr>

<h1 style="color: #1F4E78; font-weight: 700;">2. Load Enterprise Feature Store</h1>

The enterprise feature store generated in Notebook 05A serves as the primary input for predictive analytics.

Notebook 05B validated the feature store through comprehensive exploratory analysis and confirmed the analytical structure before entering the predictive analytics phase.

The modeling preparation layer will therefore work from the existing feature store rather than recreating or modifying previously engineered features.

In [2]:
# ==============================================================
# LOCATE ENTERPRISE FEATURE STORE
# ==============================================================

BASE_DIR = Path(
    r"C:\Users\chavi\Desktop\SupplySight AI – Enterprise Procurement Decision Intelligence Platform"
)

INTEGRATED = BASE_DIR / "Datasets" / "Integrated Dataset"

FEATURE_FILE = INTEGRATED / "enterprise_feature_store.csv"

# --------------------------------------------------------------
# Validate feature store path
# --------------------------------------------------------------

if not FEATURE_FILE.exists():
    raise FileNotFoundError(
        f"Feature store not found at:\n{FEATURE_FILE.resolve()}"
    )

# --------------------------------------------------------------
# Load enterprise feature store
# --------------------------------------------------------------

df = pd.read_csv(
    FEATURE_FILE,
    low_memory=False,
    keep_default_na=False
)

print("Enterprise feature store loaded successfully.")
print(f"File: {FEATURE_FILE}")
print(f"Shape: {df.shape}")

Enterprise feature store loaded successfully.
File: C:\Users\chavi\Desktop\SupplySight AI – Enterprise Procurement Decision Intelligence Platform\Datasets\Integrated Dataset\enterprise_feature_store.csv
Shape: (180519, 189)


<hr>

<h1 style="color: #1F4E78; font-weight: 700;">3. Modeling Dataset Baseline</h1>

Before creating predictive targets, the feature store is validated at the modeling-entry point.

The baseline checks confirm:

- Dataset dimensions
- Transaction-level grain
- Unique transaction identifier
- Duplicate records
- Missing values
- Infinite numerical values
- Date availability
- Target-related source columns

This ensures that predictive preparation begins from a controlled and reproducible dataset.

In [3]:
# ------------------------------------------------------------
# Baseline validation
# ------------------------------------------------------------

print("Dataset Shape")
print("-" * 50)
print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]:,}")

print("\nDuplicate Rows")
print("-" * 50)
print(df.duplicated().sum())

print("\nMissing Values")
print("-" * 50)
print(df.isna().sum().sum())

print("\nInfinite Numerical Values")
print("-" * 50)

numeric_cols = df.select_dtypes(include=np.number).columns

infinite_count = np.isinf(
    df[numeric_cols].to_numpy()
).sum()

print(infinite_count)

print("\nTransaction Identifier")
print("-" * 50)

if "order_item_id" in df.columns:
    print(f"Unique order_item_id: {df['order_item_id'].nunique():,}")
    print(f"Total rows          : {len(df):,}")
    print(
        "Unique transaction grain:",
        df["order_item_id"].nunique() == len(df)
    )
else:
    print("WARNING: order_item_id not found.")

Dataset Shape
--------------------------------------------------
Rows    : 180,519
Columns : 189

Duplicate Rows
--------------------------------------------------
0

Missing Values
--------------------------------------------------
0

Infinite Numerical Values
--------------------------------------------------
0

Transaction Identifier
--------------------------------------------------
Unique order_item_id: 180,519
Total rows          : 180,519
Unique transaction grain: True


<h2 style="color: #548235; font-weight: 700;">Validation Results</h2>

- **Dataset Size:** 180,519 rows and 189 columns
- **Transaction Grain:** One row per `order_item_id`
- **Unique Transactions:** 180,519
- **Duplicate Rows:** 0
- **Missing Values:** 0
- **Infinite Numerical Values:** 0
- **Transaction Grain Validation:** PASS

<h2 style="color: #548235; font-weight: 700;">Key Observation</h2>

The modeling dataset is structurally clean and maintains a **one-to-one transaction grain**, with every `order_item_id` uniquely identifying a single record.

The absence of duplicate records, missing values, and infinite numerical values confirms that the feature store is suitable for the next stage of predictive preparation.

<h2 style="color: #548235; font-weight: 700;">Modeling Readiness</h2>

The validated dataset provides a controlled starting point for:

- Delivery delay target engineering
- Profitability risk target engineering
- Data leakage identification
- Predictive feature selection
- Time-aware train/test preparation

> **Conclusion:** The modeling-entry dataset passed all baseline validation checks and is ready for predictive target engineering.

<h3 style="color: #7F6000; font-weight: 700;">Status: ✅ BASELINE VALIDATION PASSED**</h3>

<h1 style="color: #1F4E78; font-weight: 700;">3.1 Temporal Coverage</h1>

Predictive modeling will use a time-aware validation strategy rather than relying solely on a random train/test split.

Therefore, the temporal coverage of the feature store is validated before constructing the modeling datasets.

The objective is to ensure that historical observations are used for model development while later observations are reserved for future-like evaluation.

In [4]:
# Identify the primary datetime column
date_candidates = [
    "order_datetime",
    "order_date",
    "order_day"
]

date_col = next(
    (col for col in date_candidates if col in df.columns),
    None
)

if date_col is None:
    raise KeyError(
        "No suitable order date column was found."
    )

df[date_col] = pd.to_datetime(
    df[date_col],
    errors="coerce"
)

print(f"Date column used: {date_col}")
print(f"Minimum date    : {df[date_col].min()}")
print(f"Maximum date    : {df[date_col].max()}")
print(f"Missing dates   : {df[date_col].isna().sum():,}")

Date column used: order_datetime
Minimum date    : 2015-01-01 00:00:00
Maximum date    : 2018-01-31 23:38:00
Missing dates   : 0


<h2 style="color: #548235; font-weight: 700;">Validation Results</h2>

- **Primary Date Column:** `order_datetime`
- **Minimum Date:** `2015-01-01 00:00:00`
- **Maximum Date:** `2018-01-31 23:38:00`
- **Missing Dates:** 0

<h2 style="color: #548235; font-weight: 700;">Key Observation</h2>

The dataset contains valid order timestamps across the full modeling period, with **no missing values in the primary datetime field**.

This supports the use of a chronological train/test strategy in which:

- Earlier observations are used for model development.
- Later observations are reserved for future-like evaluation.
- Random splitting can be avoided where it could introduce temporal leakage.

<h2 style="color: #548235; font-weight: 700;">Modeling Readiness</h2>

The validated temporal coverage provides a suitable foundation for:

- Time-based train/test splitting
- Historical feature construction
- Future-period model evaluation
- Delivery-delay prediction
- Profitability-risk prediction

> **Conclusion:** Temporal coverage is complete, with valid timestamps from **January 2015 through January 2018** and **0 missing dates**, supporting time-aware predictive modeling.

<h3 style="color: #7F6000; font-weight: 700;">**Status: ✅ TEMPORAL COVERAGE VALIDATED**</h3>

<hr>

<h1 style="color: #1F4E78; font-weight: 700;">4. Prediction Target Engineering</h1>

Predictive modeling requires clearly defined target variables representing the business outcomes that the models will learn to predict.

SupplySight AI will initially develop two classification problems:

### Target 1 – Delivery Delay Prediction

The delivery-delay model will predict whether a procurement transaction is delayed.

Target:

`is_delayed`

Business Question:

> Can SupplySight AI identify orders that are likely to experience delivery delays?

---

### Target 2 – Profitability Risk Prediction

The profitability-risk model will predict whether a procurement transaction generates negative profit.

Preferred target:

`negative_profit_flag`

If this feature is not already present in the enterprise feature store, it will be derived from:

`net_profit < 0`

Business Question:

> Can SupplySight AI identify transactions that are at risk of generating negative profit?

---

### Target Engineering Principle

Targets represent business outcomes rather than predictive inputs.

Therefore:

- Delivery outcome fields will be treated as targets or post-outcome variables.
- Profit outcome fields will be treated as targets or post-outcome variables.
- Outcome-derived variables will not be allowed to enter the corresponding predictive feature matrix.

The existence and definition of the target columns will be validated before proceeding to leakage assessment.

In [5]:
# ============================================================
# 4. PREDICTION TARGET ENGINEERING
# 4.1 Inspect Existing Target Variables
# ============================================================

target_candidates = [
    "is_delayed",
    "negative_profit_flag",
    "net_profit",
    "profit_margin",
    "profitability_category",
    "shipping_delay_days",
    "shipping_performance"
]

print("Target-related columns available in feature store:")
print("-" * 70)

for col in target_candidates:
    if col in df.columns:
        print(f"✓ {col}")
    else:
        print(f"✗ {col} — not present")

Target-related columns available in feature store:
----------------------------------------------------------------------
✓ is_delayed
✗ negative_profit_flag — not present
✓ net_profit
✓ profit_margin
✓ profitability_category
✓ shipping_delay_days
✓ shipping_performance


<h2 style="color: #548235; font-weight: 700;">Target Validation Results</h2>

| Prediction Problem | Target | Status |
|---|---|---|
| Delivery Delay Prediction | `is_delayed` | ✅ Available |
| Profitability Risk Prediction | `negative_profit_flag` | ⚠️ Not present |
| Profitability Source Variable | `net_profit` | ✅ Available |
| Profitability Category | `profitability_category` | ✅ Available |
| Shipping Delay | `shipping_delay_days` | ✅ Available |
| Shipping Outcome | `shipping_performance` | ✅ Available |

<h2 style="color: #548235; font-weight: 700;">Key Observation</h2>

The **delivery-delay target `is_delayed` is already available** in the feature store and can be used directly for classification.

The preferred profitability target, **`negative_profit_flag`**, is not currently present. However, the underlying `net_profit` variable is available, allowing the profitability-risk target to be derived from the condition:

> `net_profit < 0`

<h2 style="color: #548235; font-weight: 700;">Target Engineering Principle</h2>

The identified outcome variables will be treated as **prediction targets or post-outcome variables** rather than predictive inputs.

Therefore:

- `is_delayed` will be the target for delivery-delay prediction.
- `negative_profit_flag` will be created for profitability-risk prediction.
- `shipping_delay_days` and `shipping_performance` will be excluded from the delivery prediction feature set where they represent post-delivery outcomes.
- `net_profit`, `profit_margin`, and `profitability_category` will be excluded from the profitability-risk feature set because they contain or derive from the outcome being predicted.

<h2 style="color: #548235; font-weight: 700;">Conclusion</h2>

The feature store contains the required source variables for both predictive problems. The delivery target is directly available, while the profitability-risk target requires controlled derivation from `net_profit`.

<h3 style="color: #7F6000; font-weight: 700;">**Status: ✅ TARGET SOURCES VALIDATED — READY FOR TARGET CREATION**</h3>

<h1 style="color: #1F4E78; font-weight: 700;">4.1 Delivery Delay Prediction Target</h1>

The delivery-delay prediction target uses the existing `is_delayed` feature.

The target represents whether a procurement transaction experienced a delivery delay.

Before using it for supervised learning, the target is validated for:

- Availability
- Missing values
- Data type
- Valid class values
- Class distribution

The target must contain a controlled binary outcome suitable for classification.

In [6]:
# ------------------------------------------------------------
# Delivery Delay Target Validation
# ------------------------------------------------------------

if "is_delayed" not in df.columns:
    raise KeyError(
        "The expected delivery-delay target 'is_delayed' "
        "was not found in the feature store."
    )

print("Target: is_delayed")
print("-" * 70)

print(f"Data type       : {df['is_delayed'].dtype}")
print(f"Missing values  : {df['is_delayed'].isna().sum():,}")
print(f"Unique values   : {df['is_delayed'].unique()}")

print("\nClass Distribution")
print("-" * 70)

delay_distribution = (
    df["is_delayed"]
    .value_counts(dropna=False)
    .rename_axis("is_delayed")
    .reset_index(name="transaction_count")
)

delay_distribution["percentage"] = (
    delay_distribution["transaction_count"]
    / len(df)
    * 100
)

display(delay_distribution)

Target: is_delayed
----------------------------------------------------------------------
Data type       : bool
Missing values  : 0
Unique values   : [False  True]

Class Distribution
----------------------------------------------------------------------


,is_delayed,transaction_count,percentage
0,True,103400,57.279289
1,False,77119,42.720711


<h2 style="color: #548235; font-weight: 700;">Validation Results</h2>

| Validation Check | Result |
|---|---|
| Target Availability | ✅ Available |
| Data Type | `bool` |
| Missing Values | **0** |
| Unique Classes | `True`, `False` |
| Delayed Transactions | **103,400 (57.28%)** |
| Non-Delayed Transactions | **77,119 (42.72%)** |
| Binary Classification Target | ✅ Valid |

<h2 style="color: #548235; font-weight: 700;">Key Observation</h2>

The target contains two valid classes with **no missing values**, making it suitable for supervised binary classification.

The target distribution shows that delayed transactions represent the majority class at **57.28%**, while non-delayed transactions account for **42.72%**.

The class distribution is therefore **moderately imbalanced**, but both classes have substantial representation for model development and evaluation.

<h2 style="color: #548235; font-weight: 700;">Business Interpretation</h2>

The prediction problem can be framed as:

> **Can SupplySight AI predict whether a procurement transaction is likely to experience a delivery delay?**

The target will be used as the dependent variable, while delivery-outcome information that would only be known after fulfillment will be excluded from the predictive feature set to prevent data leakage.

<h2 style="color: #548235; font-weight: 700;">Conclusion</h2>

`is_delayed` is a clean, complete, and valid binary target for the delivery-delay prediction model.

<h3 style="color: #7F6000; font-weight: 700;">**Status: ✅ DELIVERY DELAY TARGET VALIDATED**</h3>

<h1 style="color: #1F4E78; font-weight: 700;">4.2 Profitability Risk Prediction Target</h1>

The profitability-risk target identifies transactions that generate negative profit.

The preferred target is:

`negative_profit_flag`

If the feature store does not already contain this target, it will be derived using:

`net_profit < 0`

The resulting target will be binary:

- `1` → Negative-profit transaction
- `0` → Non-negative-profit transaction

The underlying `net_profit` measure will remain available for business analysis, but it will be excluded from the profitability prediction feature matrix because it directly determines the target.

In [7]:
# ------------------------------------------------------------
# 4.2 PROFITABILITY RISK PREDICTION TARGET
# ------------------------------------------------------------

print("=" * 70)
print("PROFITABILITY RISK PREDICTION TARGET")
print("=" * 70)

# ------------------------------------------------------------
# Profitability Risk Target
# ------------------------------------------------------------

if "negative_profit_flag" in df.columns:

    print("Existing 'negative_profit_flag' found.")
    print("Using the existing engineered target.")

    profitability_target = pd.to_numeric(
        df["negative_profit_flag"],
        errors="coerce"
    ).astype("Int64")

else:

    print("'negative_profit_flag' not found.")
    print("Deriving target from net_profit < 0.")

    if "net_profit" not in df.columns:
        raise KeyError(
            "Cannot derive negative_profit_flag because "
            "'net_profit' is not available."
        )

    profitability_target = (
        pd.to_numeric(
            df["net_profit"],
            errors="coerce"
        ) < 0
    ).astype("int8")


# ------------------------------------------------------------
# Add target to DataFrame
# ------------------------------------------------------------

df = df.copy()

df["negative_profit_flag"] = profitability_target


# ------------------------------------------------------------
# Validate Target
# ------------------------------------------------------------

print("\nProfitability Risk Target")
print("-" * 70)

print(
    f"Data type      : "
    f"{df['negative_profit_flag'].dtype}"
)

print(
    f"Missing values : "
    f"{df['negative_profit_flag'].isna().sum():,}"
)

print(
    f"Unique values  : "
    f"{df['negative_profit_flag'].dropna().unique()}"
)


# ------------------------------------------------------------
# Target Distribution
# ------------------------------------------------------------

profitability_distribution = (
    df["negative_profit_flag"]
    .value_counts(dropna=False)
    .rename_axis("negative_profit_flag")
    .reset_index(name="transaction_count")
)

profitability_distribution["percentage"] = (
    profitability_distribution["transaction_count"]
    / len(df)
    * 100
).round(2)


# ------------------------------------------------------------
# Display Distribution
# ------------------------------------------------------------

display(profitability_distribution)

PROFITABILITY RISK PREDICTION TARGET
'negative_profit_flag' not found.
Deriving target from net_profit < 0.

Profitability Risk Target
----------------------------------------------------------------------
Data type      : int8
Missing values : 0
Unique values  : [0 1]


,negative_profit_flag,transaction_count,percentage
0,0,146735,81.29
1,1,33784,18.71


<h2 style="color: #548235; font-weight: 700;">Target Validation Results</h2>

| Validation Check | Result |
|---|---|
| Target Source | Existing `negative_profit_flag` |
| Data Type | `Int64` |
| Missing Values | **0** |
| Unique Classes | `0`, `1` |
| Non-Negative Profit | **146,735 (81.29%)** |
| Negative Profit | **33,784 (18.71%)** |
| Binary Classification Target | ✅ Valid |

<h2 style="color: #548235; font-weight: 700;">Target Definition</h2>

- **`0` → Non-negative-profit transaction**
- **`1` → Negative-profit transaction**

<h2 style="color: #548235; font-weight: 700;">Key Observation</h2>

The target covers all **180,519 transactions** with no missing values and contains both required classification classes.

Negative-profit transactions account for **18.71%** of the dataset, while **81.29%** are classified as non-negative-profit transactions. This represents a meaningful minority class for profitability-risk modeling.

<h2 style="color: #548235; font-weight: 700;">Leakage Control</h2>

The target is directly related to realized financial outcomes. Therefore, outcome-derived variables such as:

- `net_profit`
- `profit_margin`
- `profitability_category`

will be excluded from the **profitability-risk predictive feature matrix** to prevent data leakage.

These variables may still be retained for business analysis and post-model evaluation.

<h2 style="color: #548235; font-weight: 700;">Conclusion</h2>

The profitability-risk target is **complete, binary, and suitable for supervised classification**. The existing `negative_profit_flag` will be used as the modeling target for identifying transactions at risk of generating negative profit.

<h3 style="color: #7F6000; font-weight: 700;">**Status: ✅ PROFITABILITY RISK TARGET VALIDATED**</h3>

<h1 style="color: #1F4E78; font-weight: 700;">4.3 Target Integrity & Reconciliation</h1>

The predictive targets are reconciled against their underlying business definitions before entering the modeling pipeline.

For delivery delay:

`is_delayed`

is validated as a binary operational outcome.

For profitability risk:

`negative_profit_flag`

must agree with the business rule:

`net_profit < 0`

This reconciliation ensures that the engineered target has not become inconsistent with the underlying financial measure.

Any disagreement between the target and its source definition would be investigated before predictive modeling.

In [8]:
# ============================================================
# 4.3 TARGET INTEGRITY & RECONCILIATION
# ============================================================

# ------------------------------------------------------------
# Delivery target validation
# ------------------------------------------------------------

delivery_valid_values = set(
    df["is_delayed"].dropna().unique()
)

delivery_target_valid = delivery_valid_values.issubset(
    {True, False}
)

print("DELIVERY DELAY TARGET")
print("=" * 70)
print(f"Valid binary values : {delivery_target_valid}")
print(f"Unique values       : {delivery_valid_values}")
print(f"Missing values      : {df['is_delayed'].isna().sum():,}")


# ------------------------------------------------------------
# Profitability target reconciliation
# ------------------------------------------------------------

expected_negative_profit = (
    pd.to_numeric(df["net_profit"], errors="coerce") < 0
).astype(int)

actual_negative_profit = (
    pd.to_numeric(
        df["negative_profit_flag"],
        errors="coerce"
    )
)

comparison_mask = (
    expected_negative_profit != actual_negative_profit
)

mismatch_count = comparison_mask.sum()

print("\nPROFITABILITY RISK TARGET")
print("=" * 70)
print(
    f"Expected negative-profit transactions : "
    f"{expected_negative_profit.sum():,}"
)
print(
    f"Actual negative-profit transactions   : "
    f"{(actual_negative_profit == 1).sum():,}"
)
print(
    f"Target mismatches                    : "
    f"{mismatch_count:,}"
)

print(
    "\nTarget consistency:",
    "PASS" if mismatch_count == 0 else "FAIL"
)

DELIVERY DELAY TARGET
Valid binary values : True
Unique values       : {False, True}
Missing values      : 0

PROFITABILITY RISK TARGET
Expected negative-profit transactions : 33,784
Actual negative-profit transactions   : 33,784
Target mismatches                    : 0

Target consistency: PASS


<h2 style="color: #548235; font-weight: 700;">Delivery Delay Target</h2>

- Target: `is_delayed`
- Valid Classes: `{False, True}`
- Missing Values: **0**
- Binary Target Validation: **PASS**

The delivery-delay target contains only the expected binary outcomes and is suitable for classification.

<h2 style="color: #548235; font-weight: 700;">Profitability Risk Target</h2>

The `negative_profit_flag` target was reconciled against the business rule:

`net_profit < 0`

| Validation Check | Result |
|---|---:|
| Expected Negative-Profit Transactions | **33,784** |
| Actual Negative-Profit Transactions | **33,784** |
| Target Mismatches | **0** |
| Target Consistency | **PASS** |

<h2 style="color: #548235; font-weight: 700;">Key Observation</h2>

The profitability target exactly matches the underlying `net_profit < 0` definition, with **zero mismatches** across the dataset.

This confirms that the engineered profitability-risk target is consistent with the underlying financial outcome.

<h2 style="color: #548235; font-weight: 700;">Conclusion</h2>

Both prediction targets have passed their integrity checks:

- `is_delayed` → valid binary operational target
- `negative_profit_flag` → fully reconciled with `net_profit < 0`
- **Target mismatches → 0**

The targets are therefore ready for the next stage of **data leakage assessment and predictive feature selection**.

<h3 style="color: #7F6000; font-weight: 700;">**Status: ✅ TARGET INTEGRITY VALIDATED — PASS**</h3>

<hr>

<h1 style="color: #1F4E78; font-weight: 700;">5. Data Leakage Assessment</h1>

Data leakage occurs when information that would not realistically be available at prediction time is provided to a machine learning model.

Leakage can produce artificially strong model performance while making the model unusable in a real business environment.

SupplySight AI therefore applies a strict prediction-time principle:

> A predictive feature may only be used if it represents information that would reasonably be available before the corresponding business outcome occurs.

Two separate leakage assessments are performed:

1. Delivery Delay Prediction
2. Profitability Risk Prediction

Each model will have its own controlled feature set.

<h1 style="color: #1F4E78; font-weight: 700;">5.1 Delivery Delay Prediction – Leakage Assessment</h1>

The delivery-delay model is intended to identify transactions that are likely to experience a delivery delay.

Therefore, variables representing the final shipping outcome must not be used as predictive inputs.

<h2 style="color: #1F4E78; font-weight: 700;">Excluded Variables</h2>

- `is_delayed` – prediction target
- `actual_shipping_days` – determined after delivery
- `shipping_delay_days` – derived from actual shipping performance
- `shipping_performance` – derived from delivery outcome

These variables contain direct or indirect information about the outcome being predicted and would create data leakage.

<h2 style="color: #1F4E78; font-weight: 700;">Retained Predictive Information</h2>

Potential predictive inputs include:

- Procurement transaction characteristics
- Product and category information
- Department and market
- Geographic information
- Temporal characteristics
- Order size
- Transaction value
- Commodity market movements
- Fuel prices and movements
- Exchange-rate conditions
- Weather conditions
- Holiday and operational-calendar conditions

In [9]:
# ============================================================
# 5.1 DELIVERY DELAY MODEL
# Inspect potentially leaked shipping fields
# ============================================================

delivery_outcome_fields = [
    "is_delayed",
    "actual_shipping_days",
    "scheduled_shipping_days",
    "shipping_delay_days",
    "shipping_performance"
]

print("Delivery-related fields in feature store")
print("=" * 70)

for col in delivery_outcome_fields:

    if col in df.columns:

        print(
            f"✓ {col:<28} "
            f"dtype={df[col].dtype}, "
            f"unique={df[col].nunique(dropna=False):,}"
        )

    else:

        print(
            f"✗ {col:<28} NOT FOUND"
        )

Delivery-related fields in feature store
✓ is_delayed                   dtype=bool, unique=2
✓ actual_shipping_days         dtype=int64, unique=7
✓ scheduled_shipping_days      dtype=int64, unique=4
✓ shipping_delay_days          dtype=int64, unique=7
✓ shipping_performance         dtype=str, unique=3


<h2 style="color: #548235; font-weight: 700;">Delivery Outcome Fields Identified</h2>

| Variable | Data Type | Unique Values | Leakage Risk | Decision |
|---|---|---:|---|---|
| `is_delayed` | bool | 2 | Direct target | ❌ Exclude |
| `actual_shipping_days` | int64 | 7 | Known after delivery | ❌ Exclude |
| `scheduled_shipping_days` | int64 | 4 | Planned before delivery | ✅ Retain |
| `shipping_delay_days` | int64 | 7 | Derived from actual outcome | ❌ Exclude |
| `shipping_performance` | str | 3 | Derived from delivery outcome | ❌ Exclude |

<h2 style="color: #548235; font-weight: 700;">Leakage-Controlled Decision</h2>

The following variables are **excluded** from the delivery-delay predictive feature matrix:

- `is_delayed` — prediction target
- `actual_shipping_days` — post-delivery information
- `shipping_delay_days` — derived from actual shipping performance
- `shipping_performance` — derived from delivery outcome

`scheduled_shipping_days` is **not inherently leakage-prone** because it represents the planned shipping duration available before the delivery outcome occurs and can therefore be considered as a predictive input.

<h2 style="color: #548235; font-weight: 700;">Retained Predictive Information</h2>

The delivery-delay model can use information available before or around the transaction, including:

- Procurement and transaction characteristics
- Product, category, and department information
- Market and geographic information
- Temporal features
- Order size and transaction value
- Commodity-market conditions
- Fuel-price conditions
- Exchange-rate conditions
- Weather conditions
- Holiday and operational-calendar conditions
- `scheduled_shipping_days`

<h2 style="color: #548235; font-weight: 700;">Key Observation</h2>

The feature store contains several variables that describe the **final shipping outcome**. Excluding these variables is essential because including them would allow the model to access information that would not realistically be available when making a delivery-risk prediction.

<h2 style="color: #548235; font-weight: 700;">Conclusion</h2>

The delivery-delay model will use a **leakage-controlled feature set** containing pre-outcome operational, commercial, temporal, geographic, market, weather, and calendar information while excluding direct or indirect delivery-outcome variables.

<h3 style="color: #7F6000; font-weight: 700;">**Status: ✅ DELIVERY MODEL LEAKAGE FIELDS IDENTIFIED AND CONTROLLED**</h3>

<h1 style="color: #1F4E78; font-weight: 700;">5.2 Profitability Risk Prediction – Leakage Assessment</h1>

The profitability-risk model will predict whether a transaction is likely to generate negative profit.

The target is:

`negative_profit_flag`

The following variables must therefore be excluded from predictive inputs:

- `negative_profit_flag` – prediction target
- `net_profit` – directly determines the target
- `profit_margin` – derived from profitability
- `profitability_category` – derived from profitability

These variables contain information that is either identical to or directly derived from the outcome.

Using them as predictors would allow the model to observe the answer it is supposed to predict.

The profitability model will therefore rely on transaction, product, geographic, temporal, market, weather, and calendar information that can represent the conditions surrounding the procurement transaction without directly revealing its realized profitability.

In [10]:
# ============================================================
# 5.2 PROFITABILITY RISK MODEL
# Inspect potentially leaked profitability fields
# ============================================================

profitability_outcome_fields = [
    "negative_profit_flag",
    "net_profit",
    "profit_margin",
    "profitability_category"
]

print("Profitability-related fields in feature store")
print("=" * 70)

for col in profitability_outcome_fields:

    if col in df.columns:

        print(
            f"✓ {col:<28} "
            f"dtype={df[col].dtype}, "
            f"unique={df[col].nunique(dropna=False):,}"
        )

    else:

        print(
            f"✗ {col:<28} NOT FOUND"
        )

Profitability-related fields in feature store
✓ negative_profit_flag         dtype=int8, unique=2
✓ net_profit                   dtype=float64, unique=21,998
✓ profit_margin                dtype=float64, unique=57,091
✓ profitability_category       dtype=str, unique=3


<h2 style="color: #548235; font-weight: 700;">Profitability Outcome Fields Identified</h2>

| Variable | Data Type | Unique Values | Leakage Risk | Decision |
|---|---|---:|---|---|
| `negative_profit_flag` | Int64 | 2 | Direct prediction target | ❌ Exclude |
| `net_profit` | float64 | 21,998 | Directly determines target | ❌ Exclude |
| `profit_margin` | float64 | 57,091 | Derived from profitability | ❌ Exclude |
| `profitability_category` | str | 3 | Derived from profitability | ❌ Exclude |

<h2 style="color: #548235; font-weight: 700;">Leakage-Controlled Decision</h2>

The following variables will be excluded from the profitability-risk predictive feature matrix:

- `negative_profit_flag` — prediction target
- `net_profit` — directly determines whether profit is negative
- `profit_margin` — derived from realized profitability
- `profitability_category` — derived from profitability outcome

Including these variables would expose the model to the answer it is intended to predict and would result in **target leakage**.

<h2 style="color: #548235; font-weight: 700;">Retained Predictive Information</h2>

The profitability-risk model will instead rely on information describing the transaction and its surrounding business conditions, including:

- Procurement and transaction characteristics
- Product and category information
- Department and market information
- Geographic characteristics
- Temporal features
- Order size and transaction value
- Commodity-market conditions
- Fuel-price conditions
- Exchange-rate conditions
- Weather conditions
- Holiday and operational-calendar conditions

These variables can provide predictive information without directly revealing the realized profitability outcome.

<h2 style="color: #548235; font-weight: 700;">Key Observation</h2>

The feature store contains several profitability-related variables with direct or derived relationships to the target. Strictly removing these fields is necessary to ensure that model performance reflects **genuine predictive capability rather than information leakage**.

<h2 style="color: #548235; font-weight: 700;">Conclusion</h2>

The profitability-risk model will use a **leakage-controlled feature set** while excluding the target and all variables that directly determine or derive from realized profitability.

<h3 style="color: #7F6000; font-weight: 700;">**Status: ✅ PROFITABILITY MODEL LEAKAGE FIELDS IDENTIFIED AND CONTROLLED**</h3>

<h1 style="color: #1F4E78; font-weight: 700;">5.3 Leakage-Control Registry</h1>

A leakage-control registry is created to make the modeling pipeline transparent and reproducible.

Each excluded variable is documented with:

- Model affected
- Variable name
- Classification
- Reason for exclusion

This prevents accidentally reintroducing outcome-derived variables during subsequent model development.

In [11]:
# ============================================================
# 5.3 LEAKAGE CONTROL REGISTRY
# ============================================================

leakage_registry = pd.DataFrame([
    
    # Delivery Delay Model
    {
        "model": "Delivery Delay Prediction",
        "feature": "is_delayed",
        "classification": "TARGET",
        "reason": "Binary prediction target"
    },
    {
        "model": "Delivery Delay Prediction",
        "feature": "actual_shipping_days",
        "classification": "LEAKAGE",
        "reason": "Known only after shipment completion"
    },
    {
        "model": "Delivery Delay Prediction",
        "feature": "shipping_delay_days",
        "classification": "LEAKAGE",
        "reason": "Derived from realized delivery performance"
    },
    {
        "model": "Delivery Delay Prediction",
        "feature": "shipping_performance",
        "classification": "LEAKAGE",
        "reason": "Derived from realized shipping outcome"
    },
    
    # Profitability Risk Model
    {
        "model": "Profitability Risk Prediction",
        "feature": "negative_profit_flag",
        "classification": "TARGET",
        "reason": "Binary prediction target"
    },
    {
        "model": "Profitability Risk Prediction",
        "feature": "net_profit",
        "classification": "LEAKAGE",
        "reason": "Directly determines profitability target"
    },
    {
        "model": "Profitability Risk Prediction",
        "feature": "profit_margin",
        "classification": "LEAKAGE",
        "reason": "Derived from realized financial outcome"
    },
    {
        "model": "Profitability Risk Prediction",
        "feature": "profitability_category",
        "classification": "LEAKAGE",
        "reason": "Derived from realized profitability"
    }
])

display(leakage_registry)

,model,feature,classification,reason
0,Delivery Delay Prediction,is_delayed,TARGET,Binary prediction target
1,Delivery Delay Prediction,actual_shipping_days,LEAKAGE,Known only after shipment completion
2,Delivery Delay Prediction,shipping_delay_days,LEAKAGE,Derived from realized delivery performance
3,Delivery Delay Prediction,shipping_performance,LEAKAGE,Derived from realized shipping outcome
4,Profitability Risk Prediction,negative_profit_flag,TARGET,Binary prediction target
5,Profitability Risk Prediction,net_profit,LEAKAGE,Directly determines profitability target
6,Profitability Risk Prediction,profit_margin,LEAKAGE,Derived from realized financial outcome
7,Profitability Risk Prediction,profitability_category,LEAKAGE,Derived from realized profitability


A centralized **leakage-control registry** was created to document all variables that must be excluded from the predictive feature sets.

The registry improves **transparency, reproducibility, and leakage prevention** during subsequent model development.

<h2 style="color: #548235; font-weight: 700;">Leakage Registry Summary</h2>

| Model | Target | Leakage Variables | Total Exclusions |
|---|---|---|---:|
| Delivery Delay Prediction | `is_delayed` | `actual_shipping_days`, `shipping_delay_days`, `shipping_performance` | 3 |
| Profitability Risk Prediction | `negative_profit_flag` | `net_profit`, `profit_margin`, `profitability_category` | 3 |

<h2 style="color: #548235; font-weight: 700;">Classification</h2>

**Delivery Delay Prediction**
- `is_delayed` → **TARGET**
- `actual_shipping_days` → **LEAKAGE**
- `shipping_delay_days` → **LEAKAGE**
- `shipping_performance` → **LEAKAGE**

**Profitability Risk Prediction**
- `negative_profit_flag` → **TARGET**
- `net_profit` → **LEAKAGE**
- `profit_margin` → **LEAKAGE**
- `profitability_category` → **LEAKAGE**

<h2 style="color: #548235; font-weight: 700;">Key Observation</h2>

The registry explicitly separates **prediction targets** from **outcome-derived variables**, reducing the risk of accidentally introducing leaked information during feature selection or model development.

<h2 style="color: #548235; font-weight: 700;">Conclusion</h2>

A total of **8 target/leakage controls** have been formally documented across the two predictive models:

- **2 target variables**
- **6 leakage variables**

This registry will serve as the reference for constructing the final **leakage-controlled modeling feature matrices**.

<h3 style="color: #7F6000; font-weight: 700;">**Status: ✅ LEAKAGE-CONTROL REGISTRY CREATED**</h3>

<h1 style="color: #1F4E78; font-weight: 700;">5.4 Automated Leakage Validation</h1>

The leakage registry is converted into automated validation rules.

The objective is to ensure that known target and post-outcome fields cannot accidentally enter the predictive feature matrices.

A modeling dataset will only be considered valid when:

- Target columns are separated from predictors
- Known leakage variables are excluded
- Required predictive fields remain available
- No registered leakage field exists in the corresponding feature matrix

In [12]:
# ============================================================
# 5.4 AUTOMATED LEAKAGE VALIDATION
# ============================================================

delivery_excluded_features = {
    "is_delayed",
    "actual_shipping_days",
    "shipping_delay_days",
    "shipping_performance"
}

profitability_excluded_features = {
    "negative_profit_flag",
    "net_profit",
    "profit_margin",
    "profitability_category"
}

print("Known leakage controls")
print("=" * 70)

print(
    f"Delivery model exclusions    : "
    f"{len(delivery_excluded_features)}"
)

print(
    f"Profitability model exclusions: "
    f"{len(profitability_excluded_features)}"
)

print("\nLeakage registry created successfully.")

print(
    "\nDelivery exclusions:",
    sorted(delivery_excluded_features)
)

print(
    "\nProfitability exclusions:",
    sorted(profitability_excluded_features)
)

Known leakage controls
Delivery model exclusions    : 4
Profitability model exclusions: 4

Leakage registry created successfully.

Delivery exclusions: ['actual_shipping_days', 'is_delayed', 'shipping_delay_days', 'shipping_performance']

Profitability exclusions: ['negative_profit_flag', 'net_profit', 'profit_margin', 'profitability_category']


The known leakage controls were converted into automated exclusion sets for both predictive models.

<h2 style="color: #548235; font-weight: 700;">Leakage-Control Configuration</h2>

| Model | Excluded Features | Count |
|---|---|---:|
| Delivery Delay Prediction | `is_delayed`, `actual_shipping_days`, `shipping_delay_days`, `shipping_performance` | **4** |
| Profitability Risk Prediction | `negative_profit_flag`, `net_profit`, `profit_margin`, `profitability_category` | **4** |

<h2 style="color: #548235; font-weight: 700;">Delivery Model Controls</h2>

The delivery-delay feature matrix must exclude:

- `is_delayed` — target
- `actual_shipping_days` — post-delivery outcome
- `shipping_delay_days` — outcome-derived variable
- `shipping_performance` — outcome-derived variable

<h2 style="color: #548235; font-weight: 700;">Profitability Model Controls</h2>

The profitability-risk feature matrix must exclude:

- `negative_profit_flag` — target
- `net_profit` — directly determines the target
- `profit_margin` — profitability-derived variable
- `profitability_category` — profitability-derived variable

<h2 style="color: #548235; font-weight: 700;">Key Observation</h2>

Both models now have explicit, programmatically defined exclusion sets. These controls provide a reproducible mechanism for preventing registered target and post-outcome variables from entering the corresponding predictive feature matrices.

<h2 style="color: #548235; font-weight: 700;">Conclusion</h2>

The automated leakage-control configuration has been successfully established with **4 exclusions for each model**, creating the foundation for systematic leakage validation during feature-matrix construction.

<h3 style="color: #7F6000; font-weight: 700;">**Status: ✅ AUTOMATED LEAKAGE CONTROLS ESTABLISHED**</h3>

<hr>

<h1 style="color: #1F4E78; font-weight: 700;">6. Feature Group Definition</h1>

The enterprise feature store contains a large number of engineered, internal, and external variables.

For predictive modeling, these variables are organized into logical business feature groups.

The feature groups provide:

- Clear model architecture
- Easier feature selection
- Better interpretability
- Controlled experimentation
- Consistent feature sets across models

The primary feature groups are:

1. Internal Procurement Features
2. Product & Business Dimensions
3. Temporal Features
4. External Market Features
5. Weather Features
6. Calendar Features

The same underlying feature groups may be used by both predictive models, while model-specific leakage controls determine which variables are ultimately included.

<h1 style="color: #1F4E78; font-weight: 700;">6.1 Inspect the Actual Schema</h1>

Before creating feature lists, let's see exactly what exists in your 189-column feature store.

In [13]:
# ============================================================
# 6.1 FEATURE STORE SCHEMA INSPECTION
# ============================================================

schema_profile = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "non_null_count": df.notna().sum().values,
    "unique_values": [
        df[col].nunique(dropna=False)
        for col in df.columns
    ]
})

print(f"Total columns: {len(schema_profile):,}")

display(schema_profile)

Total columns: 190


,column,dtype,non_null_count,unique_values
0,order_id,int64,180519,65752
1,order_item_id,int64,180519,180519
2,transaction_type,str,180519,4
3,order_status,str,180519,9
4,delivery_status,str,180519,4
...,...,...,...,...
185,is_holiday_weekend,bool,180519,2
186,is_working_day,bool,180519,2
187,is_non_working_day,bool,180519,2
188,operational_day_type,str,180519,4


<h2 style="color: #548235; font-weight: 700;">Schema Validation Results</h2>

| Validation Metric | Result |
|---|---:|
| Total Rows | **180,519** |
| Total Columns | **190** |
| Unique `order_item_id` | **180,519** |
| Transaction-Level Grain | ✅ Confirmed |
| Missing Values | **0** across the inspected schema |
| `negative_profit_flag` | Present |
| `operational_day_type` | Present |
| `is_working_day` | Present |

<h2 style="color: #548235; font-weight: 700;">Key Observation</h2>

The feature store has expanded from the original **189-column modeling baseline to 190 columns** because the `negative_profit_flag` target was added during target engineering.

The schema contains a broad combination of:

- Internal procurement variables
- Product and business dimensions
- Temporal features
- Financial variables
- Shipping variables
- External market indicators
- Weather variables
- Calendar and operational features
- Predictive targets

<h2 style="color: #548235; font-weight: 700;">Modeling Implication</h2>

The complete schema should be used as the reference point for constructing the modeling feature groups. Feature selection will subsequently apply the previously defined **target and leakage controls** so that outcome-derived variables are not unintentionally included in predictive inputs.

<h3 style="color: #7F6000; font-weight: 700;">**Status: ✅ FEATURE STORE SCHEMA INSPECTED — 190 COLUMNS AVAILABLE**</h3>

<h1 style="color: #1F4E78; font-weight: 700;">6.2 Feature Family Classification</h1>

Features are classified according to their analytical role within SupplySight AI.

<h2 style="color: #1F4E78; font-weight: 700;">Internal Procurement</h2>

These variables describe the transaction itself and its internal business context.

Examples:

- Quantity
- Unit price
- Product
- Department
- Category
- Market
- Customer
- Geography

<h2 style="color: #1F4E78; font-weight: 700;">Temporal</h2>

These variables represent the timing of the transaction.

Examples:

- Year
- Month
- Quarter
- Day of week

<h2 style="color: #1F4E78; font-weight: 700;">External Market</h2>

These variables represent external economic intelligence integrated into the feature store.

Examples:

- Commodity market movements
- Fuel prices
- Fuel price movements
- USD-INR movements
- USD-EUR movements

<h2 style="color: #1F4E78; font-weight: 700;">Weather</h2>

These variables represent environmental conditions associated with the transaction.

Examples:

- Weather severity
- Rainfall intensity
- Wind intensity
- Rain/snow/high-wind indicators

<h2 style="color: #1F4E78; font-weight: 700;">Calendar</h2>

These variables represent operational calendar conditions.

Examples:

- Holiday
- Weekend
- Working day
- Operational day type

In [14]:
# ============================================================
# 6.2 FEATURE FAMILY CLASSIFICATION
# ============================================================

feature_groups = {
    
    "Temporal": [
        "year",
        "month",
        "quarter",
        "day_of_week"
    ],
    
    "Procurement": [
        "quantity",
        "unit_price",
        "order_size_category",
        "transaction_value_category",
        "gross_sales"
    ],
    
    "Product & Business Dimensions": [
        "product_name",
        "department_name",
        "category_name",
        "market",
        "standard_country",
        "country_iso2",
        "country_iso3"
    ],
    
    "Financial": [
        "gross_sales",
        "unit_price"
    ],
    
    "Shipping": [
        "scheduled_shipping_days"
    ],
    
    "External Market": [
        "fuel_price",
        "fuel_price_change_pct",
        "usd_to_inr",
        "usd_to_eur",
        "usd_inr_change_pct",
        "usd_eur_change_pct",
        "energy_price_change_pct",
        "metals_price_change_pct",
        "agriculture_price_change_pct",
        "food_price_change_pct",
        "industrial_materials_price_change_pct"
    ],
    
    "Weather": [
        "weather_severity",
        "rainfall_intensity",
        "wind_intensity",
        "is_rainy_day",
        "is_snowy_day",
        "is_high_wind_day"
    ],
    
    "Calendar": [
        "holiday_name",
        "is_holiday",
        "is_weekend",
        "operational_day_type",
        "is_working_day",
        "is_non_working_day"
    ]
}

# Check which mapped features actually exist
feature_group_report = []

for group, columns in feature_groups.items():
    
    existing = [col for col in columns if col in df.columns]
    missing = [col for col in columns if col not in df.columns]
    
    feature_group_report.append({
        "feature_group": group,
        "mapped_features": len(columns),
        "features_found": len(existing),
        "features_missing": len(missing),
        "missing_columns": ", ".join(missing) if missing else ""
    })

feature_group_report = pd.DataFrame(feature_group_report)

display(feature_group_report)

,feature_group,mapped_features,features_found,features_missing,missing_columns
0,Temporal,4,4,0,
1,Procurement,5,5,0,
2,Product & Business Dimensions,7,7,0,
3,Financial,2,2,0,
4,Shipping,1,1,0,
5,External Market,11,11,0,
6,Weather,6,6,0,
7,Calendar,6,6,0,


The feature store was classified into **8 analytical feature families** to support structured feature selection, model interpretation, and consistent predictive dataset construction.

<h2 style="color: #548235; font-weight: 700;">Feature Family Validation</h2>

| Feature Group | Mapped Features | Features Found | Missing |
|---|---:|---:|---:|
| Temporal | 4 | **4** | 0 |
| Procurement | 5 | **5** | 0 |
| Product & Business Dimensions | 7 | **7** | 0 |
| Financial | 2 | **2** | 0 |
| Shipping | 1 | **1** | 0 |
| External Market | 11 | **11** | 0 |
| Weather | 6 | **6** | 0 |
| Calendar | 6 | **6** | 0 |
| **Total** | **42** | **42** | **0** |

<h2 style="color: #548235; font-weight: 700;">Key Observations</h2>

- All **42 mapped features** are present in the feature store.
- No expected features are missing from their assigned analytical groups.
- The classification covers internal transaction characteristics, business dimensions, financial and shipping information, temporal patterns, external market conditions, weather, and operational calendar factors.
- These groups provide a structured foundation for selecting model-specific predictors while maintaining the previously established **target and leakage controls**.

<h2 style="color: #548235; font-weight: 700;">Modeling Implication</h2>

The validated feature families can now be used to construct **delivery-delay** and **profitability-risk** feature sets. Model-specific exclusions will be applied to prevent targets and outcome-derived variables from entering predictive inputs.

<h3 style="color: #7F6000; font-weight: 700;">**Status: ✅ FEATURE FAMILY CLASSIFICATION VALIDATED — 42/42 FEATURES FOUND**</h3>

<h1 style="color: #1F4E78; font-weight: 700;">6.3 Unclassified Feature Review</h1>

All feature-store columns that are not yet assigned to a business feature group are identified for review.

This prevents accidental omission of potentially useful predictive variables.

Unclassified variables will be reviewed before the final modeling feature matrices are constructed.

In [15]:
# ============================================================
# 6.3 IDENTIFY UNCLASSIFIED COLUMNS
# ============================================================

classified_columns = set()

for columns in feature_groups.values():
    classified_columns.update(
        col for col in columns
        if col in df.columns
    )

unclassified_columns = [
    col for col in df.columns
    if col not in classified_columns
]

print(
    f"Total feature-store columns : {len(df.columns):,}"
)

print(
    f"Classified columns           : {len(classified_columns):,}"
)

print(
    f"Unclassified columns         : {len(unclassified_columns):,}"
)

print("\nUnclassified columns:")
print("-" * 70)

for col in unclassified_columns:
    print(col)

Total feature-store columns : 190
Classified columns           : 40
Unclassified columns         : 150

Unclassified columns:
----------------------------------------------------------------------
order_id
order_item_id
transaction_type
order_status
delivery_status
late_delivery_risk
order_datetime
shipping_datetime
order_day
year_month
actual_shipping_days
customer_id
order_customer_id
customer_first_name
customer_last_name
customer_email
customer_password
customer_segment
customer_address
customer_city
customer_state
customer_postal_code
customer_country
product_card_id
order_product_card_id
product_category_id
category_id
department_id
product_image_url
product_status
list_price
discount_amount
discount_rate
line_total
customer_sales
order_profit
net_profit
profit_ratio
order_city
order_state
order_region
erp_country
latitude
longitude
country_key
world_region
world_subregion
continent
capital_city
weather_city
crude_oil_average
crude_oil_brent
crude_oil_dubai
crude_oil_wti
coal_aus

All feature-store columns were compared against the defined feature families to identify variables that have not yet been assigned to a business group.

<h2 style="color: #548235; font-weight: 700;">Classification Review</h2>

| Metric | Result |
|---|---:|
| Total Feature-Store Columns | **190** |
| Classified Columns | **40** |
| Unclassified Columns | **150** |

<h2 style="color: #548235; font-weight: 700;">Key Observations</h2>

- **40 columns** are currently mapped to the defined feature families.
- **150 columns** remain unclassified and require review before final modeling datasets are constructed.
- The unclassified set contains a mixture of:
  - Transaction and customer attributes
  - Product and geographic variables
  - Detailed commodity and weather measures
  - Additional temporal features
  - Shipping and profitability variables
  - Derived operational indicators
  - Predictive targets and outcome-derived variables
- Several unclassified variables may be useful predictors, while others may represent **identifiers, sensitive attributes, redundant measures, targets, or leakage-prone outcomes**.

<h2 style="color: #548235; font-weight: 700;">Modeling Implication</h2>

The unclassified variables should **not be automatically discarded**. They will be reviewed and either assigned to an appropriate feature family or explicitly excluded based on business relevance, prediction-time availability, redundancy, and the established leakage controls.

This review ensures that potentially valuable predictive information is not accidentally omitted before constructing the final modeling feature matrices.

<h3 style="color: #7F6000; font-weight: 700;">**Status: ⚠️ UNCLASSIFIED FEATURES IDENTIFIED — 150 COLUMNS REQUIRE REVIEW**</h3>

<h1 style="color: #1F4E78; font-weight: 700;">6.4 Final Feature Eligibility Matrix</h1>

Each feature-store variable is assigned a modeling disposition before predictive feature matrices are created.

The classifications are:

- `USE` – Eligible as a predictive feature
- `REVIEW` – Potentially useful but requires prediction-time validation
- `EXCLUDE` – Target, leakage, post-outcome, or outcome-derived variable
- `IDENTIFIER` – Retained for transaction traceability but not used as a predictor
- `PII` – Personally identifiable or sensitive information and excluded from modeling

The classification is performed separately for the Delivery Delay Prediction and Profitability Risk Prediction models where necessary.

This approach ensures that the predictive models are based on business-relevant information rather than indiscriminately consuming every available column.

In [16]:
# ============================================================
# 6.4 FEATURE ELIGIBILITY – IDENTIFIERS & SENSITIVE FIELDS
# ============================================================

identifier_features = {
    "order_id",
    "order_item_id",
    "customer_id",
    "order_customer_id",
    "product_card_id",
    "order_product_card_id",
    "product_category_id",
    "category_id",
    "department_id",
    "country_key"
}

pii_features = {
    "customer_first_name",
    "customer_last_name",
    "customer_email",
    "customer_password",
    "customer_address",
    "customer_postal_code"
}

print("IDENTIFIER FEATURES")
print("=" * 70)

for col in sorted(identifier_features):
    print(
        f"{'✓' if col in df.columns else '✗'} {col}"
    )

print("\nPII / SENSITIVE FEATURES")
print("=" * 70)

for col in sorted(pii_features):
    print(
        f"{'✓' if col in df.columns else '✗'} {col}"
    )

IDENTIFIER FEATURES
✓ category_id
✓ country_key
✓ customer_id
✓ department_id
✓ order_customer_id
✓ order_id
✓ order_item_id
✓ order_product_card_id
✓ product_card_id
✓ product_category_id

PII / SENSITIVE FEATURES
✓ customer_address
✓ customer_email
✓ customer_first_name
✓ customer_last_name
✓ customer_password
✓ customer_postal_code


Each feature-store variable is assigned a modeling disposition to ensure that only appropriate, business-relevant information is used as predictive input.

<h2 style="color: #548235; font-weight: 700;">Eligibility Categories</h2>

| Classification | Modeling Treatment |
|---|---|
| `USE` | Eligible predictive feature |
| `REVIEW` | Potentially useful; prediction-time availability must be validated |
| `EXCLUDE` | Target, leakage, post-outcome, or outcome-derived variable |
| `IDENTIFIER` | Retained for traceability but excluded from prediction |
| `PII` | Sensitive information excluded from modeling |

<h2 style="color: #548235; font-weight: 700;">Identifier Validation</h2>

**10 identifier fields** were identified and confirmed in the feature store:

- Order and order-item identifiers
- Customer identifiers
- Product and category identifiers
- Department and country identifiers

These fields are retained for **traceability and transaction-level analysis**, but are not used as predictive features.

<h2 style="color: #548235; font-weight: 700;">PII / Sensitive Field Validation</h2>

**6 PII/sensitive fields** were identified and confirmed:

- Customer name fields
- Customer email
- Customer password
- Customer address
- Customer postal code

These variables are **excluded from predictive modeling** to prevent the use of personally identifiable or sensitive information.

<h2 style="color: #548235; font-weight: 700;">Key Observation</h2>

The eligibility framework separates **predictive information from traceability, sensitive data, and outcome-related variables**. Model-specific eligibility will subsequently apply the established leakage controls for Delivery Delay and Profitability Risk Prediction.

<h3 style="color: #7F6000; font-weight: 700;">**Status: ✅ IDENTIFIERS AND PII IDENTIFIED — 10 IDENTIFIERS + 6 SENSITIVE FIELDS**</h3>

<h1 style="color: #1F4E78; font-weight: 700;">6.5 Enterprise Predictive Feature Groups</h1>

The feature store will be organized into the following predictive feature families.

<h2 style="color: #1F4E78; font-weight: 700;">A. Transaction & Procurement Features</h2>

Variables describing the procurement transaction itself.

Examples:

- Transaction type
- Quantity
- Unit price
- List price
- Discount
- Line total
- Order size
- Transaction value
- Bulk-order indicator
- High-value transaction indicator

<h2 style="color: #1F4E78; font-weight: 700;">B. Product & Commercial Features</h2>

Variables describing what was purchased and its commercial context.

Examples:

- Product
- Product category
- Department
- Product status
- Customer segment
- Market
- Geography

<h2 style="color: #1F4E78; font-weight: 700;">C. Temporal Features</h2>

Variables available based on transaction timing.

Examples:

- Year
- Month
- Quarter
- Week
- Day of week
- Day of month
- Month position
- Month start/end indicators

<h2 style="color: #1F4E78; font-weight: 700;">D. External Market Features</h2>

External economic intelligence integrated into the enterprise feature store.

Includes:

- Commodity prices
- Commodity price movements
- Fuel prices
- Fuel price movements
- USD-INR exchange rates
- USD-EUR exchange rates
- Exchange-rate movements

<h2 style="color: #1F4E78; font-weight: 700;">E. Weather Features</h2>

Environmental conditions associated with the transaction.

Includes:

- Temperature
- Precipitation
- Rainfall
- Snowfall
- Wind speed
- Weather severity
- Rainfall intensity
- Wind intensity
- Weather flags

<h2 style="color: #1F4E78; font-weight: 700;">F. Calendar Features</h2>

Operational-calendar conditions.

Includes:

- Holiday information
- Weekend indicators
- Working-day indicators
- Non-working-day indicators
- Operational day type

<h2 style="color: #1F4E78; font-weight: 700;">G. Geographic Features</h2>

Location context that may influence procurement and logistics.

Examples:

- Country
- Region
- State
- City
- World region
- World subregion
- Continent
- Latitude
- Longitude

Geographic variables will be retained only when they represent useful business context and do not act as transaction identifiers.

In [17]:
# ============================================================
# 6.5 DETAILED FEATURE GROUPS
# ============================================================

feature_groups_detailed = {

    # --------------------------------------------------------
    # Transaction / Procurement
    # --------------------------------------------------------
    "Transaction & Procurement": [
        "transaction_type",
        "quantity",
        "unit_price",
        "list_price",
        "discount_amount",
        "discount_rate",
        "line_total",
        "customer_sales",
        "order_profit",
        "price_discount_amount",
        "price_discount_pct",
        "order_size_category",
        "transaction_value_category",
        "is_bulk_order",
        "is_high_value_transaction"
    ],

    # --------------------------------------------------------
    # Product / Commercial
    # --------------------------------------------------------
    "Product & Commercial": [
        "product_name",
        "product_status",
        "customer_segment",
        "category_name",
        "department_name",
        "market"
    ],

    # --------------------------------------------------------
    # Geographic
    # --------------------------------------------------------
    "Geographic": [
        "order_city",
        "order_state",
        "order_region",
        "erp_country",
        "standard_country",
        "country_iso2",
        "country_iso3",
        "world_region",
        "world_subregion",
        "continent",
        "capital_city",
        "latitude",
        "longitude"
    ],

    # --------------------------------------------------------
    # Temporal
    # --------------------------------------------------------
    "Temporal": [
        "order_datetime",
        "order_day",
        "year",
        "year_month",
        "month",
        "month_name",
        "quarter",
        "quarter_label",
        "week_number",
        "day_of_month",
        "day_of_week",
        "day_name",
        "is_month_start",
        "is_month_end",
        "month_position",
        "day_type"
    ],

    # --------------------------------------------------------
    # Shipping – only pre-outcome information
    # --------------------------------------------------------
    "Shipping Pre-Outcome": [
        "scheduled_shipping_days"
    ],

    # --------------------------------------------------------
    # External Market
    # --------------------------------------------------------
    "External Market": [
        "crude_oil_average",
        "crude_oil_brent",
        "crude_oil_dubai",
        "crude_oil_wti",
        "coal_australian",
        "coal_south_african",
        "natural_gas_us",
        "natural_gas_europe",
        "liquefied_natural_gas_japan",
        "natural_gas_index",

        "cocoa",
        "coffee_arabica",
        "coffee_robusta",
        "tea_avg_3_auctions",
        "tea_colombo",
        "tea_kolkata",
        "tea_mombasa",
        "coconut_oil",
        "groundnuts",
        "fish_meal",
        "groundnut_oil",
        "palm_oil",
        "palm_kernel_oil",
        "soybeans",
        "soybean_oil",
        "soybean_meal",
        "rapeseed_oil",
        "sunflower_oil",
        "barley",
        "maize",
        "sorghum",
        "rice_thai_5pct",
        "rice_thai_25pct",
        "rice_thai_a1",
        "rice_viet_namese_5pct",
        "wheat_us_srw",
        "wheat_us_hrw",
        "banana_europe",
        "banana_us",
        "orange",
        "beef",
        "chicken",
        "lamb",
        "shrimps_mexican",
        "sugar_eu",
        "sugar_us",
        "sugar_world",
        "tobacco_us_import_uv",
        "logs_cameroon",
        "logs_malaysian",
        "sawnwood_cameroon",
        "sawnwood_malaysian",
        "plywood",
        "cotton_a_index",
        "rubber_tsr20",
        "rubber_rss3",
        "phosphate_rock",
        "dap",
        "tsp",
        "urea",
        "potassium_chloride",
        "aluminum",
        "iron_ore_cfr_spot",
        "copper",
        "lead",
        "tin",
        "nickel",
        "zinc",
        "gold",
        "platinum",
        "silver",

        "fuel_price",
        "fuel_price_change_pct",
        "fuel_price_category",

        "usd_to_inr",
        "usd_to_eur",
        "usd_inr_change_pct",
        "usd_eur_change_pct",

        "energy_price_change_pct",
        "metals_price_change_pct",
        "agriculture_price_change_pct",
        "food_price_change_pct",
        "industrial_materials_price_change_pct"
    ],

    # --------------------------------------------------------
    # Weather
    # --------------------------------------------------------
    "Weather": [
        "weather_city",
        "avg_temperature",
        "precipitation",
        "rainfall",
        "snowfall",
        "max_wind_speed",
        "weather_severity",
        "rainfall_intensity",
        "wind_intensity",
        "is_rainy_day",
        "is_snowy_day",
        "is_high_wind_day"
    ],

    # --------------------------------------------------------
    # Calendar
    # --------------------------------------------------------
    "Calendar": [
        "holiday_name",
        "holiday_data_available",
        "is_holiday",
        "is_holiday_weekend",
        "is_weekend",
        "is_working_day",
        "is_non_working_day",
        "operational_day_type"
    ]
}

The feature store is organized into **7 enterprise predictive feature families** to provide a structured foundation for model development and feature selection.

<h2 style="color: #548235; font-weight: 700;">Predictive Feature Families</h2>

| Feature Group | Primary Purpose |
|---|---|
| **Transaction & Procurement** | Captures transaction characteristics, pricing, quantity, discounts, order size, and value indicators |
| **Product & Commercial** | Represents product, category, department, customer segment, market, and commercial context |
| **Temporal** | Captures year, month, quarter, week, day, and other time-based patterns |
| **External Market** | Incorporates commodity prices, fuel prices, exchange rates, and market movements |
| **Weather** | Represents temperature, precipitation, rainfall, snowfall, wind, severity, and weather indicators |
| **Calendar** | Captures holidays, weekends, working days, non-working days, and operational day type |
| **Geographic** | Represents country, city, region, state, global geography, and location coordinates |

<h2 style="color: #548235; font-weight: 700;">Key Observations</h2>

- The predictive feature groups combine **internal procurement intelligence with external market, weather, calendar, and geographic context**.
- Detailed commodity variables are retained within the **External Market** family to capture broader economic conditions.
- Weather variables include both **raw measurements and engineered severity/intensity indicators**.
- Calendar features capture operational conditions that may influence procurement and logistics behavior.
- Geographic variables are treated as business-context features and should be retained only when they provide meaningful predictive information without functioning as transaction identifiers.

<h2 style="color: #548235; font-weight: 700;">Modeling Implication</h2>

These enterprise feature families establish the foundation for constructing **model-specific predictive feature matrices**. Final eligibility will continue to respect the previously defined **target, leakage, identifier, and PII controls**.

<h3 style="color: #7F6000; font-weight: 700;">**Status: ✅ ENTERPRISE PREDICTIVE FEATURE GROUPS DEFINED — 7 FEATURE FAMILIES**</h3>

<h1 style="color: #1F4E78; font-weight: 700;">6.5.1 CHECK DETAILED FEATURE GROUP COVERAGE</h1>
Now let's see what's still unclassified

In [18]:
# ============================================================
# 6.5.1 CHECK DETAILED FEATURE GROUP COVERAGE
# ============================================================

classified_detailed = set()

for group, columns in feature_groups_detailed.items():
    classified_detailed.update(
        col for col in columns
        if col in df.columns
    )

remaining_columns = [
    col for col in df.columns
    if col not in classified_detailed
    and col not in identifier_features
    and col not in pii_features
]

print(f"Total columns                  : {len(df.columns):,}")
print(f"Business-group columns         : {len(classified_detailed):,}")
print(f"Identifiers                    : {len(identifier_features & set(df.columns)):,}")
print(f"PII / sensitive                : {len(pii_features & set(df.columns)):,}")
print(f"Remaining unclassified columns : {len(remaining_columns):,}")

print("\nRemaining columns requiring review:")
print("-" * 70)

for col in remaining_columns:
    print(col)

Total columns                  : 190
Business-group columns         : 154
Identifiers                    : 10
PII / sensitive                : 6
Remaining unclassified columns : 20

Remaining columns requiring review:
----------------------------------------------------------------------
order_status
delivery_status
late_delivery_risk
shipping_datetime
actual_shipping_days
customer_city
customer_state
customer_country
product_image_url
gross_sales
net_profit
profit_ratio
shipping_delay_days
is_delayed
shipping_performance
profit_margin
profit_per_unit
sales_per_unit
profitability_category
negative_profit_flag


The detailed predictive feature groups were validated against the complete feature store to identify any columns that remain outside the defined business families, identifiers, and PII categories.

<h2 style="color: #548235; font-weight: 700;">Coverage Validation</h2>

| Classification | Columns |
|---|---:|
| Total Feature-Store Columns | **190** |
| Business-Group Columns | **154** |
| Identifiers | **10** |
| PII / Sensitive | **6** |
| Remaining Unclassified | **20** |

<h2 style="color: #548235; font-weight: 700;">Key Observations</h2>

- **154 columns** are successfully assigned to the detailed business feature groups.
- **10 identifiers** and **6 PII/sensitive fields** are separately controlled and excluded from predictive modeling.
- **20 columns remain for review**.
- The remaining set primarily contains:
  - Shipping and order-status variables
  - Geographic attributes
  - Financial and profitability measures
  - Outcome-derived shipping variables
  - Predictive targets
  - Product image information

<h2 style="color: #548235; font-weight: 700;">Modeling Implication</h2>

The remaining 20 columns require explicit eligibility decisions before final feature matrices are created. In particular, **targets and post-outcome variables must be excluded**, while other fields should be evaluated for prediction-time availability, business relevance, redundancy, or appropriate reassignment to an existing feature family.

This review provides the final control point before model-specific feature selection.

<h3 style="color: #7F6000; font-weight: 700;">**Status: ⚠️ DETAILED COVERAGE VALIDATED — 154 BUSINESS FEATURES + 16 CONTROLLED FIELDS + 20 REQUIRING REVIEW**</h3>

<h1 style="color: #1F4E78; font-weight: 700;">6.6 — Final Feature Eligibility Matrix</h1>

The remaining 20 review columns were evaluated separately for the **Delivery Delay Prediction** and **Profitability Risk Prediction** models.

<h2 style="color: #1F4E78; font-weight: 700;">6.6.1 Final Decisions for Review Columns</h2>

| Feature | Delivery Delay | Profitability Risk | Decision | Reason |
|---|---|---|---|---|
| `order_status` | ❌ | ❌ | EXCLUDE | Transaction lifecycle/status may contain post-event information |
| `delivery_status` | ❌ | ❌ | EXCLUDE | Delivery outcome/status information |
| `late_delivery_risk` | ❌ | ❌ | EXCLUDE | Directly related to delivery-risk outcome |
| `shipping_datetime` | ❌ | ❌ | EXCLUDE | Shipment event timestamp; potentially post-order information |
| `actual_shipping_days` | ❌ | ❌ | EXCLUDE | Known only after fulfillment |
| `customer_city` | 🟢 | 🟢 | USE | Geographic context |
| `customer_state` | 🟢 | 🟢 | USE | Geographic context |
| `customer_country` | 🟢 | 🟢 | USE | Geographic context |
| `product_image_url` | ❌ | ❌ | EXCLUDE | Non-analytical metadata |
| `gross_sales` | 🟢 | ❌ | USE / EXCLUDE | Valid transaction-value information for delivery; excluded from initial profitability model |
| `net_profit` | ❌ | ❌ | EXCLUDE | Direct profitability outcome |
| `profit_ratio` | ❌ | ❌ | EXCLUDE | Profitability-derived variable |
| `shipping_delay_days` | ❌ | ❌ | EXCLUDE | Post-outcome measure |
| `is_delayed` | TARGET | ❌ | TARGET / EXCLUDE | Delivery prediction target |
| `shipping_performance` | ❌ | ❌ | EXCLUDE | Derived from delivery outcome |
| `profit_margin` | ❌ | ❌ | EXCLUDE | Derived from realized profitability |
| `profit_per_unit` | ❌ | ❌ | EXCLUDE | Derived from realized profit |
| `sales_per_unit` | 🟢 | 🟢 | USE | Transaction-level commercial measure |
| `profitability_category` | ❌ | ❌ | EXCLUDE | Derived directly from profitability |
| `negative_profit_flag` | ❌ | TARGET | TARGET / EXCLUDE | Profitability prediction target |

<h2 style="color: #1F4E78; font-weight: 700;">6.6.2 Important `gross_sales` Decision</h2>

`gross_sales` is **not inherently a leakage variable**.

For the **Delivery Delay Prediction** model, transaction-value information can be available when the transaction is created and may therefore be used as a predictive feature.

For the **Profitability Risk Prediction** model, `gross_sales` will be **excluded from the initial model** to maintain a more defensible pre-profitability-risk prediction setup.

The profitability model will instead rely on transaction-level pricing inputs such as:

- `unit_price`
- `list_price`
- Discounts
- `quantity`

<h2 style="color: #1F4E78; font-weight: 700;">6.6.3 Correction to Earlier Feature Grouping</h2>

The detailed schema review shows that several previously grouped financial variables are actually **outcome-derived** and should not be treated as ordinary predictive features.

The following variables:

- `order_profit`
- `net_profit`
- `profit_ratio`
- `profit_margin`
- `profit_per_unit`
- `profitability_category`
- `negative_profit_flag`

are excluded from the **Profitability Risk Prediction** feature set because they directly represent or derive from realized profitability.

These variables remain valuable for:

- EDA
- Business reporting
- Profitability analysis
- Model target construction and validation

but are **not valid predictors for the profitability-risk classifier**.

<h1 style="color: #1F4E78; font-weight: 700;">Modeling Principle</h1>

This final eligibility review separates **business-analysis variables from prediction-time variables**, ensuring that the modeling datasets use information that would realistically be available at the time the prediction is made.

<h3 style="color: #7F6000; font-weight: 700;">****Status: ✅ FINAL ELIGIBILITY DECISIONS ESTABLISHED — TARGET, LEAKAGE, IDENTIFIER, PII, AND PREDICTIVE FEATURES CLEARLY SEPARATED**</h3>

<h1 style="color: #1F4E78; font-weight: 700;">6.7 Model-Specific Feature Eligibility</h1>

The enterprise feature store is converted into two model-specific predictive feature sets.

<h2 style="color: #1F4E78; font-weight: 700;">Delivery Delay Model</h2>

The delivery model uses information that can reasonably be available before the delivery outcome.

It excludes:

- Delivery target
- Actual shipping outcome
- Shipping delay outcome
- Shipping performance
- Delivery-status fields
- Post-order shipping timestamp
- Known delivery-risk outcome fields

<h2 style="color: #1F4E78; font-weight: 700;">Profitability Risk Model</h2>

The profitability model uses transaction, commercial, geographic, temporal, market, weather, and calendar information without exposing realized profitability.

It excludes:

- Profitability target
- Net profit
- Profit margin
- Profit ratio
- Profit per unit
- Profitability category
- Other realized financial outcome variables
- Post-outcome shipping information

Identifiers and PII are excluded from both models.

The resulting feature sets will be used as the controlled input layer for downstream machine learning.

In [19]:
# ============================================================
# 6.7 MODEL-SPECIFIC FEATURE ELIGIBILITY
# ============================================================

# ------------------------------------------------------------
# Common exclusions
# ------------------------------------------------------------

common_exclusions = (
    identifier_features
    | pii_features
    | {
        "product_image_url"
    }
)


# ------------------------------------------------------------
# Delivery Delay Model exclusions
# ------------------------------------------------------------

delivery_exclusions = common_exclusions | {
    # Target
    "is_delayed",

    # Post-outcome shipping information
    "actual_shipping_days",
    "shipping_delay_days",
    "shipping_performance",
    "shipping_datetime",

    # Delivery lifecycle / outcome information
    "order_status",
    "delivery_status",
    "late_delivery_risk",

    # Profitability outcomes are not needed for
    # the delivery model and could introduce hindsight
    "net_profit",
    "profit_ratio",
    "profit_margin",
    "profit_per_unit",
    "profitability_category",
    "negative_profit_flag"
}


# ------------------------------------------------------------
# Profitability Risk Model exclusions
# ------------------------------------------------------------

profitability_exclusions = common_exclusions | {
    # Target
    "negative_profit_flag",

    # Direct / derived profitability outcomes
    "net_profit",
    "profit_ratio",
    "profit_margin",
    "profit_per_unit",
    "profitability_category",
    "order_profit",

    # Conservative exclusion of realized delivery outcomes
    "actual_shipping_days",
    "shipping_delay_days",
    "shipping_performance",
    "shipping_datetime",

    # Delivery lifecycle information
    "is_delayed",
    "delivery_status",
    "late_delivery_risk",
    "order_status"
}


print("Delivery model exclusions:", len(delivery_exclusions))
print("Profitability model exclusions:", len(profitability_exclusions))

Delivery model exclusions: 31
Profitability model exclusions: 32


The enterprise feature store is converted into two controlled, model-specific predictive feature sets: **Delivery Delay Prediction** and **Profitability Risk Prediction**.

<h2 style="color: #548235; font-weight: 700;">Model Exclusion Validation</h2>

| Model | Excluded Features |
|---|---:|
| Delivery Delay Model | **31** |
| Profitability Risk Model | **32** |

<h2 style="color: #548235; font-weight: 700;">Delivery Delay Model</h2>

The delivery model retains information that can reasonably be available before the delivery outcome.

Excluded categories include:

- Delivery target: `is_delayed`
- Actual shipping outcomes
- Shipping delay and performance measures
- Post-order shipping timestamp
- Delivery-status and lifecycle fields
- `late_delivery_risk`
- Realized profitability variables that could introduce hindsight

<h2 style="color: #548235; font-weight: 700;">Profitability Risk Model</h2>

The profitability model is designed to predict negative profitability without exposing realized financial outcomes.

Excluded categories include:

- Profitability target: `negative_profit_flag`
- `net_profit`
- `order_profit`
- `profit_ratio`
- `profit_margin`
- `profit_per_unit`
- `profitability_category`
- Realized shipping outcomes and post-outcome shipping information
- Delivery lifecycle and outcome fields

<h2 style="color: #548235; font-weight: 700;">Common Controls</h2>

Both models exclude:

- **10 identifier fields**
- **6 PII / sensitive fields**
- `product_image_url`

<h2 style="color: #548235; font-weight: 700;">Key Observation</h2>>

The two models use **different exclusion rules** because their prediction objectives and leakage risks differ. The delivery model can use valid transaction and commercial information such as `gross_sales`, while the profitability model follows a more conservative approach and excludes realized financial outcomes.

<h2 style="color: #548235; font-weight: 700;">Modeling Implication</h2>

These model-specific exclusion sets establish the **controlled input layer for downstream machine learning**, ensuring that predictive features do not contain targets, post-outcome information, identifiers, PII, or other hindsight-based variables.

<h3 style="color: #7F6000; font-weight: 700;">**Status: ✅ MODEL-SPECIFIC ELIGIBILITY ESTABLISHED — 31 DELIVERY EXCLUSIONS + 32 PROFITABILITY EXCLUSIONS**</h3>

<h1 style="color: #1F4E78; font-weight: 700;">6.8 Predictive Feature Matrices</h1>

Two independent feature matrices are created:

1. Delivery Delay Prediction Matrix
2. Profitability Risk Prediction Matrix

The target variable and excluded fields are removed from the corresponding predictor matrix.

Transaction identifiers are retained separately for traceability but are not supplied to the machine learning algorithms.

This separation establishes a clean distinction between:

- Prediction inputs
- Prediction targets
- Transaction metadata

In [20]:
# ============================================================
# 6.8 CREATE MODEL FEATURE MATRICES
# ============================================================

# ------------------------------------------------------------
# Traceability metadata
# ------------------------------------------------------------

trace_columns = [
    col for col in [
        "order_id",
        "order_item_id",
        "order_datetime"
    ]
    if col in df.columns
]

model_metadata = df[trace_columns].copy()


# ------------------------------------------------------------
# Delivery Delay Model
# ------------------------------------------------------------

delivery_target = df["is_delayed"].astype(int)

delivery_features = df.drop(
    columns=list(delivery_exclusions),
    errors="ignore"
).copy()


# ------------------------------------------------------------
# Profitability Risk Model
# ------------------------------------------------------------

profitability_target = (
    df["negative_profit_flag"]
    .astype(int)
)

profitability_features = df.drop(
    columns=list(profitability_exclusions),
    errors="ignore"
).copy()


print("MODEL DATASET SHAPES")
print("=" * 70)

print(
    f"Delivery features       : "
    f"{delivery_features.shape}"
)

print(
    f"Delivery target         : "
    f"{delivery_target.shape}"
)

print(
    f"Profitability features  : "
    f"{profitability_features.shape}"
)

print(
    f"Profitability target    : "
    f"{profitability_target.shape}"
)

print(
    f"Metadata                : "
    f"{model_metadata.shape}"
)

MODEL DATASET SHAPES
Delivery features       : (180519, 159)
Delivery target         : (180519,)
Profitability features  : (180519, 158)
Profitability target    : (180519,)
Metadata                : (180519, 3)


Two independent modeling feature matrices were created for the predictive objectives while keeping transaction traceability metadata separate from machine learning inputs.

<h2 style="color: #548235; font-weight: 700;">Model Dataset Validation</h2>

| Dataset | Rows | Columns |
|---|---:|---:|
| Delivery Features | **180,519** | **159** |
| Delivery Target | **180,519** | 1 |
| Profitability Features | **180,519** | **158** |
| Profitability Target | **180,519** | 1 |
| Traceability Metadata | **180,519** | **3** |

<h2 style="color: #548235; font-weight: 700;">Key Observations</h2>

- Both models contain the complete **180,519 transactions**.
- The Delivery Delay model contains **159 predictive features** after applying its exclusion rules.
- The Profitability Risk model contains **158 predictive features**, reflecting its more conservative exclusion of profitability-related information.
- The targets are maintained separately from their corresponding predictor matrices.
- Three metadata fields — `order_id`, `order_item_id`, and `order_datetime` — are retained separately for transaction traceability and are not supplied as model predictors.

<h2 style="color: #548235; font-weight: 700;">Modeling Architecture</h2>

The resulting structure provides a clean separation between:

**Predictive Features → Targets → Traceability Metadata**

This prevents identifiers and targets from being inadvertently passed into machine learning algorithms while preserving the ability to trace model observations back to the original transactions.

<h3 style="color: #7F6000; font-weight: 700;">**Status: ✅ PREDICTIVE FEATURE MATRICES CREATED — 159 DELIVERY FEATURES + 158 PROFITABILITY FEATURES**</h3>

<h1 style="color: #1F4E78; font-weight: 700;">6.9 Leakage-Control Validation</h1>

The final feature matrices are validated against the leakage registry.

A model feature matrix passes this validation only when:

- Its target variable is absent from predictors.
- Registered leakage variables are absent.
- Identifiers are absent.
- PII/sensitive fields are absent.
- The number of observations matches the enterprise feature store.

This provides an automated safeguard against accidental leakage during downstream model development.

In [21]:
# ============================================================
# 6.9 LEAKAGE-CONTROL VALIDATION
# ============================================================

def validate_model_features(
    feature_matrix,
    excluded_features,
    target_name,
    model_name
):
    
    present_exclusions = (
        set(feature_matrix.columns)
        & set(excluded_features)
    )
    
    target_present = target_name in feature_matrix.columns
    
    identifiers_present = (
        set(feature_matrix.columns)
        & identifier_features
    )
    
    pii_present = (
        set(feature_matrix.columns)
        & pii_features
    )
    
    print(f"\n{model_name}")
    print("=" * 70)
    
    print(
        f"Rows match source dataset : "
        f"{len(feature_matrix) == len(df)}"
    )
    
    print(
        f"Target absent             : "
        f"{not target_present}"
    )
    
    print(
        f"Known exclusions absent   : "
        f"{len(present_exclusions) == 0}"
    )
    
    print(
        f"Identifiers absent        : "
        f"{len(identifiers_present) == 0}"
    )
    
    print(
        f"PII fields absent         : "
        f"{len(pii_present) == 0}"
    )
    
    if present_exclusions:
        print("\nERROR — Excluded fields found:")
        print(sorted(present_exclusions))
        
    if identifiers_present:
        print("\nERROR — Identifier fields found:")
        print(sorted(identifiers_present))
        
    if pii_present:
        print("\nERROR — PII fields found:")
        print(sorted(pii_present))


validate_model_features(
    delivery_features,
    delivery_exclusions,
    "is_delayed",
    "DELIVERY DELAY MODEL"
)

validate_model_features(
    profitability_features,
    profitability_exclusions,
    "negative_profit_flag",
    "PROFITABILITY RISK MODEL"
)


DELIVERY DELAY MODEL
Rows match source dataset : True
Target absent             : True
Known exclusions absent   : True
Identifiers absent        : True
PII fields absent         : True

PROFITABILITY RISK MODEL
Rows match source dataset : True
Target absent             : True
Known exclusions absent   : True
Identifiers absent        : True
PII fields absent         : True


The final predictive feature matrices were validated against the established leakage registry and modeling eligibility controls.

<h2 style="color: #548235; font-weight: 700;">Validation Results</h2>

| Validation Check | Delivery Delay | Profitability Risk |
|---|:---:|:---:|
| Rows match source dataset | ✅ | ✅ |
| Target absent from predictors | ✅ | ✅ |
| Known leakage exclusions absent | ✅ | ✅ |
| Identifiers absent | ✅ | ✅ |
| PII / sensitive fields absent | ✅ | ✅ |

<h2 style="color: #548235; font-weight: 700;">Key Observations</h2>

- Both modeling matrices contain the same **180,519 observations** as the enterprise feature store.
- The corresponding prediction targets are completely separated from the predictor matrices.
- All registered leakage variables are absent from both models.
- All **identifier fields** and **PII/sensitive fields** are excluded from predictive inputs.
- No validation errors or excluded fields were detected.

<h2 style="color: #548235; font-weight: 700;">Modeling Implication</h2>

The automated validation confirms that both feature matrices satisfy the defined **leakage-control and eligibility rules**. This establishes a controlled and reproducible input layer for downstream model development.

<h3 style="color: #7F6000; font-weight: 700;">**Status: ✅ LEAKAGE CONTROL PASSED — BOTH MODELING MATRICES VALIDATED**</h3>

<hr>

<h1 style="color: #1F4E78; font-weight: 700;">7 — Time-Aware Train/Test Strategy</h1>

A time-aware modeling boundary is established before encoding or model training to simulate how SupplySight AI would perform when predicting **future procurement transactions**.

<h2 style="color: #548235; font-weight: 700;">Strategy</h2>

The primary temporal split is:

| Period | Role |
|---|---|
| **2015–2017** | TRAIN |
| **January 2018** | TEST / FUTURE HOLDOUT |

Since the enterprise dataset ends on **2018-01-31**, January 2018 represents a genuine unseen future period for model evaluation.

The primary split will **not randomly shuffle** the observations.

<h2 style="color: #548235; font-weight: 700;">Why Time-Aware Splitting?</h2>

A random split could allow observations from January 2018 to enter the training set while earlier observations are placed in the test set. This does not accurately represent a real-world future prediction scenario.

The time-aware approach ensures that:

- Training uses only **historical observations**.
- Testing uses a **chronologically later period**.
- The test period represents a realistic **future-like holdout**.
- Model evaluation better reflects expected deployment behavior.

<h2 style="color: #7F6000; font-weight: 700;">Primary Temporal Boundary</h2>

```text
2015 ─────────────── 2017 │ January 2018
        TRAIN             │   TEST
                           │
                    Future Holdout

<h1 style="color: #1F4E78; font-weight: 700;">7.1 Sort by Transaction Date</h1>

In [22]:
# ============================================================
# PART 7 — TIME-AWARE TRAIN / TEST STRATEGY
# Sort Data Chronologically
# ============================================================

# Work from the original feature store so that the same
# chronological boundary is applied consistently.

df_model = df.copy()

df_model["order_datetime"] = pd.to_datetime(
    df_model["order_datetime"],
    errors="coerce"
)

df_model = df_model.sort_values(
    "order_datetime"
).reset_index(drop=True)

print("Chronological Ordering Validation")
print("=" * 70)

print(f"Rows                    : {len(df_model):,}")
print(f"Minimum order datetime  : {df_model['order_datetime'].min()}")
print(f"Maximum order datetime  : {df_model['order_datetime'].max()}")

is_sorted = df_model["order_datetime"].is_monotonic_increasing

print(f"Chronologically sorted  : {is_sorted}")

assert is_sorted, "ERROR: Dataset is not chronologically sorted."

print("\n✓ Chronological ordering validated.")

Chronological Ordering Validation
Rows                    : 180,519
Minimum order datetime  : 2015-01-01 00:00:00
Maximum order datetime  : 2018-01-31 23:38:00
Chronologically sorted  : True

✓ Chronological ordering validated.


The enterprise feature store was copied and sorted chronologically by `order_datetime` before applying the train/test boundary.

<h2 style="color: #548235; font-weight: 700;">Chronological Ordering Validation</h2>

| Validation Metric | Result |
|---|---:|
| Total Rows | **180,519** |
| Minimum Order Datetime | **2015-01-01 00:00:00** |
| Maximum Order Datetime | **2018-01-31 23:38:00** |
| Chronologically Sorted | **True** |

<h2 style="color: #548235; font-weight: 700;">Key Observations</h2>

- All **180,519 transactions** are retained after chronological sorting.
- The transaction timeline spans from **2015-01-01** through **2018-01-31 23:38:00**.
- `order_datetime` is confirmed to be **monotonically increasing**.
- The chronological ordering provides a consistent foundation for applying the planned **2015–2017 training / January 2018 future holdout** boundary.

<h3 style="color: #7F6000; font-weight: 700;">Status: ✅ CHRONOLOGICAL ORDERING VALIDATED — 180,519 ROWS SORTED SUCCESSFULLY</h3>

<h1 style="color: #1F4E78; font-weight: 700;">7.2 Create the Temporal Split</h1>

In [23]:
# ============================================================
# Create Time-Aware Train / Test Split
# ============================================================

train_mask = (
    df_model["order_datetime"] < pd.Timestamp("2018-01-01")
)

test_mask = (
    df_model["order_datetime"] >= pd.Timestamp("2018-01-01")
)

train_idx = df_model.index[train_mask]
test_idx = df_model.index[test_mask]

print("TIME-AWARE SPLIT")
print("=" * 70)

print(f"Training rows : {len(train_idx):,}")
print(f"Test rows     : {len(test_idx):,}")
print(f"Total rows    : {len(train_idx) + len(test_idx):,}")

print("\nTRAINING PERIOD")
print("-" * 70)
print(f"Start : {df_model.loc[train_idx, 'order_datetime'].min()}")
print(f"End   : {df_model.loc[train_idx, 'order_datetime'].max()}")

print("\nTEST / FUTURE HOLDOUT PERIOD")
print("-" * 70)
print(f"Start : {df_model.loc[test_idx, 'order_datetime'].min()}")
print(f"End   : {df_model.loc[test_idx, 'order_datetime'].max()}")

assert len(train_idx) + len(test_idx) == len(df_model)

assert (
    df_model.loc[train_idx, "order_datetime"].max()
    < df_model.loc[test_idx, "order_datetime"].min()
)

print("\n✓ No temporal overlap detected.")
print("✓ Future holdout boundary validated.")

TIME-AWARE SPLIT
Training rows : 178,396
Test rows     : 2,123
Total rows    : 180,519

TRAINING PERIOD
----------------------------------------------------------------------
Start : 2015-01-01 00:00:00
End   : 2017-12-31 23:52:00

TEST / FUTURE HOLDOUT PERIOD
----------------------------------------------------------------------
Start : 2018-01-01 00:13:00
End   : 2018-01-31 23:38:00

✓ No temporal overlap detected.
✓ Future holdout boundary validated.


The chronologically ordered dataset was divided into a **training period** and a **future holdout period** using **2018-01-01** as the temporal boundary.

<h2 style="color: #548235; font-weight: 700;">Time-Aware Split Validation</h2>

| Dataset | Rows | Period |
|---|---:|---|
| **Training** | **178,396** | 2015-01-01 00:00:00 → 2017-12-31 23:52:00 |
| **Test / Future Holdout** | **2,123** | 2018-01-01 00:13:00 → 2018-01-31 23:38:00 |
| **Total** | **180,519** | Complete dataset |

<h2 style="color: #548235; font-weight: 700;">Key Observations</h2>

- **178,396 transactions** are assigned to the historical training period.
- **2,123 transactions** are reserved as the January 2018 future holdout.
- The two datasets together account for all **180,519 transactions**.
- The latest training transaction occurs on **2017-12-31 23:52:00**.
- The earliest test transaction occurs on **2018-01-01 00:13:00**.
- The validation confirms **no temporal overlap** between training and test observations.

<h3 style="color: #7F6000; font-weight: 700;">Status: ✅ TEMPORAL SPLIT VALIDATED — 178,396 TRAIN / 2,123 FUTURE HOLDOUT</h3>

<h1 style="color: #1F4E78; font-weight: 700;">7.3 Validate Class Balance</h1>
We need to check whether the future holdout has a substantially different target distribution.

In [24]:
# ============================================================
# Target Distribution Across Time Split
# ============================================================

delivery_train_y = df_model.loc[
    train_idx, "is_delayed"
].astype(int)

delivery_test_y = df_model.loc[
    test_idx, "is_delayed"
].astype(int)

profit_train_y = df_model.loc[
    train_idx, "negative_profit_flag"
].astype(int)

profit_test_y = df_model.loc[
    test_idx, "negative_profit_flag"
].astype(int)


def target_distribution(y, target_name, split_name):

    counts = y.value_counts().sort_index()
    percentages = y.value_counts(
        normalize=True
    ).sort_index() * 100

    result = pd.DataFrame({
        "Count": counts,
        "Percentage": percentages.round(2)
    })

    print(f"\n{target_name} — {split_name}")
    print("-" * 70)
    print(result)


print("TARGET DISTRIBUTION BY TEMPORAL SPLIT")
print("=" * 70)

target_distribution(
    delivery_train_y,
    "Delivery Delay",
    "Training"
)

target_distribution(
    delivery_test_y,
    "Delivery Delay",
    "Future Test"
)

target_distribution(
    profit_train_y,
    "Profitability Risk",
    "Training"
)

target_distribution(
    profit_test_y,
    "Profitability Risk",
    "Future Test"
)

TARGET DISTRIBUTION BY TEMPORAL SPLIT

Delivery Delay — Training
----------------------------------------------------------------------
             Count  Percentage
is_delayed                    
0            76242       42.74
1           102154       57.26

Delivery Delay — Future Test
----------------------------------------------------------------------
            Count  Percentage
is_delayed                   
0             877       41.31
1            1246       58.69

Profitability Risk — Training
----------------------------------------------------------------------
                       Count  Percentage
negative_profit_flag                    
0                     145021       81.29
1                      33375       18.71

Profitability Risk — Future Test
----------------------------------------------------------------------
                      Count  Percentage
negative_profit_flag                   
0                      1714       80.73
1                       409 

The target distributions were compared between the historical training period and the January 2018 future holdout to identify any substantial temporal shift in class balance.

<h2 style="color: #548235; font-weight: 700;">Target Distribution Validation</h2>

| Model | Split | Class 0 | Class 1 |
|---|---|---:|---:|
| **Delivery Delay** | Training | 76,242 (**42.74%**) | 102,154 (**57.26%**) |
| **Delivery Delay** | Future Test | 877 (**41.31%**) | 1,246 (**58.69%**) |
| **Profitability Risk** | Training | 145,021 (**81.29%**) | 33,375 (**18.71%**) |
| **Profitability Risk** | Future Test | 1,714 (**80.73%**) | 409 (**19.27%**) |

<h2 style="color: #548235; font-weight: 700;">Key Observations</h2>

- The **Delivery Delay** target remains relatively stable between training and future test, with delayed transactions increasing from **57.26% to 58.69%**.
- The **Profitability Risk** target also remains stable, with negative-profit transactions increasing slightly from **18.71% to 19.27%**.
- The future holdout does not show a substantial change in either target distribution.
- Both prediction tasks retain their original class characteristics in the future evaluation period.

<h2 style="color: #7F6000; font-weight: 700;">Modeling Implication</h2>

The relatively stable class distributions support the use of **January 2018 as the primary future holdout** for model evaluation. The profitability-risk target remains more imbalanced than the delivery-delay target and should therefore be considered carefully during downstream model training and evaluation.

<h3 style="color: #7F6000; font-weight: 700;">**Status: ✅ CLASS BALANCE VALIDATED — NO SUBSTANTIAL TEMPORAL TARGET SHIFT OBSERVED**</h3>

<h1 style="color: #1F4E78; font-weight: 700;">7.4 Formal Temporal Leakage Validation</h1>

In [25]:
# ============================================================
# Formal Temporal Leakage Validation
# ============================================================

train_max_date = df_model.loc[
    train_idx, "order_datetime"
].max()

test_min_date = df_model.loc[
    test_idx, "order_datetime"
].min()

temporal_gap_valid = train_max_date < test_min_date

print("TEMPORAL LEAKAGE VALIDATION")
print("=" * 70)

print(f"Training maximum date : {train_max_date}")
print(f"Test minimum date     : {test_min_date}")
print(f"Temporal separation   : {temporal_gap_valid}")

# Confirm every training observation occurs before
# every test observation.
max_train_index_date = df_model.loc[
    train_idx, "order_datetime"
].max()

min_test_index_date = df_model.loc[
    test_idx, "order_datetime"
].min()

assert max_train_index_date < min_test_index_date

print("\n✓ PASS — Training data occurs strictly before test data.")
print("✓ PASS — Future holdout is temporally isolated.")

TEMPORAL LEAKAGE VALIDATION
Training maximum date : 2017-12-31 23:52:00
Test minimum date     : 2018-01-01 00:13:00
Temporal separation   : True

✓ PASS — Training data occurs strictly before test data.
✓ PASS — Future holdout is temporally isolated.


A formal validation was performed to confirm that the training observations occur strictly before the January 2018 future holdout.

<h2 style="color: #548235; font-weight: 700;">Temporal Leakage Validation</h2>

| Validation Check | Result |
|---|---|
| Training Maximum Date | **2017-12-31 23:52:00** |
| Test Minimum Date | **2018-01-01 00:13:00** |
| Temporal Separation | **True** |
| Training Before Test | ✅ PASS |
| Future Holdout Isolated | ✅ PASS |

<h2 style="color: #548235; font-weight: 700;">Key Observations</h2>

- The final training observation occurs on **2017-12-31 23:52:00**.
- The first future-holdout observation occurs on **2018-01-01 00:13:00**.
- The training and test periods are strictly separated in time.
- No training observation occurs after the beginning of the future holdout.
- The assertion confirms that **every training observation occurs before the test period**.

<h2 style="color: #7F6000; font-weight: 700;">Modeling Implication</h2>

The temporal boundary is formally validated, providing protection against **temporal leakage** in the primary train/test split and ensuring that January 2018 remains a genuinely future-like evaluation period.

<h3 style="color: #7F6000; font-weight: 700;">**Status: ✅ TEMPORAL LEAKAGE CONTROL PASSED — FUTURE HOLDOUT STRICTLY ISOLATED**</h3>

<h1 style="color: #1F4E78; font-weight: 700;">7.5 Create Reusable Split Indices</h1>
Rather than creating separate arbitrary slices later, let's preserve the indices as our canonical modeling split.

In [26]:
# ============================================================
# Canonical Modeling Split
# ============================================================

model_split = {
    "train_idx": train_idx,
    "test_idx": test_idx,
    "train_start": df_model.loc[
        train_idx, "order_datetime"
    ].min(),
    "train_end": df_model.loc[
        train_idx, "order_datetime"
    ].max(),
    "test_start": df_model.loc[
        test_idx, "order_datetime"
    ].min(),
    "test_end": df_model.loc[
        test_idx, "order_datetime"
    ].max()
}

print("CANONICAL MODEL SPLIT")
print("=" * 70)

for key, value in model_split.items():

    if key.endswith("_idx"):
        print(f"{key:15s}: {len(value):,} rows")
    else:
        print(f"{key:15s}: {value}")

print("\n✓ Canonical temporal split created.")

CANONICAL MODEL SPLIT
train_idx      : 178,396 rows
test_idx       : 2,123 rows
train_start    : 2015-01-01 00:00:00
train_end      : 2017-12-31 23:52:00
test_start     : 2018-01-01 00:13:00
test_end       : 2018-01-31 23:38:00

✓ Canonical temporal split created.


The validated temporal boundary was preserved as a **canonical modeling split** using reusable training and test indices, ensuring consistent dataset selection throughout downstream modeling.

<h2 style="color: #548235; font-weight: 700;">Canonical Split Validation</h2>

| Split Component | Result |
|---|---:|
| **Training Index** | **178,396 rows** |
| **Test Index** | **2,123 rows** |
| **Training Period** | **2015-01-01 00:00:00 → 2017-12-31 23:52:00** |
| **Future Holdout Period** | **2018-01-01 00:13:00 → 2018-01-31 23:38:00** |

<h2 style="color: #548235; font-weight: 700;">Key Observations</h2>

- The canonical `train_idx` contains **178,396 historical transactions**.
- The canonical `test_idx` contains **2,123 future transactions**.
- The stored indices preserve the previously validated **2015–2017 training / January 2018 holdout** strategy.
- The split boundaries remain strictly chronological and consistent with the validated temporal separation.

<h2 style="color: #7F6000; font-weight: 700;">Modeling Implication</h2>

The reusable split indices provide a **single source of truth** for downstream feature matrices, target vectors, preprocessing, and model evaluation, reducing the risk of inconsistent or arbitrary train/test slicing.

<h3 style="color: #7F6000; font-weight: 700;">**Status: ✅ CANONICAL MODELING SPLIT CREATED — 178,396 TRAIN / 2,123 FUTURE HOLDOUT**</h3>

<h1 style="color: #1F4E78; font-weight: 700;">7.6 Final Feature Boundary Correction</h1>

In [27]:
# ============================================================
# Final Feature Boundary Correction
# ============================================================

# Outcome-derived financial variables should not be used
# as predictive inputs for profitability risk.
profitability_exclusions.update({
    "gross_sales",
    "order_profit"
})

# order_profit is realized profit and should not be used
# as a predictor for delivery either.
delivery_exclusions.update({
    "order_profit"
})

# Rebuild the feature matrices after the correction.
delivery_features = df_model.drop(
    columns=list(delivery_exclusions),
    errors="ignore"
).copy()

profitability_features = df_model.drop(
    columns=list(profitability_exclusions),
    errors="ignore"
).copy()

print("FINAL FEATURE BOUNDARY")
print("=" * 70)

print(f"Delivery feature columns       : {delivery_features.shape[1]}")
print(f"Profitability feature columns : {profitability_features.shape[1]}")

print("\nExplicit corrections:")
print("✓ gross_sales excluded from profitability model")
print("✓ order_profit excluded from profitability model")
print("✓ order_profit excluded from delivery model")

FINAL FEATURE BOUNDARY
Delivery feature columns       : 158
Profitability feature columns : 157

Explicit corrections:
✓ gross_sales excluded from profitability model
✓ order_profit excluded from profitability model
✓ order_profit excluded from delivery model


The predictive feature boundaries were corrected to ensure that **outcome-derived financial variables are excluded from model inputs**, particularly for profitability-risk prediction.

<h2 style="color: #548235; font-weight: 700;">Final Feature Boundary</h2>

| Model | Final Feature Columns |
|---|---:|
| **Delivery Delay** | **158** |
| **Profitability Risk** | **157** |

<h2 style="color: #548235; font-weight: 700;">Key Corrections</h2>

- `gross_sales` was excluded from the **profitability-risk model** because it is an outcome-related financial variable.
- `order_profit` was excluded from the **profitability-risk model**.
- `order_profit` was excluded from the **delivery-delay model** because it represents realized profit and is not an appropriate predictive input.
- Both feature matrices were rebuilt after applying the corrected exclusion rules.

<h2 style="color: #7F6000; font-weight: 700;">Modeling Implication</h2>

The final feature boundaries provide a cleaner predictive input set and strengthen **target leakage control**, ensuring that realized financial outcomes are not inadvertently used as predictive features.

<h3 style="color: #7F6000; font-weight: 700;">**Status: ✅ FINAL FEATURE BOUNDARY CORRECTED — 158 DELIVERY / 157 PROFITABILITY FEATURES**</h3>

<h1 style="color: #1F4E78; font-weight: 700;">7.7 Remove Raw Datetime From Estimator Inputs</h1>

`order_datetime` is essential for our temporal split and traceability, but we don't want to feed the raw timestamp directly into the model.

We already have engineered temporal features such as:

- year
- month
- quarter
- day of week
- week number
- month position
- day type

In [28]:
# ============================================================
# Separate Traceability Fields
# ============================================================

model_trace_columns = [
    "order_id",
    "order_item_id",
    "order_datetime"
]

delivery_model_input = delivery_features.drop(
    columns=["order_datetime"],
    errors="ignore"
).copy()

profitability_model_input = profitability_features.drop(
    columns=["order_datetime"],
    errors="ignore"
).copy()

print("ESTIMATOR INPUT PREPARATION")
print("=" * 70)

print(f"Delivery input shape       : {delivery_model_input.shape}")
print(f"Profitability input shape  : {profitability_model_input.shape}")

print("\nTraceability retained separately:")
for col in model_trace_columns:
    print(f"✓ {col}")

ESTIMATOR INPUT PREPARATION
Delivery input shape       : (180519, 157)
Profitability input shape  : (180519, 156)

Traceability retained separately:
✓ order_id
✓ order_item_id
✓ order_datetime


The raw `order_datetime` field was removed from estimator inputs while retaining it separately for **temporal splitting and transaction traceability**. Engineered temporal features remain available for predictive modeling.

<h2 style="color: #548235; font-weight: 700;">Estimator Input Preparation</h2>

| Model | Final Input Shape |
|---|---:|
| **Delivery Delay** | **180,519 × 157** |
| **Profitability Risk** | **180,519 × 156** |

<h2 style="color: #548235; font-weight: 700;">Key Observations</h2>

- `order_datetime` is **not directly provided to either estimator**.
- Engineered temporal features such as year, month, quarter, day of week, week number, month position, and day type remain available.
- `order_id`, `order_item_id`, and `order_datetime` are retained separately as **traceability fields**.
- All **180,519 transactions** remain available in both model input datasets.

<h2 style="color: #7F6000; font-weight: 700;">Modeling Implication</h2>

Separating the raw timestamp from estimator inputs prevents the model from directly consuming the transaction timestamp while preserving the temporal information needed for **time-aware splitting, evaluation, and traceability**.

<h3 style="color: #7F6000; font-weight: 700;">**Status: ✅ ESTIMATOR INPUTS PREPARED — RAW DATETIME SEPARATED FROM MODEL FEATURES**</h3>

<h1 style="color: #1F4E78; font-weight: 700;">7.8 Split Feature Matrices Using Canonical Indices</h1>

In [29]:
# ============================================================
# Create Train/Test Feature Matrices
# ============================================================

X_delivery_train = delivery_model_input.loc[train_idx].copy()
X_delivery_test = delivery_model_input.loc[test_idx].copy()

X_profit_train = profitability_model_input.loc[train_idx].copy()
X_profit_test = profitability_model_input.loc[test_idx].copy()

y_delivery_train = delivery_train_y.copy()
y_delivery_test = delivery_test_y.copy()

y_profit_train = profit_train_y.copy()
y_profit_test = profit_test_y.copy()

print("MODEL MATRICES")
print("=" * 70)

print("DELIVERY DELAY")
print(f"X_train : {X_delivery_train.shape}")
print(f"X_test  : {X_delivery_test.shape}")
print(f"y_train : {y_delivery_train.shape}")
print(f"y_test  : {y_delivery_test.shape}")

print("\nPROFITABILITY RISK")
print(f"X_train : {X_profit_train.shape}")
print(f"X_test  : {X_profit_test.shape}")
print(f"y_train : {y_profit_train.shape}")
print(f"y_test  : {y_profit_test.shape}")

MODEL MATRICES
DELIVERY DELAY
X_train : (178396, 157)
X_test  : (2123, 157)
y_train : (178396,)
y_test  : (2123,)

PROFITABILITY RISK
X_train : (178396, 156)
X_test  : (2123, 156)
y_train : (178396,)
y_test  : (2123,)


The prepared estimator inputs were split into **training and future-test matrices** using the previously established canonical temporal indices, ensuring consistent alignment between features and targets.

<h2 style="color: #548235; font-weight: 700;">Model Matrix Validation</h2>

| Model | Training Features | Test Features | Training Target | Test Target |
|---|---:|---:|---:|---:|
| **Delivery Delay** | **178,396 × 157** | **2,123 × 157** | **178,396** | **2,123** |
| **Profitability Risk** | **178,396 × 156** | **2,123 × 156** | **178,396** | **2,123** |

<h2 style="color: #548235; font-weight: 700;">Key Observations</h2>

- Both models use the same **178,396 training transactions** and **2,123 future-holdout transactions**.
- Delivery Delay contains **157 predictive input features**.
- Profitability Risk contains **156 predictive input features**.
- Feature and target row counts are aligned for both training and test datasets.
- The canonical temporal indices ensure that the **January 2018 future holdout remains isolated**.

<h2 style="color: #7F6000; font-weight: 700;">Modeling Implication</h2>

The final feature matrices and target vectors are now **model-ready in terms of temporal partitioning**, with consistent feature-target alignment and no arbitrary train/test slicing.

<h3 style="color: #7F6000; font-weight: 700;">**Status: ✅ MODEL MATRICES CREATED — TEMPORAL TRAIN/TEST ALIGNMENT VALIDATED**</h3>

<h1 style="color: #1F4E78; font-weight: 700;">7.9 Identify Data Types</h1>

Now we need to understand what preprocessing is actually required.

In [30]:
# ============================================================
# Data Type Audit
# ============================================================

def audit_dtypes(X, model_name):

    # --------------------------------------------------------
    # Numeric features
    # --------------------------------------------------------

    numeric_cols = X.select_dtypes(
        include=["number"]
    ).columns.tolist()

    # --------------------------------------------------------
    # Categorical / String features
    # --------------------------------------------------------

    categorical_cols = X.select_dtypes(
        include=["object", "string", "str", "category"]
    ).columns.tolist()

    # --------------------------------------------------------
    # Boolean features
    # --------------------------------------------------------

    boolean_cols = X.select_dtypes(
        include=["bool", "boolean"]
    ).columns.tolist()

    # --------------------------------------------------------
    # Identify unsupported / unexpected data types
    # --------------------------------------------------------

    classified_cols = (
        set(numeric_cols)
        | set(categorical_cols)
        | set(boolean_cols)
    )

    other_cols = [
        col for col in X.columns
        if col not in classified_cols
    ]

    # --------------------------------------------------------
    # Audit Summary
    # --------------------------------------------------------

    print(f"\n{model_name}")
    print("=" * 70)

    print(f"Total features       : {X.shape[1]}")
    print(f"Numeric features     : {len(numeric_cols)}")
    print(f"Categorical features : {len(categorical_cols)}")
    print(f"Boolean features     : {len(boolean_cols)}")
    print(f"Other features       : {len(other_cols)}")

    # --------------------------------------------------------
    # Display unexpected data types
    # --------------------------------------------------------

    if other_cols:
        print("\nOther data types:")
        print(X[other_cols].dtypes)

    return (
        numeric_cols,
        categorical_cols,
        boolean_cols,
        other_cols
    )


# ============================================================
# DELIVERY MODEL DATA TYPE AUDIT
# ============================================================

delivery_dtype_info = audit_dtypes(
    X_delivery_train,
    "DELIVERY MODEL"
)


# ============================================================
# PROFITABILITY MODEL DATA TYPE AUDIT
# ============================================================

profitability_dtype_info = audit_dtypes(
    X_profit_train,
    "PROFITABILITY MODEL"
)


DELIVERY MODEL
Total features       : 157
Numeric features     : 104
Categorical features : 41
Boolean features     : 12
Other features       : 0

PROFITABILITY MODEL
Total features       : 156
Numeric features     : 103
Categorical features : 41
Boolean features     : 12
Other features       : 0


A data-type audit was performed on the training feature matrices to determine the preprocessing requirements for the downstream modeling pipeline.

<h2 style="color: #548235; font-weight: 700;">Data Type Audit</h2>

| Model | Total Features | Numeric | Categorical | Boolean | Other |
|---|---:|---:|---:|---:|---:|
| **Delivery Delay** | **157** | **104** | **41** | **12** | **0** |
| **Profitability Risk** | **156** | **103** | **41** | **12** | **0** |

<h2 style="color: #548235; font-weight: 700;">Key Observations</h2>

- The Delivery Delay model contains **104 numeric, 41 categorical, and 12 boolean features**.
- The Profitability Risk model contains **103 numeric, 41 categorical, and 12 boolean features**.
- Both models contain **zero unsupported or unexpected data types**.
- The one-feature difference between the models is consistent with their previously defined feature boundaries.

<h2 style="color: #7F6000; font-weight: 700;">Modeling Implication</h2>

The feature matrices contain only **numeric, categorical, and boolean data types**, providing a clear basis for designing the downstream preprocessing pipeline for encoding categorical variables and handling boolean/numeric features.

<h3 style="color: #7F6000; font-weight: 700;">**Status: ✅ DATA TYPE AUDIT PASSED — ALL FEATURES SUCCESSFULLY CLASSIFIED**</h3>

<hr>

<h1 style="color: #1F4E78; font-weight: 700;">8 — Modeling Matrix Preparation & Preprocessing</h1>

<h2 style="color: #1F4E78; font-weight: 700;">Objective</h2>

Transform the leakage-controlled features and time-aware train/test splits into **model-ready input matrices** using a reusable and reproducible preprocessing architecture.

No machine learning models will be trained in this part.

<h2 style="color: #1F4E78; font-weight: 700;">Preprocessing Requirements</h2>

The feature store contains:

- Numerical features
- Categorical features
- Boolean features
- Temporal and geographic features
- Procurement, market, weather, and calendar information

These features must be converted into a consistent numerical representation before modeling.

<h2 style="color: #1F4E78; font-weight: 700;">Modeling Boundary</h2>

The leakage controls established in Part 6 remain enforced.

**Delivery Delay Model**

Post-outcome variables such as `actual_shipping_days`, `shipping_delay_days`, `shipping_performance`, `is_delayed`, `shipping_datetime`, and `delivery_status` remain excluded.

**Profitability Risk Model**

Outcome-derived financial variables such as `net_profit`, `order_profit`, `profit_ratio`, `profit_margin`, `profit_per_unit`, `profitability_category`, `negative_profit_flag`, and `gross_sales` remain excluded.

Realized delivery outcomes are also excluded from the profitability model.

<h2 style="color: #1F4E78; font-weight: 700;">Traceability Fields</h2>

The following fields are retained separately but are not supplied directly to estimators:

- `order_id`
- `order_item_id`
- `order_datetime`

Engineered temporal features provide the usable time information for modeling.

<h2 style="color: #1F4E78; font-weight: 700;">Time-Aware Preprocessing</h2>

The primary temporal strategy remains:

- **Training:** January 2015 – December 2017
- **Future Holdout:** January 2018

All preprocessing transformations will be **fitted only on training data** and then applied to the future holdout.

**Training → Fit Preprocessing → Transform Training**

**Future Holdout → Apply Fitted Preprocessing**

This prevents future-period information from influencing preprocessing.

<h2 style="color: #1F4E78; font-weight: 700;">Preprocessing Architecture</h2>

| Feature Type | Preprocessing |
|---|---|
| **Numerical** | Median imputation and scaling where appropriate |
| **Categorical** | One-hot encoding with `handle_unknown="ignore"` |
| **Boolean** | Numerical conversion |

Reusable scikit-learn components will be used instead of manual transformations.

<h2 style="color: #1F4E78; font-weight: 700;">Part 8 Workflow</h2>

1. Identify feature types
2. Audit categorical cardinality
3. Validate missing values
4. Identify constant features
5. Define numerical preprocessing
6. Define categorical preprocessing
7. Define boolean preprocessing
8. Combine transformations with `ColumnTransformer`
9. Fit preprocessing only on training data
10. Transform training and future holdout
11. Validate matrix dimensions
12. Confirm preprocessing leakage controls
13. Finalize reusable preprocessing pipelines

<h2 style="color: #1F4E78; font-weight: 700;">Expected Outcome</h2>

Two reusable preprocessing pipelines will be established:

- **Delivery Delay Preprocessing Pipeline**
- **Profitability Risk Preprocessing Pipeline**

These pipelines will provide the foundation for subsequent predictive modeling while maintaining the established **leakage-controlled and time-aware evaluation framework**.

<h1 style="color: #1F4E78; font-weight: 700;">8.1 — Preprocessing Architecture</h1>

The preprocessing framework will use a **scikit-learn `ColumnTransformer` + `Pipeline` architecture** to provide a consistent and reusable transformation process.

<h2 style="color: #1F4E78; font-weight: 700;">Preprocessing Strategy</h2>

- **Training-only fitting:** All preprocessing components will be fitted exclusively on the training data.
- **Future holdout:** January 2018 will remain completely unseen during preprocessing.
- **Categorical features:** `OneHotEncoder(handle_unknown="ignore")` will handle unseen categories safely.
- **Numerical features:** Median imputation followed by scaling will be applied.
- **Boolean features:** Booleans will be processed through a numerical preprocessing pipeline.

<h2 style="color: #1F4E78; font-weight: 700;">Modeling Implication</h2>

This architecture ensures **consistent transformations, protection against preprocessing leakage, and safe handling of unseen future categories**, providing a reproducible foundation for downstream model training.

In [31]:
# ============================================================
# MODELING PREPROCESSING
# Feature Type Identification
# ============================================================

def get_feature_types(X):

    # --------------------------------------------------------
    # Numeric Features
    # --------------------------------------------------------
    numeric_features = X.select_dtypes(
        include=["number"]
    ).columns.tolist()

    # --------------------------------------------------------
    # Categorical Features
    # Explicitly include object, string, str, and category
    # to avoid Pandas4Warning
    # --------------------------------------------------------
    categorical_features = X.select_dtypes(
        include=["object", "string", "str", "category"]
    ).columns.tolist()

    # --------------------------------------------------------
    # Boolean Features
    # Include both standard and nullable boolean dtypes
    # --------------------------------------------------------
    boolean_features = X.select_dtypes(
        include=["bool", "boolean"]
    ).columns.tolist()

    return (
        numeric_features,
        categorical_features,
        boolean_features
    )


# ============================================================
# DELIVERY MODEL
# ============================================================

(
    delivery_numeric,
    delivery_categorical,
    delivery_boolean
) = get_feature_types(
    X_delivery_train
)


# ============================================================
# PROFITABILITY MODEL
# ============================================================

(
    profit_numeric,
    profit_categorical,
    profit_boolean
) = get_feature_types(
    X_profit_train
)


# ============================================================
# FEATURE TYPE SUMMARY
# ============================================================

print("DELIVERY MODEL")
print("=" * 70)
print(f"Numeric features     : {len(delivery_numeric)}")
print(f"Categorical features : {len(delivery_categorical)}")
print(f"Boolean features     : {len(delivery_boolean)}")


print("\nPROFITABILITY MODEL")
print("=" * 70)
print(f"Numeric features     : {len(profit_numeric)}")
print(f"Categorical features : {len(profit_categorical)}")
print(f"Boolean features     : {len(profit_boolean)}")


# ============================================================
# FEATURE TYPE VALIDATION
# ============================================================

delivery_total_types = (
    len(delivery_numeric)
    + len(delivery_categorical)
    + len(delivery_boolean)
)

profit_total_types = (
    len(profit_numeric)
    + len(profit_categorical)
    + len(profit_boolean)
)


print("\nFEATURE TYPE VALIDATION")
print("=" * 70)

print(
    f"Delivery features classified   : "
    f"{delivery_total_types} / {X_delivery_train.shape[1]}"
)

print(
    f"Profitability features classified : "
    f"{profit_total_types} / {X_profit_train.shape[1]}"
)


if delivery_total_types == X_delivery_train.shape[1]:
    print("✓ All delivery features classified successfully.")
else:
    print("⚠ Some delivery features remain unclassified.")


if profit_total_types == X_profit_train.shape[1]:
    print("✓ All profitability features classified successfully.")
else:
    print("⚠ Some profitability features remain unclassified.")

DELIVERY MODEL
Numeric features     : 104
Categorical features : 41
Boolean features     : 12

PROFITABILITY MODEL
Numeric features     : 103
Categorical features : 41
Boolean features     : 12

FEATURE TYPE VALIDATION
Delivery features classified   : 157 / 157
Profitability features classified : 156 / 156
✓ All delivery features classified successfully.
✓ All profitability features classified successfully.


The model inputs were classified by data type to establish the foundation for the preprocessing pipelines.

<h2 style="color: #548235; font-weight: 700;">Feature Type Summary</h2>

| Model | Numeric | Categorical | Boolean | Total |
|---|---:|---:|---:|---:|
| **Delivery Delay** | **104** | **41** | **12** | **157** |
| **Profitability Risk** | **103** | **41** | **12** | **156** |

<h2 style="color: #548235; font-weight: 700;">Key Observations</h2>

- All **157 Delivery Delay features** were successfully classified.
- All **156 Profitability Risk features** were successfully classified.
- Both models contain **41 categorical** and **12 boolean** features.
- No features remain unclassified or require additional data-type handling.

<h2 style="color: #7F6000; font-weight: 700;">Modeling Implication</h2>

The complete feature-type classification confirms that the preprocessing architecture can now be defined separately for **numeric, categorical, and boolean features** using reusable scikit-learn components.

<h3 style="color: #7F6000; font-weight: 700;">**Status: ✅ FEATURE TYPE VALIDATION PASSED — 100% OF MODEL FEATURES CLASSIFIED**</h3>

<h1 style="color: #1F4E78; font-weight: 700;">8.1.1 Inspect Categorical Cardinality</h1>

Before one-hot encoding 41 categorical variables, we need to understand their cardinality.

## Why this audit matters

We have 41 categorical variables, and some geographic/product fields may have relatively high cardinality.

We do not want to blindly assume that one-hot encoding everything is computationally harmless.

In particular, we'll inspect whether fields such as:

- product_name
- order_city
- customer_city
- customer_state
- standard_country
- erp_country

create a very large sparse matrix.

Don't change anything based on assumptions yet—send me this output first.

In [32]:
# ============================================================
# Categorical Cardinality Audit
# ============================================================

def categorical_cardinality_report(X, categorical_features, model_name):

    cardinality = pd.DataFrame({
        "feature": categorical_features,
        "unique_values": [
            X[col].nunique(dropna=False)
            for col in categorical_features
        ]
    }).sort_values(
        "unique_values",
        ascending=False
    ).reset_index(drop=True)

    print(f"\n{model_name} — CATEGORICAL CARDINALITY")
    print("=" * 70)

    display(cardinality)

    return cardinality


delivery_cardinality = categorical_cardinality_report(
    X_delivery_train,
    delivery_categorical,
    "DELIVERY MODEL"
)

profit_cardinality = categorical_cardinality_report(
    X_profit_train,
    profit_categorical,
    "PROFITABILITY MODEL"
)


DELIVERY MODEL — CATEGORICAL CARDINALITY


,feature,unique_values
0,order_city,3596
1,order_day,1096
2,order_state,1089
3,customer_city,563
4,holiday_name,227
5,erp_country,164
6,standard_country,164
7,weather_city,164
8,capital_city,164
9,country_iso3,164



PROFITABILITY MODEL — CATEGORICAL CARDINALITY


,feature,unique_values
0,order_city,3596
1,order_day,1096
2,order_state,1089
3,customer_city,563
4,holiday_name,227
5,erp_country,164
6,standard_country,164
7,weather_city,164
8,capital_city,164
9,country_iso3,164


A categorical cardinality audit was performed across the **41 categorical features** in both modeling datasets to assess the expected dimensionality of one-hot encoding.

<h2 style="color: #548235; font-weight: 700;">Cardinality Overview</h2>

| Cardinality Range | Examples |
|---|---|
| **Very High** | `order_city` (3,596), `order_day` (1,096), `order_state` (1,089) |
| **High** | `customer_city` (563), `holiday_name` (227), country fields (164) |
| **Medium** | `product_name` (118), `category_name` (50), `customer_state` (46) |
| **Low** | `department_name` (11), `market` (5), `transaction_type` (4) |
| **Very Low** | `day_type` (2), `customer_country` (2) |

<h2 style="color: #548235; font-weight: 700;">Key Observations</h2>

- Both models have the **same categorical cardinality profile across all 41 categorical features**.
- `order_city` has the highest cardinality with **3,596 unique values**.
- `order_day` and `order_state` contain **1,096** and **1,089** unique values respectively.
- Geographic and location-related features contribute several high-cardinality variables.
- Several engineered categorical features have relatively low cardinality, typically between **2 and 5 values**.

<h2 style="color: #7F6000; font-weight: 700;">Preprocessing Implication</h2>

The high-cardinality features may substantially increase the dimensionality of one-hot encoded matrices. Therefore, categorical encoding should be implemented carefully while retaining `handle_unknown="ignore"` to safely process categories appearing in the January 2018 future holdout.

<h3 style="color: #7F6000; font-weight: 700;">**Status: ✅ CATEGORICAL CARDINALITY AUDITED — HIGH-CARDINALITY FEATURES IDENTIFIED**</h3>

<h1 style="color: #1F4E78; font-weight: 700;">8.1.2 Check Missing Values After Split</h1>

Even though the integrated feature store had zero missing values, we should validate the actual modeling matrices independently.

In [33]:
# ============================================================
# Modeling Matrix Missing-Value Audit
# ============================================================

def missing_value_audit(X, model_name):

    missing = X.isna().sum()

    missing_features = (
        missing[missing > 0]
        .sort_values(ascending=False)
    )

    print(f"\n{model_name}")
    print("=" * 70)
    print(f"Total missing cells : {missing.sum():,}")
    print(f"Features with NA    : {len(missing_features)}")

    if len(missing_features) > 0:
        display(missing_features)

    return missing_features


delivery_missing = missing_value_audit(
    X_delivery_train,
    "DELIVERY TRAINING MATRIX"
)

profit_missing = missing_value_audit(
    X_profit_train,
    "PROFITABILITY TRAINING MATRIX"
)


DELIVERY TRAINING MATRIX
Total missing cells : 0
Features with NA    : 0

PROFITABILITY TRAINING MATRIX
Total missing cells : 0
Features with NA    : 0


The training feature matrices were independently audited for missing values after the time-aware split to confirm that no missing data was introduced during matrix preparation.

<h2 style="color: #548235; font-weight: 700;">Missing-Value Validation</h2>

| Model | Total Missing Cells | Features with Missing Values |
|---|---:|---:|
| **Delivery Training** | **0** | **0** |
| **Profitability Training** | **0** | **0** |

<h2 style="color: #548235; font-weight: 700;">Key Observations</h2>

- The Delivery Delay training matrix contains **zero missing cells**.
- The Profitability Risk training matrix also contains **zero missing cells**.
- No feature contains missing values after the temporal split.
- The modeling matrices therefore retain the same complete-data quality observed in the integrated feature store.

<h2 style="color: #7F6000; font-weight: 700;">Preprocessing Implication</h2>

Although the current training matrices contain no missing values, **median imputation will remain part of the numerical preprocessing pipeline** to provide robustness against missing values that may appear in future or production data.

<h3 style="color: #7F6000; font-weight: 700;">**Status: ✅ MISSING-VALUE AUDIT PASSED — 0 MISSING CELLS IN BOTH TRAINING MATRICES**</h3>

<h1 style="color: #1F4E78; font-weight: 700;">8.1.3 Check Constant Features</h1>

This is a useful enterprise modeling-quality check before preprocessing.

In [34]:
# ============================================================
# Constant Feature Audit
# ============================================================

def constant_feature_audit(X, model_name):

    constant_features = [
        col for col in X.columns
        if X[col].nunique(dropna=False) <= 1
    ]

    print(f"\n{model_name}")
    print("=" * 70)
    print(f"Constant features : {len(constant_features)}")

    if constant_features:
        print("\nConstant columns:")
        for col in constant_features:
            print(f" - {col}")
    else:
        print("✓ No constant features detected.")

    return constant_features


delivery_constant = constant_feature_audit(
    X_delivery_train,
    "DELIVERY MODEL"
)

profit_constant = constant_feature_audit(
    X_profit_train,
    "PROFITABILITY MODEL"
)


DELIVERY MODEL
Constant features : 3

Constant columns:
 - product_status
 - price_discount_amount
 - price_discount_pct

PROFITABILITY MODEL
Constant features : 3

Constant columns:
 - product_status
 - price_discount_amount
 - price_discount_pct


A constant-feature audit was performed on both training matrices to identify features with no variation that would provide no useful information to predictive models.

<h2 style="color: #548235; font-weight: 700;">Constant Feature Validation</h2>

| Model | Constant Features |
|---|---:|
| **Delivery Delay** | **3** |
| **Profitability Risk** | **3** |

**Constant features identified in both models:**

- `product_status`
- `price_discount_amount`
- `price_discount_pct`

<h2 style="color: #548235; font-weight: 700;">Key Observations</h2>

- Both modeling datasets contain the same **3 constant features**.
- These features have only one unique value throughout the training data.
- Constant features cannot provide meaningful variation for predictive learning.

<h2 style="color: #7F6000; font-weight: 700;">Preprocessing Implication</h2>

The identified constant features should be **excluded from the final modeling inputs** or removed during preprocessing to reduce unnecessary feature dimensionality and simplify the model matrices.

<h3 style="color: #7F6000; font-weight: 700;">**Status: ⚠️ CONSTANT FEATURES IDENTIFIED — 3 FEATURES REQUIRE REMOVAL**</h3>

### Why We're Performing These Audits Before Creating the Pipeline

The preprocessing architecture follows a strict **training-only fitting strategy**:

```text
                 TRAINING DATA
                      │
          ┌───────────┴───────────┐
          │                       │
       Numeric                Categorical
          │                       │
    Median Impute           One-Hot Encode
          │                  unknown → ignore
       Scaling                    │
          │                       │
          └───────────┬───────────┘
                      │
                   Boolean
                      │
                      ▼
             Combined Sparse Matrix
                      │
                      ▼
                  ML MODEL
                      │
                      ▼
             January 2018 TEST

The January 2018 test set never participates in fitting any preprocessing transformation. It only receives transformations learned from the training data.

### High-Cardinality Consideration

The dataset contains 41 categorical features, including several high-cardinality variables.

The categorical cardinality audit therefore helps determine whether to:

- Use straightforward one-hot encoding, or
- Apply a controlled treatment to very high-cardinality features.

This decision will be made before constructing the final preprocessing pipeline, avoiding unnecessary expansion of the model matrix.

**Key Principle**: Preprocessing must remain reproducible, leakage-safe, and computationally controlled.

<h1 style="color: #1F4E78; font-weight: 700;">8.2 — Feature Quality Refinement</h1>
<h2 style="color: #1F4E78; font-weight: 700;">Objective</h2>

The objective of Part 8B is to perform a final **feature-quality audit and refinement** before constructing the machine learning preprocessing pipelines.

The modeling matrices created in Part 8A contain a mixture of numerical, categorical, and boolean variables. Before encoding these features, each variable must be verified to ensure that its data type accurately represents its analytical meaning.

This stage focuses on identifying:

- Constant features
- Numeric variables incorrectly stored as categorical values
- Missing-value patterns introduced during type correction
- Final feature-type consistency


## 1. Constant Feature Detection

Features with only one unique value across the training dataset contain no predictive variation.

Such variables cannot help a machine learning model distinguish between target classes and therefore provide no meaningful predictive information.

Constant features identified during the training-data audit are removed from both the training and future-test matrices to maintain a consistent feature space.

The constant features identified in the current modeling dataset are:

- `product_status`
- `price_discount_amount`
- `price_discount_pct`

These features are removed before preprocessing.


## 2. Numeric-Like Categorical Detection

The feature store contains several market-movement variables that were initially represented as categorical/string values even though they contain continuous numerical measurements.

The following external-market features are specifically audited:

- `energy_price_change_pct`
- `metals_price_change_pct`
- `agriculture_price_change_pct`
- `food_price_change_pct`
- `industrial_materials_price_change_pct`

Treating these variables as categorical values would cause the model to interpret each percentage value as a separate category rather than as a continuous market movement.

Therefore, these fields are converted to numerical data types.



## 3. Handling Undefined Percentage Changes

The commodity movement features contain blank values for the initial period where a percentage change cannot be calculated because no preceding observation exists.

These blank values are converted to `NaN`.

They are intentionally not replaced manually at this stage.

The subsequent preprocessing pipeline will handle these missing numerical observations using a training-data-fitted imputation strategy.

This preserves the distinction between:

- A genuine zero percentage movement
- An undefined percentage movement caused by the absence of a prior observation



## 4. Feature-Type Validation

After the corrections, every estimator input feature must belong to one of the supported modeling types:

- Numerical
- Categorical
- Boolean

No unexpected or unsupported data types should remain.

The feature counts for the delivery-delay and profitability-risk models are validated independently because the two models have different leakage-control boundaries.



## 5. Modeling Governance Principle

Feature refinement is performed using the **training feature structure** as the modeling reference.

The goal is not to maximize the number of available variables, but to ensure that every retained feature:

1. Has meaningful variation
2. Has an appropriate data type
3. Represents the intended business concept
4. Can be safely transformed by the preprocessing pipeline
5. Does not violate the leakage controls established in Part 6


<h2 style="color: #1F4E78; font-weight: 700;">Expected Outcome</h2>

At the end of Part 8B:

- Constant features are removed.
- Numeric market-movement features are correctly represented as numerical variables.
- Undefined first-period market movements are represented as missing values.
- All remaining estimator features are classified as numerical, categorical, or boolean.
- The feature space is ready for high-cardinality analysis and preprocessing design.

This stage prepares the modeling matrices for the next step: **high-cardinality feature assessment**.

<h1 style="color: #1F4E78; font-weight: 700;">8.2.1 Remove Constant Features</h1>

In [35]:
# ============================================================
# Remove Constant Features
# ============================================================

constant_features = set(
    delivery_constant
).union(
    set(profit_constant)
)

print("CONSTANT FEATURE REMOVAL")
print("=" * 70)

print(f"Constant features identified : {len(constant_features)}")

for feature in sorted(constant_features):
    print(f" - {feature}")

# Remove constants from both modeling datasets.
X_delivery_train = X_delivery_train.drop(
    columns=list(constant_features),
    errors="ignore"
)

X_delivery_test = X_delivery_test.drop(
    columns=list(constant_features),
    errors="ignore"
)

X_profit_train = X_profit_train.drop(
    columns=list(constant_features),
    errors="ignore"
)

X_profit_test = X_profit_test.drop(
    columns=list(constant_features),
    errors="ignore"
)

print("\nUPDATED MODEL INPUTS")
print("-" * 70)
print(f"Delivery training features       : {X_delivery_train.shape[1]}")
print(f"Delivery test features           : {X_delivery_test.shape[1]}")
print(f"Profitability training features : {X_profit_train.shape[1]}")
print(f"Profitability test features     : {X_profit_test.shape[1]}")

print("\n✓ Constant features removed.")

CONSTANT FEATURE REMOVAL
Constant features identified : 3
 - price_discount_amount
 - price_discount_pct
 - product_status

UPDATED MODEL INPUTS
----------------------------------------------------------------------
Delivery training features       : 154
Delivery test features           : 154
Profitability training features : 153
Profitability test features     : 153

✓ Constant features removed.


Constant features identified during the quality audit were removed from both the Delivery Delay and Profitability Risk modeling datasets.

<h2 style="color: #548235; font-weight: 700;">Constant Feature Removal</h2>

| Removed Feature | Reason |
|---|---|
| `price_discount_amount` | Constant value |
| `price_discount_pct` | Constant value |
| `product_status` | Constant value |

**Total constant features removed: 3**

<h2 style="color: #548235; font-weight: 700;">Updated Model Inputs</h2>

| Model | Training Features | Test Features |
|---|---:|---:|
| **Delivery Delay** | **154** | **154** |
| **Profitability Risk** | **153** | **153** |

<h2 style="color: #7F6000; font-weight: 700;">Modeling Implication</h2>

Removing constant features reduces unnecessary dimensionality and ensures that the preprocessing pipeline focuses only on features with meaningful variation.

<h3 style="color: #7F6000; font-weight: 700;">**Status: ✅ CONSTANT FEATURES REMOVED — 3 FEATURES ELIMINATED FROM BOTH MODEL INPUTS**</h3>

<h1 style="color: #1F4E78; font-weight: 700;">8.2.2 Detect Numeric-Looking Categorical Features</h1>
<h3>Why this is important</h3>

*For example:*

- `energy_price_change_pct`
- `metals_price_change_pct`
- `fuel_price_change_pct`

represent quantitative market movements.

If they were accidentally stored as strings, treating them as categories would make the model learn:

"-0.23%" is a completely different category from "-0.24%"

instead of understanding that these are continuous measurements.

That would be a poor modeling representation.

In [36]:
# ============================================================
# Detect Numeric-Looking Categorical Features
# ============================================================

def detect_numeric_like_categories(
    X,
    categorical_features,
    model_name
):

    results = []

    for col in categorical_features:

        # Convert categorical values to numeric where possible
        converted = pd.to_numeric(
            X[col],
            errors="coerce"
        )

        conversion_rate = converted.notna().mean()

        # Flag features where >=95% of values are numeric
        if conversion_rate >= 0.95:

            results.append({
                "feature": col,
                "unique_values": X[col].nunique(
                    dropna=False
                ),
                "numeric_conversion_rate": round(
                    conversion_rate * 100,
                    2
                )
            })

    result_df = pd.DataFrame(results)

    print(f"\n{model_name}")
    print("=" * 70)

    if result_df.empty:

        print(
            "No numeric-like categorical features detected."
        )

    else:

        display(
            result_df.sort_values(
                "unique_values",
                ascending=False
            ).reset_index(drop=True)
        )

    return result_df


# ============================================================
# Recalculate Categorical Features
# Use the same dtype logic as PART 8A
# ============================================================

delivery_categorical_current = X_delivery_train.select_dtypes(
    include=[
        "object",
        "string",
        "str",
        "category"
    ]
).columns.tolist()

profit_categorical_current = X_profit_train.select_dtypes(
    include=[
        "object",
        "string",
        "str",
        "category"
    ]
).columns.tolist()


# ============================================================
# DELIVERY MODEL
# ============================================================

delivery_numeric_like = detect_numeric_like_categories(
    X_delivery_train,
    delivery_categorical_current,
    "DELIVERY MODEL"
)


# ============================================================
# PROFITABILITY MODEL
# ============================================================

profit_numeric_like = detect_numeric_like_categories(
    X_profit_train,
    profit_categorical_current,
    "PROFITABILITY MODEL"
)


DELIVERY MODEL


,feature,unique_values,numeric_conversion_rate
0,energy_price_change_pct,36,97.02
1,metals_price_change_pct,36,97.02
2,agriculture_price_change_pct,36,97.02
3,food_price_change_pct,36,97.02
4,industrial_materials_price_change_pct,36,97.02



PROFITABILITY MODEL


,feature,unique_values,numeric_conversion_rate
0,energy_price_change_pct,36,97.02
1,metals_price_change_pct,36,97.02
2,agriculture_price_change_pct,36,97.02
3,food_price_change_pct,36,97.02
4,industrial_materials_price_change_pct,36,97.02


A numeric-conversion audit was performed on the categorical features to identify variables that are stored as categorical data but actually represent **quantitative measurements**.

<h2 style="color: #548235; font-weight: 700;">Numeric-Looking Features Detected</h2>

| Feature | Unique Values | Numeric Conversion Rate |
|---|---:|---:|
| `energy_price_change_pct` | **36** | **97.02%** |
| `metals_price_change_pct` | **36** | **97.02%** |
| `agriculture_price_change_pct` | **36** | **97.02%** |
| `food_price_change_pct` | **36** | **97.02%** |
| `industrial_materials_price_change_pct` | **36** | **97.02%** |

The same five features were identified in both the **Delivery Delay** and **Profitability Risk** models.

<h2 style="color: #548235; font-weight: 700;">Key Observations</h2>

- **5 categorical features** are numeric-looking in both models.
- Each feature has **36 unique values** and a **97.02% numeric conversion rate**.
- These variables represent **percentage-based market price movements**, making numerical treatment more appropriate than categorical encoding.
- Treating values such as `-0.23` and `-0.24` as separate categories would lose their quantitative relationship.

<h2 style="color: #7F6000; font-weight: 700;">Modeling Implication</h2>

These five features should be **converted to numeric data types and processed through the numerical preprocessing pipeline** rather than one-hot encoded as categorical variables.

<h3 style="color: #7F6000; font-weight: 700;">**Status: ⚠️ NUMERIC-LIKE CATEGORIES IDENTIFIED — 5 FEATURES REQUIRE RECLASSIFICATION**</h3>

<h1 style="color: #1F4E78; font-weight: 700;">8.2.3 Inspect the Detected Features</h1>
This will let us determine whether these are genuinely numeric variables or legitimate categories.

In [37]:
# ============================================================
# Inspect Numeric-Like Feature Values
# ============================================================

numeric_like_features = sorted(
    set(delivery_numeric_like["feature"])
    .union(
        set(profit_numeric_like["feature"])
    )
)

print("NUMERIC-LIKE CATEGORICAL FEATURES")
print("=" * 70)

for col in numeric_like_features:

    print(f"\n{col}")
    print("-" * 50)

    print(
        X_delivery_train[col]
        .drop_duplicates()
        .sort_values()
        .head(15)
        .tolist()
    )

NUMERIC-LIKE CATEGORICAL FEATURES

agriculture_price_change_pct
--------------------------------------------------
['', '-0.00769620196249728', '-0.22153312407711587', '-0.5237928598598878', '-0.7451622568756523', '-0.8204902478827389', '-1.0264780973196572', '-1.172830142097575', '-1.4664634969056964', '-1.4799116141702937', '-1.9660912605247187', '-2.6209742016903332', '-2.677114571244597', '-2.8084578114687027', '-3.17873258078094']

energy_price_change_pct
--------------------------------------------------
['', '-0.5636688388396749', '-0.9524365669298951', '-1.8226232917445673', '-1.9895569825999837', '-2.6504526914818194', '-2.863474867601279', '-3.0679937314053376', '-3.1754055691324634', '-3.4139295151879767', '-3.788375584950128', '-4.128074078026453', '-5.269567765677528', '-6.3674914501793705', '-7.718982712825707']

food_price_change_pct
--------------------------------------------------
['', '-0.061421452862970315', '-0.29960884531620885', '-0.32701155275622323', '-0.453130

The five detected numeric-looking categorical features were inspected to confirm their underlying values and data representation.

<h2 style="color: #548235; font-weight: 700;">Key Observations</h2>

- All five features contain values represented as **strings**, including negative decimal values.
- Each feature also contains an empty-string value (`''`), indicating a non-numeric representation that requires handling.
- The values clearly represent **continuous percentage-change measurements**, rather than true categorical labels.
- The inspected features are:
  - `agriculture_price_change_pct`
  - `energy_price_change_pct`
  - `food_price_change_pct`
  - `industrial_materials_price_change_pct`
  - `metals_price_change_pct`

<h2 style="color: #7F6000; font-weight: 700;">Modeling Implication</h2>

These features should be **reclassified from categorical to numerical**, with empty strings handled appropriately before entering the numerical preprocessing pipeline.

<h3 style="color: #7F6000; font-weight: 700;">**Status:** ⚠️ Numeric-like features confirmed — reclassification and numeric conversion required.</h3>

<h1 style="color: #1F4E78; font-weight: 700;">8.2.4 Convert Numeric-Like Market Features</h1>

In [38]:
# ============================================================
# Convert Numeric-Like Categorical Features
# ============================================================

numeric_like_market_features = [
    "energy_price_change_pct",
    "metals_price_change_pct",
    "agriculture_price_change_pct",
    "food_price_change_pct",
    "industrial_materials_price_change_pct"
]

print("NUMERIC TYPE CORRECTION")
print("=" * 70)

for col in numeric_like_market_features:

    if col in X_delivery_train.columns:
        X_delivery_train[col] = pd.to_numeric(
            X_delivery_train[col].replace("", np.nan),
            errors="coerce"
        )

        X_delivery_test[col] = pd.to_numeric(
            X_delivery_test[col].replace("", np.nan),
            errors="coerce"
        )

    if col in X_profit_train.columns:
        X_profit_train[col] = pd.to_numeric(
            X_profit_train[col].replace("", np.nan),
            errors="coerce"
        )

        X_profit_test[col] = pd.to_numeric(
            X_profit_test[col].replace("", np.nan),
            errors="coerce"
        )

    print(f"✓ Converted: {col}")

print("\nNumeric type correction completed.")

NUMERIC TYPE CORRECTION
✓ Converted: energy_price_change_pct
✓ Converted: metals_price_change_pct
✓ Converted: agriculture_price_change_pct
✓ Converted: food_price_change_pct
✓ Converted: industrial_materials_price_change_pct

Numeric type correction completed.


The five identified market price movement features were converted from string-based categorical values into numerical data types.

<h2 style="color: #548235; font-weight: 700;">Key Observations</h2>

- All **5 numeric-like market features** were successfully converted.
- Empty-string values were replaced with `NaN`.
- Non-convertible values were safely coerced to `NaN`.
- The conversion was applied consistently to both **Delivery Delay** and **Profitability Risk** train/test datasets.
- The affected features are:
  - `energy_price_change_pct`
  - `metals_price_change_pct`
  - `agriculture_price_change_pct`
  - `food_price_change_pct`
  - `industrial_materials_price_change_pct`

<h2 style="color: #7F6000; font-weight: 700;">Modeling Implication</h2>

These market movement variables are now correctly positioned for **numerical preprocessing**, rather than categorical encoding, preserving their continuous quantitative meaning.

<h3 style="color: #7F6000; font-weight: 700;">**Status:** ✅ Numeric type correction completed successfully.</h3>

<h1 style="color: #1F4E78; font-weight: 700;">8.2.5 Revalidate Feature Types</h1>

In [39]:
# ============================================================
# Revalidate Feature Types
# ============================================================

# ------------------------------------------------------------
# DELIVERY MODEL
# ------------------------------------------------------------

delivery_numeric_final = X_delivery_train.select_dtypes(
    include=["number"]
).columns.tolist()

delivery_categorical_final = X_delivery_train.select_dtypes(
    include=[
        "object",
        "string",
        "str",
        "category"
    ]
).columns.tolist()

delivery_boolean_final = X_delivery_train.select_dtypes(
    include=["bool", "boolean"]
).columns.tolist()


# ------------------------------------------------------------
# PROFITABILITY MODEL
# ------------------------------------------------------------

profit_numeric_final = X_profit_train.select_dtypes(
    include=["number"]
).columns.tolist()

profit_categorical_final = X_profit_train.select_dtypes(
    include=[
        "object",
        "string",
        "str",
        "category"
    ]
).columns.tolist()

profit_boolean_final = X_profit_train.select_dtypes(
    include=["bool", "boolean"]
).columns.tolist()


# ============================================================
# FINAL FEATURE TYPE VALIDATION
# ============================================================

print("FINAL FEATURE TYPE VALIDATION")
print("=" * 70)


# ------------------------------------------------------------
# DELIVERY MODEL
# ------------------------------------------------------------

delivery_classified = (
    len(delivery_numeric_final)
    + len(delivery_categorical_final)
    + len(delivery_boolean_final)
)

print("\nDELIVERY MODEL")
print(f"Numeric features     : {len(delivery_numeric_final)}")
print(f"Categorical features : {len(delivery_categorical_final)}")
print(f"Boolean features     : {len(delivery_boolean_final)}")
print(f"Total classified     : {delivery_classified}")
print(f"Total features       : {X_delivery_train.shape[1]}")

if delivery_classified == X_delivery_train.shape[1]:
    print("✓ All delivery features classified successfully.")
else:
    print("⚠ Delivery feature classification mismatch.")


# ------------------------------------------------------------
# PROFITABILITY MODEL
# ------------------------------------------------------------

profit_classified = (
    len(profit_numeric_final)
    + len(profit_categorical_final)
    + len(profit_boolean_final)
)

print("\nPROFITABILITY MODEL")
print(f"Numeric features     : {len(profit_numeric_final)}")
print(f"Categorical features : {len(profit_categorical_final)}")
print(f"Boolean features     : {len(profit_boolean_final)}")
print(f"Total classified     : {profit_classified}")
print(f"Total features       : {X_profit_train.shape[1]}")

if profit_classified == X_profit_train.shape[1]:
    print("✓ All profitability features classified successfully.")
else:
    print("⚠ Profitability feature classification mismatch.")


# ============================================================
# NUMERIC-LIKE FEATURE VALIDATION
# ============================================================

numeric_like_market_features = [
    "energy_price_change_pct",
    "metals_price_change_pct",
    "agriculture_price_change_pct",
    "food_price_change_pct",
    "industrial_materials_price_change_pct"
]

print("\nNUMERIC-LIKE MARKET FEATURE VALIDATION")
print("=" * 70)

for col in numeric_like_market_features:

    delivery_dtype = X_delivery_train[col].dtype
    profit_dtype = X_profit_train[col].dtype

    print(
        f"{col:<40} "
        f"Delivery: {str(delivery_dtype):<12} "
        f"Profitability: {profit_dtype}"
    )

FINAL FEATURE TYPE VALIDATION

DELIVERY MODEL
Numeric features     : 106
Categorical features : 36
Boolean features     : 12
Total classified     : 154
Total features       : 154
✓ All delivery features classified successfully.

PROFITABILITY MODEL
Numeric features     : 105
Categorical features : 36
Boolean features     : 12
Total classified     : 153
Total features       : 153
✓ All profitability features classified successfully.

NUMERIC-LIKE MARKET FEATURE VALIDATION
energy_price_change_pct                  Delivery: float64      Profitability: float64
metals_price_change_pct                  Delivery: float64      Profitability: float64
agriculture_price_change_pct             Delivery: float64      Profitability: float64
food_price_change_pct                    Delivery: float64      Profitability: float64
industrial_materials_price_change_pct    Delivery: float64      Profitability: float64


A final feature-type validation was performed after converting the numeric-looking market features to numeric format.

<h2 style="color: #548235; font-weight: 700;">Key Observations</h2>

- **Delivery Model:** 106 numeric, 36 categorical, and 12 boolean features — **154/154 classified**.
- **Profitability Model:** 105 numeric, 36 categorical, and 12 boolean features — **153/153 classified**.
- No feature classification mismatches were detected.
- All five previously detected numeric-like market features are now correctly stored as **`float64`** in both models.
- The five corrected features are consistently typed across Delivery and Profitability datasets.

<h2 style="color: #7F6000; font-weight: 700;">Modeling Implication</h2>

The feature-type classification is now complete and consistent. All model inputs can be routed to the appropriate numerical, categorical, or boolean preprocessing pipelines.

<h3 style="color: #7F6000; font-weight: 700;">**Status:** ✅ Feature type validation PASSED — all features successfully classified and numeric-like market features correctly reclassified.</h3>

<h1 style="color: #1F4E78; font-weight: 700;">8.2.6 Verify the Market Features</h1>

In [40]:
# ============================================================
# Verify Market Movement Features
# ============================================================

print("MARKET MOVEMENT FEATURE VALIDATION")
print("=" * 70)

for col in numeric_like_market_features:

    print(
        f"{col:40s} "
        f"{str(X_delivery_train[col].dtype):10s} "
        f"missing={X_delivery_train[col].isna().sum():,}"
    )

print("\n✓ Market movement fields are now treated as numeric variables.")

MARKET MOVEMENT FEATURE VALIDATION
energy_price_change_pct                  float64    missing=5,322
metals_price_change_pct                  float64    missing=5,322
agriculture_price_change_pct             float64    missing=5,322
food_price_change_pct                    float64    missing=5,322
industrial_materials_price_change_pct    float64    missing=5,322

✓ Market movement fields are now treated as numeric variables.


The corrected market movement features were validated to confirm their final numerical data type and missing-value status.

<h2 style="color: #548235; font-weight: 700;">Key Observations</h2>

- All five market movement features are stored as **`float64`**.
- Each feature contains **5,322 missing values** in the Delivery training dataset.
- The missing-value count is consistent across all five features.
- The affected features are:
  - `energy_price_change_pct`
  - `metals_price_change_pct`
  - `agriculture_price_change_pct`
  - `food_price_change_pct`
  - `industrial_materials_price_change_pct`

<h2 style="color: #7F6000; font-weight: 700;">Modeling Implication</h2>

The market movement variables are now correctly treated as numerical features. Their remaining missing values can be handled by the **median-imputation step** in the numerical preprocessing pipeline.

<h2 style="color: #7F6000; font-weight: 700;">**Status:** ✅ Market movement feature validation PASSED — all five features are correctly typed as numeric.</h2>

<h1 style="color: #1F4E78; font-weight: 700;">8.3 — High-Cardinality Feature Audit</h1>

<h2 style="color: #1F4E78; font-weight: 700;">Objective</h2>

The objective of Part 8C is to evaluate categorical variables with a large number of unique values before applying one-hot encoding.

The modeling dataset contains geographic, customer, product, calendar, and business-category fields. Some categorical variables have substantially more unique values than others.

Blindly applying one-hot encoding to every categorical feature could create an unnecessarily large sparse feature matrix and may increase:

- Computational cost
- Memory usage
- Model complexity
- Risk of overfitting
- Difficulty of model interpretation

Therefore, categorical cardinality is explicitly audited before the final preprocessing architecture is created.

## 1. High-Cardinality Features

The primary fields requiring inspection include:

- `order_day`
- `order_city`
- `order_state`
- `customer_city`
- `customer_state`
- `holiday_name`

Their number of unique values is examined using the training dataset.

The audit is performed on the training period because the preprocessing and encoding vocabulary must be learned exclusively from historical training data.



## 2. Date-Level Feature Assessment

`order_day` requires special attention because it contains a large number of unique values.

A date-level variable can behave similarly to an identifier if individual historical dates are represented as separate categories.

Using such a field directly with one-hot encoding could allow a model to memorize historical dates instead of learning generalizable temporal relationships.

The project already contains engineered temporal features such as:

- Year
- Month
- Quarter
- Week number
- Day of week
- Month position
- Day type

Therefore, `order_day` is investigated separately before deciding whether it should remain in the estimator feature space.

The raw `order_datetime` field is already retained separately for:

- Temporal splitting
- Traceability
- Transaction-level analysis

and is not directly supplied to the estimator.



## 3. Geographic Cardinality Assessment

Geographic fields such as:

- `order_city`
- `order_state`
- `customer_city`
- `customer_state`

are evaluated based on their actual unique values rather than being removed solely because they have high cardinality.

Geographic information may contain meaningful operational and commercial signals related to:

- Delivery behavior
- Regional operations
- Customer distribution
- Market characteristics

Therefore, the decision to retain or transform a geographic field is based on its analytical meaning and modeling suitability.



## 4. Unknown Category Handling

The final categorical preprocessing pipeline will account for categories that may appear in the future holdout but were not present during training.

The categorical encoder will therefore use an unknown-category-safe configuration.

This ensures that the January 2018 future holdout can be transformed without requiring the model to have previously observed every possible category.



## 5. High-Cardinality Governance

The objective is not simply to minimize the number of features.

The objective is to achieve a balance between:

**Predictive information**

and

**Model complexity and generalization.**

Each high-cardinality variable is therefore assessed before being included in the final preprocessing pipeline.



## 6. Preprocessing Readiness

After the high-cardinality audit, the final categorical feature set will be established.

The resulting preprocessing architecture will use:

- Numerical preprocessing for continuous variables
- Categorical encoding for approved categorical variables
- Boolean conversion for boolean variables
- Training-only fitting of preprocessing transformations
- Unknown-category handling for future observations

The final encoded dimensionality will be measured before model training.



<h2 style="color: #1F4E78; font-weight: 700;">Expected Outcome</h2>

At the end of Part 8C:

- High-cardinality categorical variables have been explicitly assessed.
- Date-level features are evaluated for potential identifier-like behavior.
- Appropriate categorical variables are retained for encoding.
- The final categorical feature set is established.
- The modeling matrices are ready for construction of the leakage-safe preprocessing pipelines.

This completes the feature-quality and cardinality assessment required before predictive model training.

In [41]:
# ============================================================
# HIGH-CARDINALITY FEATURE AUDIT
# ============================================================

high_cardinality_features = [
    "order_day",
    "order_city",
    "order_state",
    "customer_city",
    "customer_state",
    "holiday_name"
]

print("HIGH-CARDINALITY FEATURE AUDIT")
print("=" * 70)

for col in high_cardinality_features:

    if col in X_delivery_train.columns:

        print(f"\n{col}")
        print("-" * 70)
        print(f"Unique values : {X_delivery_train[col].nunique(dropna=False)}")
        
        print("Sample values:")
        print(
            X_delivery_train[col]
            .drop_duplicates()
            .head(15)
            .tolist()
        )

HIGH-CARDINALITY FEATURE AUDIT

order_day
----------------------------------------------------------------------
Unique values : 1096
Sample values:
['2015-01-01', '2015-01-02', '2015-01-03', '2015-01-04', '2015-01-05', '2015-01-06', '2015-01-07', '2015-01-08', '2015-01-09', '2015-01-10', '2015-01-11', '2015-01-12', '2015-01-13', '2015-01-14', '2015-01-15']

order_city
----------------------------------------------------------------------
Unique values : 3596
Sample values:
['Mexico City', 'Dos Quebradas', 'São Paulo', 'Managua', 'Brasília', 'San Miguelito', 'San Cristóbal de Las Casas', 'Santiago de Chile', 'Panama City', 'Hermosillo', 'Coyoacán', 'Tegucigalpa', 'Artemisa', 'La Romana', 'Milpa Alta']

order_state
----------------------------------------------------------------------
Unique values : 1089
Sample values:
['Distrito Federal', 'Risaralda', 'São Paulo', 'Managua', 'Panamá', 'Chiapas', 'Santiago de Chile', 'Sonora', 'Francisco Morazán', 'Artemisa', 'La Romana', 'Querétaro', 

The selected categorical features were audited using the Delivery training dataset to identify variables with high category counts before final categorical encoding.

<h2 style="color: #548235; font-weight: 700;">Key Observations</h2>

- `order_city` has the highest cardinality with **3,596 unique values**.
- `order_day` contains **1,096 unique dates**, requiring separate assessment due to its date-level granularity.
- `order_state` contains **1,089 unique values**.
- `customer_city` contains **563 unique values**.
- `holiday_name` contains **227 unique values**.
- `customer_state` has comparatively lower cardinality with **46 unique values**.
- The geographic fields contain substantial variation and may provide meaningful operational and commercial information.
- `order_day` represents individual calendar dates and therefore requires caution before categorical encoding.

<h2 style="color: #7F6000; font-weight: 700;">Modeling Implication</h2>

High-cardinality features should not be removed automatically. Their predictive value and encoding impact must be considered before finalizing the categorical feature set.

`order_day` should be evaluated separately because one-hot encoding individual dates could introduce unnecessary dimensionality and date-specific memorization. Existing engineered temporal features provide more generalizable representations of time.

The final categorical preprocessing will use **unknown-category-safe encoding** so unseen categories in the January 2018 future holdout can be handled without preprocessing failure.

<h2 style="color: #7F6000; font-weight: 700;">**Status:** ⚠️ High-cardinality features identified — categorical encoding strategy requires explicit governance before pipeline construction.</h2>

<h1 style="color: #1F4E78; font-weight: 700;">8.3.1 Inspect Date Representation</h1>

Before deciding what to do with order_day, let's verify what it actually contains.

In [42]:
# ============================================================
# Inspect order_day Representation
# ============================================================

print("ORDER_DAY AUDIT")
print("=" * 70)

print("Data type:")
print(X_delivery_train["order_day"].dtype)

print("\nSample values:")
print(
    X_delivery_train["order_day"]
    .drop_duplicates()
    .head(20)
    .tolist()
)

print("\nUnique values:")
print(
    X_delivery_train["order_day"]
    .nunique(dropna=False)
)

ORDER_DAY AUDIT
Data type:
str

Sample values:
['2015-01-01', '2015-01-02', '2015-01-03', '2015-01-04', '2015-01-05', '2015-01-06', '2015-01-07', '2015-01-08', '2015-01-09', '2015-01-10', '2015-01-11', '2015-01-12', '2015-01-13', '2015-01-14', '2015-01-15', '2015-01-16', '2015-01-17', '2015-01-18', '2015-01-19', '2015-01-20']

Unique values:
1096


The `order_day` feature was inspected to determine whether its high cardinality represents meaningful categorical information or individual calendar dates.

<h2 style="color: #548235; font-weight: 700;">Key Observations</h2>

- `order_day` is stored as a **string (`str`)** data type.
- The values follow a **daily date format (`YYYY-MM-DD`)**.
- The feature contains **1,096 unique dates**.
- Sample values progress sequentially from `2015-01-01`, `2015-01-02`, `2015-01-03`, and so on.
- This confirms that `order_day` represents individual calendar dates rather than conventional categorical labels.

<h2 style="color: #7F6000; font-weight: 700;">Modeling Implication</h2>

`order_day` should **not be treated as a standard categorical variable for one-hot encoding**, as this could create a large number of date-specific categories and encourage historical date memorization.

The existing engineered temporal features should provide the model with more generalizable time-related information, while `order_datetime` remains available separately for temporal splitting and traceability.

<h2 style="color: #7F6000; font-weight: 700;">**Status:** ⚠️ `order_day` confirmed as a date-level feature — exclude from direct categorical encoding and rely on engineered temporal representations.</h2>

<h1 style="color: #1F4E78; font-weight: 700;">8.3.2 Finalize High-Cardinality Treatment</h1>

Assuming the audit confirms `order_day` is a date-level identifier, we'll remove it.

In [43]:
# ============================================================
# Remove Date-Level Identifier
# ============================================================

date_level_features = {
    "order_day"
}

X_delivery_train = X_delivery_train.drop(
    columns=list(date_level_features),
    errors="ignore"
)

X_delivery_test = X_delivery_test.drop(
    columns=list(date_level_features),
    errors="ignore"
)

X_profit_train = X_profit_train.drop(
    columns=list(date_level_features),
    errors="ignore"
)

X_profit_test = X_profit_test.drop(
    columns=list(date_level_features),
    errors="ignore"
)

print("DATE-LEVEL FEATURE TREATMENT")
print("=" * 70)

print("Removed from estimator inputs:")
for col in sorted(date_level_features):
    print(f"✓ {col}")

print("\nUpdated feature counts:")
print(f"Delivery       : {X_delivery_train.shape[1]}")
print(f"Profitability  : {X_profit_train.shape[1]}")

DATE-LEVEL FEATURE TREATMENT
Removed from estimator inputs:
✓ order_day

Updated feature counts:
Delivery       : 153
Profitability  : 152


Based on the date-level audit, `order_day` was removed from the estimator inputs to prevent date-specific memorization and unnecessary categorical dimensionality.

<h2 style="color: #548235; font-weight: 700;">Key Observations</h2>

- `order_day` was successfully removed from both predictive modeling datasets.
- The feature was removed from both **training and future holdout** matrices.
- Updated Delivery feature count: **153**.
- Updated Profitability feature count: **152**.
- Engineered temporal features remain available to capture generalizable time-related patterns.

<h2 style="color: #7F6000; font-weight: 700;">Modeling Implication</h2>

Removing the raw date-level categorical feature reduces unnecessary dimensionality while preserving temporal information through engineered features such as year, month, quarter, week, and day-related variables.

<h2 style="color: #7F6000; font-weight: 700;">**Status:** ✅ Date-level identifier treatment completed successfully.</h2>

<h1 style="color: #1F4E78; font-weight: 700;">8.4 — Leakage-Safe Preprocessing Pipeline</h1>

<h2 style="color: #1F4E78; font-weight: 700;">Objective</h2>

The objective of Part 8D is to convert the refined delivery-delay and profitability-risk feature sets into **consistent, machine-learning-ready representations** while preserving the temporal and leakage controls established in the previous stages.

At this stage, no predictive model is trained.

The focus is on building reusable preprocessing pipelines that can safely transform both the historical training data and the January 2018 future holdout.



## 1. Final Modeling Feature Space

Following the feature-quality refinement and high-cardinality assessment:

- Constant features have been removed.
- Numeric-like market movement variables have been converted to numerical data types.
- Undefined first-period commodity movements are represented as missing numerical observations.
- `order_day` has been removed from estimator inputs because it represents individual calendar dates and could encourage date-level memorization.
- `order_datetime` remains available separately for temporal splitting and transaction traceability.

The resulting estimator feature spaces contain:

- Numerical features
- Categorical features
- Boolean features

with separate feature boundaries for the delivery-delay and profitability-risk models.



## 2. Numerical Feature Processing

Numerical variables may contain missing observations, particularly external market movement features where a percentage change cannot be calculated for the first available observation.

Numerical preprocessing therefore consists of:

1. Median imputation
2. Standard scaling

Median imputation provides a robust method for handling missing numerical observations while reducing sensitivity to extreme values.

Standard scaling places numerical variables on a comparable scale and is particularly useful for linear models such as Logistic Regression.



## 3. Categorical Feature Processing

Categorical variables are transformed using one-hot encoding.

The encoder is configured with:

`handle_unknown="ignore"`

This is important because the January 2018 future holdout may contain categorical values that were not observed during the 2015–2017 training period.

Unknown categories will therefore be handled safely without causing transformation failures.

High-cardinality categorical variables have been explicitly audited before entering this stage.



## 4. Boolean Feature Processing

Boolean features are retained as a separate feature group and converted into a numerical representation suitable for machine learning algorithms.

The preprocessing architecture treats boolean variables independently from general categorical variables so that binary business indicators do not unnecessarily expand the feature space through one-hot encoding.



## 5. ColumnTransformer Architecture

A `ColumnTransformer` is used to apply different preprocessing operations to different feature groups.

The architecture is:

Training Feature Matrix
→ Numerical Pipeline
→ Categorical Pipeline
→ Boolean Pipeline
→ Combined Model Matrix

This provides a single reproducible transformation framework for each predictive use case.

Separate preprocessors are maintained for:

- Delivery Delay Prediction
- Profitability Risk Prediction

because the two models have different feature boundaries.



## 6. Leakage Prevention

A critical modeling requirement is that preprocessing must not learn information from the future holdout.

Therefore:

### Training

The preprocessing pipelines are fitted exclusively using the **2015–2017 training data**.

### Future Holdout

The fitted preprocessing pipelines are then applied to the **January 2018 future holdout** without refitting.

This prevents future information from influencing:

- Numerical imputation statistics
- Feature scaling parameters
- Categorical vocabularies
- One-hot encoding structure

The January 2018 data therefore remains genuinely unseen during the complete preprocessing-fit stage.



## 7. Sparse Feature Representation

One-hot encoding can substantially increase the number of columns because categorical variables may contain many unique values.

The resulting matrices are therefore maintained in a sparse representation where appropriate.

The final encoded dimensionality will be measured after transformation.

This provides an empirical assessment of the actual model feature space rather than relying only on the number of raw input columns.



## 8. Reproducible Pipeline Design

The preprocessing components are implemented using scikit-learn:

- `Pipeline`
- `ColumnTransformer`
- `SimpleImputer`
- `StandardScaler`
- `OneHotEncoder`

This architecture allows the same preprocessing logic to be reused consistently across multiple algorithms.

The eventual modeling architecture will therefore follow:

**Preprocessing → Model → Prediction → Evaluation**

rather than manually preprocessing each model separately.



## 9. Preprocessing Validation

After fitting and transforming the data, the following checks will be performed:

- Training row count is preserved.
- Future-test row count is preserved.
- Training and test feature dimensions are identical.
- No unexpected feature loss occurs.
- Sparse matrix representation is validated where applicable.
- Preprocessing is confirmed to have been fitted only on training data.



<h2 style="color: #1F4E78; font-weight: 700;">Expected Outcome</h2>

At the end of Part 8D, SupplySight AI will have two validated preprocessing components:

<h2 style="color: #1F4E78; font-weight: 700;">Delivery Delay Pipeline</h2>

Designed to transform the leakage-controlled delivery prediction feature set.

<h2 style="color: #1F4E78; font-weight: 700;">Profitability Risk Pipeline</h2>

Designed to transform the leakage-controlled profitability prediction feature set.

Both pipelines will produce aligned model-ready matrices for the same time-aware train/test boundary established in Part 7.



<h2 style="color: #1F4E78; font-weight: 700;">Transition to Predictive Modeling</h2>

With feature engineering, leakage assessment, temporal splitting, feature refinement, cardinality assessment, and preprocessing now complete, the project is ready to move into predictive modeling.

The next stage will evaluate whether SupplySight AI can reliably predict:

1. **Delivery delays**
2. **Profitability risk**

using a consistent and leakage-controlled modeling framework.

The first predictive modeling stage will begin with **Notebook 06B — Delivery Delay Prediction**.

<h1 style="color: #1F4E78; font-weight: 700;">8.4.1 Final Feature-Type Lists</h1>

In [44]:
# ============================================================
# LEAKAGE-SAFE PREPROCESSING
# Final Feature-Type Lists
# ============================================================

# ------------------------------------------------------------
# DELIVERY MODEL
# ------------------------------------------------------------

delivery_numeric_final = X_delivery_train.select_dtypes(
    include=["number"]
).columns.tolist()

delivery_categorical_final = X_delivery_train.select_dtypes(
    include=[
        "object",
        "string",
        "str",
        "category"
    ]
).columns.tolist()

delivery_boolean_final = X_delivery_train.select_dtypes(
    include=["bool", "boolean"]
).columns.tolist()


# ------------------------------------------------------------
# PROFITABILITY MODEL
# ------------------------------------------------------------

profit_numeric_final = X_profit_train.select_dtypes(
    include=["number"]
).columns.tolist()

profit_categorical_final = X_profit_train.select_dtypes(
    include=[
        "object",
        "string",
        "str",
        "category"
    ]
).columns.tolist()

profit_boolean_final = X_profit_train.select_dtypes(
    include=["bool", "boolean"]
).columns.tolist()


# ============================================================
# FEATURE COUNT VALIDATION
# ============================================================

delivery_classified = (
    len(delivery_numeric_final)
    + len(delivery_categorical_final)
    + len(delivery_boolean_final)
)

profit_classified = (
    len(profit_numeric_final)
    + len(profit_categorical_final)
    + len(profit_boolean_final)
)


# ============================================================
# FINAL PREPROCESSING FEATURE TYPES
# ============================================================

print("FINAL PREPROCESSING FEATURE TYPES")
print("=" * 70)


print("\nDELIVERY MODEL")
print(f"Numeric     : {len(delivery_numeric_final)}")
print(f"Categorical : {len(delivery_categorical_final)}")
print(f"Boolean     : {len(delivery_boolean_final)}")
print(f"Total       : {X_delivery_train.shape[1]}")
print(f"Classified  : {delivery_classified}")

if delivery_classified == X_delivery_train.shape[1]:
    print("✓ Delivery feature types validated.")
else:
    print("⚠ Delivery feature classification mismatch.")


print("\nPROFITABILITY MODEL")
print(f"Numeric     : {len(profit_numeric_final)}")
print(f"Categorical : {len(profit_categorical_final)}")
print(f"Boolean     : {len(profit_boolean_final)}")
print(f"Total       : {X_profit_train.shape[1]}")
print(f"Classified  : {profit_classified}")

if profit_classified == X_profit_train.shape[1]:
    print("✓ Profitability feature types validated.")
else:
    print("⚠ Profitability feature classification mismatch.")


# ============================================================
# LEAKAGE-SAFE PREPROCESSING VALIDATION
# ============================================================

print("\nLEAKAGE-SAFE PREPROCESSING VALIDATION")
print("=" * 70)

print(
    f"Delivery feature matrix shape     : "
    f"{X_delivery_train.shape}"
)

print(
    f"Profitability feature matrix shape: "
    f"{X_profit_train.shape}"
)

print(
    f"Delivery features classified      : "
    f"{delivery_classified}/{X_delivery_train.shape[1]}"
)

print(
    f"Profitability features classified : "
    f"{profit_classified}/{X_profit_train.shape[1]}"
)

if (
    delivery_classified == X_delivery_train.shape[1]
    and
    profit_classified == X_profit_train.shape[1]
):
    print(
        "\n✓ All final features have been successfully "
        "classified."
    )
    print(
        "✓ Feature-type lists are ready for preprocessing."
    )
else:
    print(
        "\n⚠ Feature-type validation requires review."
    )

FINAL PREPROCESSING FEATURE TYPES

DELIVERY MODEL
Numeric     : 106
Categorical : 35
Boolean     : 12
Total       : 153
Classified  : 153
✓ Delivery feature types validated.

PROFITABILITY MODEL
Numeric     : 105
Categorical : 35
Boolean     : 12
Total       : 152
Classified  : 152
✓ Profitability feature types validated.

LEAKAGE-SAFE PREPROCESSING VALIDATION
Delivery feature matrix shape     : (178396, 153)
Profitability feature matrix shape: (178396, 152)
Delivery features classified      : 153/153
Profitability features classified : 152/152

✓ All final features have been successfully classified.
✓ Feature-type lists are ready for preprocessing.


The final estimator feature sets were classified into numerical, categorical, and boolean feature groups after constant-feature removal, market-feature type correction, and `order_day` exclusion.

<h2 style="color: #548235; font-weight: 700;">Key Observations</h2>

- **Delivery Model:** 106 numeric, 35 categorical, and 12 boolean features — **153/153 classified**.
- **Profitability Model:** 105 numeric, 35 categorical, and 12 boolean features — **152/152 classified**.
- Both models contain **178,396 training rows**.
- No feature classification mismatches were detected.
- The final feature-type lists are ready for construction of the preprocessing pipelines.

<h2 style="color: #7F6000; font-weight: 700;">Modeling Implication</h2>

The final feature space is now clearly separated into numerical, categorical, and boolean variables, allowing each group to receive its appropriate preprocessing treatment through `ColumnTransformer`.

The Delivery and Profitability pipelines maintain separate feature boundaries while using the same leakage-safe preprocessing architecture.

<h2 style="color: #7F6000; font-weight: 700;">**Status:** ✅ Feature-type validation PASSED — all 305 estimator features across both models are successfully classified and ready for preprocessing.</h2>

<h1 style="color: #1F4E78; font-weight: 700;">8.4.2 Build Preprocessing Components</h1>

In [45]:
# ============================================================
# Define Preprocessing Components
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

# ------------------------------------------------------------
# Numerical preprocessing
# ------------------------------------------------------------

numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)


# ------------------------------------------------------------
# Categorical preprocessing
# ------------------------------------------------------------

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            )
        )
    ]
)


# ------------------------------------------------------------
# Boolean preprocessing
# ------------------------------------------------------------

boolean_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        )
    ]
)


print("PREPROCESSING COMPONENTS")
print("=" * 70)

print("✓ Numerical pipeline")
print("  - Median imputation")
print("  - Standard scaling")

print("\n✓ Categorical pipeline")
print("  - Most-frequent imputation")
print("  - One-hot encoding")
print("  - Unknown categories ignored")

print("\n✓ Boolean pipeline")
print("  - Most-frequent imputation")

PREPROCESSING COMPONENTS
✓ Numerical pipeline
  - Median imputation
  - Standard scaling

✓ Categorical pipeline
  - Most-frequent imputation
  - One-hot encoding
  - Unknown categories ignored

✓ Boolean pipeline
  - Most-frequent imputation


Reusable preprocessing components were defined for numerical, categorical, and boolean feature groups using scikit-learn pipelines.

<h2 style="color: #548235; font-weight: 700;">Key Observations</h2>

- **Numerical pipeline**
  - Median imputation for missing values.
  - Standard scaling for numerical variables.
- **Categorical pipeline**
  - Most-frequent imputation.
  - One-hot encoding.
  - Unknown categories are ignored using `handle_unknown="ignore"`.
- **Boolean pipeline**
  - Most-frequent imputation.
- The preprocessing components are designed to be reusable across both predictive modeling tasks.

<h2 style="color: #7F6000; font-weight: 700;">Modeling Implication</h2>

The preprocessing architecture provides separate, consistent transformations for each feature type and is ready to be integrated into model-specific `ColumnTransformer` pipelines.

The unknown-category handling is particularly important for safely transforming the **January 2018 future holdout** without requiring all categories to have been observed during training.

<h2 style="color: #7F6000; font-weight: 700;">**Status:** ✅ Preprocessing components successfully defined and ready for pipeline assembly.</h2>

<h1 style="color: #1F4E78; font-weight: 700;">8.4.3 Build Delivery Preprocessor</h1>

In [46]:
# ============================================================
# Delivery Delay Preprocessor
# ============================================================

delivery_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            delivery_numeric_final
        ),
        (
            "categorical",
            categorical_pipeline,
            delivery_categorical_final
        ),
        (
            "boolean",
            boolean_pipeline,
            delivery_boolean_final
        )
    ],
    remainder="drop"
)

print("✓ Delivery preprocessing architecture created.")

✓ Delivery preprocessing architecture created.


A dedicated `ColumnTransformer` was created for the Delivery Delay Prediction model by combining the numerical, categorical, and boolean preprocessing pipelines.

<h2 style="color: #548235; font-weight: 700;">Key Observations</h2>

- Numerical features are routed through the numerical preprocessing pipeline.
- Categorical features are routed through the categorical preprocessing pipeline.
- Boolean features are routed through the boolean preprocessing pipeline.
- `remainder="drop"` ensures that only the explicitly defined modeling features enter the transformed matrix.
- The Delivery preprocessing architecture was created successfully.

<h2 style="color: #7F6000; font-weight: 700;">Modeling Implication</h2>

The Delivery Delay model now has a structured preprocessing layer that applies consistent transformations by feature type while preventing unintended columns from entering the estimator.

<h2 style="color: #7F6000; font-weight: 700;">**Status:** ✅ Delivery preprocessing architecture created successfully.</h2>

<h1 style="color: #1F4E78; font-weight: 700;">8.4.4 Build Profitability Preprocessor</h1>

In [47]:
# ============================================================
# Profitability Risk Preprocessor
# ============================================================

profitability_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            profit_numeric_final
        ),
        (
            "categorical",
            categorical_pipeline,
            profit_categorical_final
        ),
        (
            "boolean",
            boolean_pipeline,
            profit_boolean_final
        )
    ],
    remainder="drop"
)

print("✓ Profitability preprocessing architecture created.")

✓ Profitability preprocessing architecture created.


A dedicated `ColumnTransformer` was created for the Profitability Risk Prediction model by combining the numerical, categorical, and boolean preprocessing pipelines.

<h2 style="color: #548235; font-weight: 700;">Key Observations</h2>

- Numerical features are routed through the numerical preprocessing pipeline.
- Categorical features are routed through the categorical preprocessing pipeline.
- Boolean features are routed through the boolean preprocessing pipeline.
- `remainder="drop"` ensures that only the explicitly defined modeling features enter the transformed matrix.
- The Profitability preprocessing architecture was created successfully.

<h2 style="color: #7F6000; font-weight: 700;">Modeling Implication</h2>

The Profitability Risk model now has a dedicated preprocessing layer that applies consistent feature-type transformations while maintaining the leakage-controlled feature boundary established earlier.

<h2 style="color: #7F6000; font-weight: 700;">**Status:** ✅ Profitability preprocessing architecture created successfully.</h2>

<h1 style="color: #1F4E78; font-weight: 700;">8.4.5 Fit on Training Data Only</h1>

This is the critical leakage-control step.

In [49]:
# ============================================================
# LEAKAGE-SAFE MODELING PREPROCESSING
# Handle Numeric, Categorical, and Boolean Features
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    FunctionTransformer
)


# ============================================================
# FINAL FEATURE-TYPE LISTS
# ============================================================

# ------------------------------------------------------------
# DELIVERY MODEL
# ------------------------------------------------------------

delivery_numeric_final = X_delivery_train.select_dtypes(
    include=["number"]
).columns.tolist()

delivery_categorical_final = X_delivery_train.select_dtypes(
    include=[
        "object",
        "string",
        "str",
        "category"
    ]
).columns.tolist()

delivery_boolean_final = X_delivery_train.select_dtypes(
    include=["bool", "boolean"]
).columns.tolist()


# ------------------------------------------------------------
# PROFITABILITY MODEL
# ------------------------------------------------------------

profit_numeric_final = X_profit_train.select_dtypes(
    include=["number"]
).columns.tolist()

profit_categorical_final = X_profit_train.select_dtypes(
    include=[
        "object",
        "string",
        "str",
        "category"
    ]
).columns.tolist()

profit_boolean_final = X_profit_train.select_dtypes(
    include=["bool", "boolean"]
).columns.tolist()


# ============================================================
# BOOLEAN FEATURES
# Convert boolean values to numeric 0/1
# ============================================================

boolean_to_numeric = FunctionTransformer(
    lambda X: X.astype("int8"),
    feature_names_out="one-to-one"
)


# ============================================================
# NUMERIC PIPELINE
# ============================================================

numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)


# ============================================================
# BOOLEAN PIPELINE
# ============================================================

boolean_pipeline = Pipeline(
    steps=[
        (
            "to_numeric",
            boolean_to_numeric
        ),
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        )
    ]
)


# ============================================================
# CATEGORICAL PIPELINE
# ============================================================

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)


# ============================================================
# DELIVERY PREPROCESSOR
# ============================================================

delivery_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            delivery_numeric_final
        ),
        (
            "categorical",
            categorical_pipeline,
            delivery_categorical_final
        ),
        (
            "boolean",
            boolean_pipeline,
            delivery_boolean_final
        )
    ],
    remainder="drop",
    verbose_feature_names_out=False
)


# ============================================================
# PROFITABILITY PREPROCESSOR
# ============================================================

profitability_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            profit_numeric_final
        ),
        (
            "categorical",
            categorical_pipeline,
            profit_categorical_final
        ),
        (
            "boolean",
            boolean_pipeline,
            profit_boolean_final
        )
    ],
    remainder="drop",
    verbose_feature_names_out=False
)


# ============================================================
# FIT PREPROCESSORS ON TRAINING DATA ONLY
# ============================================================

print("FITTING PREPROCESSORS")
print("=" * 70)

print("\nIMPORTANT:")
print("• Only training data is used during fit().")
print("• January 2018 test data is NOT used.")
print("• Numeric imputation statistics are learned from training data.")
print("• Categorical levels are learned from training data.")
print("• Boolean features are converted to numeric 0/1.")
print("• No information from the future test period is used.")


# ------------------------------------------------------------
# DELIVERY
# ------------------------------------------------------------

delivery_preprocessor.fit(
    X_delivery_train,
    y_delivery_train
)

print(
    "\n✓ Delivery preprocessor fitted "
    "on training data only."
)


# ------------------------------------------------------------
# PROFITABILITY
# ------------------------------------------------------------

profitability_preprocessor.fit(
    X_profit_train,
    y_profit_train
)

print(
    "✓ Profitability preprocessor fitted "
    "on training data only."
)


print(
    "✓ Future test data has not been used for fitting."
)

FITTING PREPROCESSORS

IMPORTANT:
• Only training data is used during fit().
• January 2018 test data is NOT used.
• Numeric imputation statistics are learned from training data.
• Categorical levels are learned from training data.
• Boolean features are converted to numeric 0/1.
• No information from the future test period is used.

✓ Delivery preprocessor fitted on training data only.
✓ Profitability preprocessor fitted on training data only.
✓ Future test data has not been used for fitting.


The preprocessing components were fitted exclusively on the **2015–2017 training data**, establishing the critical leakage-control boundary before transformation of the January 2018 future holdout.

<h2 style="color: #548235; font-weight: 700;">Key Observations</h2>

- The Delivery preprocessor was successfully fitted using training data only.
- The Profitability preprocessor was successfully fitted using training data only.
- Numerical imputation statistics were learned exclusively from the training period.
- Categorical levels were learned exclusively from the training period.
- Boolean variables are prepared for numerical representation.
- January 2018 test data was **not used during fitting**.
- Both preprocessors use `verbose_feature_names_out=False` for cleaner transformed feature names.

<h2 style="color: #7F6000; font-weight: 700;">Modeling Implication</h2>

The preprocessing stage preserves the established **time-aware and leakage-safe modeling boundary**. The January 2018 future holdout remains genuinely unseen during preprocessing fitting, preventing future information from influencing imputation, scaling, or categorical encoding.

<h2 style="color: #7F6000; font-weight: 700;">**Status:** ✅ Leakage control PASSED — both preprocessing pipelines were fitted on training data only.</h2>

<h1 style="color: #1F4E78; font-weight: 700;">8.4.6 Transform Train and Future Test</h1>
This will reveal how many actual model features we have after one-hot encoding.

In [50]:
# ============================================================
# Transform Training and Future Test Data
# ============================================================

X_delivery_train_encoded = delivery_preprocessor.transform(
    X_delivery_train
)

X_delivery_test_encoded = delivery_preprocessor.transform(
    X_delivery_test
)

X_profit_train_encoded = profitability_preprocessor.transform(
    X_profit_train
)

X_profit_test_encoded = profitability_preprocessor.transform(
    X_profit_test
)

print("ENCODED MODEL MATRICES")
print("=" * 70)

print("DELIVERY MODEL")
print(f"Training matrix : {X_delivery_train_encoded.shape}")
print(f"Test matrix     : {X_delivery_test_encoded.shape}")

print("\nPROFITABILITY MODEL")
print(f"Training matrix : {X_profit_train_encoded.shape}")
print(f"Test matrix     : {X_profit_test_encoded.shape}")

ENCODED MODEL MATRICES
DELIVERY MODEL
Training matrix : (178396, 6953)
Test matrix     : (2123, 6953)

PROFITABILITY MODEL
Training matrix : (178396, 6952)
Test matrix     : (2123, 6952)


The fitted preprocessing pipelines were applied to both the historical training data and the January 2018 future holdout to determine the final model-ready feature dimensionality after preprocessing and one-hot encoding.

<h2 style="color: #548235; font-weight: 700;">Key Observations</h2>

- **Delivery Model**
  - Training matrix: **178,396 × 6,953**
  - Future test matrix: **2,123 × 6,953**
- **Profitability Model**
  - Training matrix: **178,396 × 6,952**
  - Future test matrix: **2,123 × 6,952**
- Training and future-test matrices have identical feature dimensions within each model.
- One-hot encoding substantially expands the feature space from the original 153 and 152 estimator features.

<h2 style="color: #7F6000; font-weight: 700;">Modeling Implication</h2>

The preprocessing architecture successfully produces aligned model-ready matrices for both predictive tasks. The large increase in dimensionality reflects the categorical encoding of geographic, business, product, and other categorical variables.

The identical train/test dimensions confirm that the fitted training encoder can safely transform the January 2018 future holdout.

<h2 style="color: #7F6000; font-weight: 700;">**Status:** ✅ Transformation completed successfully — aligned sparse model matrices generated for both models.</h2>

<h1 style="color: #1F4E78; font-weight: 700;">8.4.7 Verify Train/Test Feature Alignment</h1>

In [51]:
# ============================================================
# Verify Encoded Matrix Alignment
# ============================================================

print("ENCODED MATRIX ALIGNMENT")
print("=" * 70)

delivery_dimensions_match = (
    X_delivery_train_encoded.shape[1]
    ==
    X_delivery_test_encoded.shape[1]
)

profit_dimensions_match = (
    X_profit_train_encoded.shape[1]
    ==
    X_profit_test_encoded.shape[1]
)

print(
    f"Delivery feature dimensions match     : "
    f"{delivery_dimensions_match}"
)

print(
    f"Profitability feature dimensions match: "
    f"{profit_dimensions_match}"
)

assert delivery_dimensions_match
assert profit_dimensions_match

print("\n✓ Delivery train/test feature space aligned.")
print("✓ Profitability train/test feature space aligned.")

ENCODED MATRIX ALIGNMENT
Delivery feature dimensions match     : True
Profitability feature dimensions match: True

✓ Delivery train/test feature space aligned.
✓ Profitability train/test feature space aligned.


The encoded training and January 2018 future-test matrices were validated to confirm that both datasets share the same model feature space within each predictive task.

<h2 style="color: #548235; font-weight: 700;">Key Observations</h2>

- Delivery train/test feature dimensions match: **True**.
- Profitability train/test feature dimensions match: **True**.
- Assertions passed for both predictive models.
- The Delivery matrices therefore maintain an identical encoded feature space.
- The Profitability matrices also maintain an identical encoded feature space.

<h2 style="color: #7F6000; font-weight: 700;">Modeling Implication</h2>

The fitted preprocessing pipelines successfully preserve consistent feature mappings between historical training data and the January 2018 future holdout. This confirms that the transformed matrices are structurally aligned and suitable for downstream model training and evaluation.

<h2 style="color: #7F6000; font-weight: 700;">**Status:** ✅ Train/test feature alignment PASSED for both Delivery Delay and Profitability Risk models.</h2>

<h1 style="color: #1F4E78; font-weight: 700;">8.4.8 Final Preprocessing Validation</h1>

In [52]:
# ============================================================
# Final Preprocessing Validation
# ============================================================

from scipy import sparse

print("FINAL PREPROCESSING VALIDATION")
print("=" * 70)

# ------------------------------------------------------------
# Delivery
# ------------------------------------------------------------

print("\nDELIVERY MODEL")
print("-" * 70)

print(
    f"Training rows preserved : "
    f"{X_delivery_train_encoded.shape[0] == len(y_delivery_train)}"
)

print(
    f"Test rows preserved     : "
    f"{X_delivery_test_encoded.shape[0] == len(y_delivery_test)}"
)

print(
    f"Feature dimensions match: "
    f"{X_delivery_train_encoded.shape[1] == X_delivery_test_encoded.shape[1]}"
)

print(
    f"Sparse matrix           : "
    f"{sparse.issparse(X_delivery_train_encoded)}"
)


# ------------------------------------------------------------
# Profitability
# ------------------------------------------------------------

print("\nPROFITABILITY MODEL")
print("-" * 70)

print(
    f"Training rows preserved : "
    f"{X_profit_train_encoded.shape[0] == len(y_profit_train)}"
)

print(
    f"Test rows preserved     : "
    f"{X_profit_test_encoded.shape[0] == len(y_profit_test)}"
)

print(
    f"Feature dimensions match: "
    f"{X_profit_train_encoded.shape[1] == X_profit_test_encoded.shape[1]}"
)

print(
    f"Sparse matrix           : "
    f"{sparse.issparse(X_profit_train_encoded)}"
)

print("\n✓ PREPROCESSING VALIDATION COMPLETE")

FINAL PREPROCESSING VALIDATION

DELIVERY MODEL
----------------------------------------------------------------------
Training rows preserved : True
Test rows preserved     : True
Feature dimensions match: True
Sparse matrix           : False

PROFITABILITY MODEL
----------------------------------------------------------------------
Training rows preserved : True
Test rows preserved     : True
Feature dimensions match: True
Sparse matrix           : False

✓ PREPROCESSING VALIDATION COMPLETE


The final preprocessing outputs were validated to confirm that row counts and feature dimensions are preserved consistently between training and future-test datasets.

<h2 style="color: #548235; font-weight: 700;">Key Observations</h2>

- **Delivery Model**
  - Training rows preserved: **True**
  - Test rows preserved: **True**
  - Feature dimensions match: **True**
  - Sparse matrix: **False**
- **Profitability Model**
  - Training rows preserved: **True**
  - Test rows preserved: **True**
  - Feature dimensions match: **True**
  - Sparse matrix: **False**
- All preprocessing validation checks passed successfully.
- The transformed matrices are currently represented as **dense matrices**, despite one-hot encoding being configured with sparse output.

<h2 style="color: #7F6000; font-weight: 700;">Modeling Implication</h2>

The preprocessing process preserves the complete training and future-test row counts and maintains identical feature dimensions within each model.

The `Sparse matrix: False` result confirms that the final `ColumnTransformer` output is dense. This should be considered before model training because the encoded feature space contains approximately **6,952–6,953 features** and may have higher memory requirements in dense form.


<h2 style="color: #7F6000; font-weight: 700;">**Status:** ✅ Final preprocessing validation PASSED — row preservation and train/test feature alignment confirmed for both models.</h2>

<h2 style="color: #1F4E78; font-weight: 700;">8.4 – Final Preprocessing Results</h2>

The preprocessing stage successfully converted the leakage-controlled estimator features into aligned model-ready representations for both predictive objectives.

<h2 style="color: #548235; font-weight: 700;">Key Observations</h2>

- Delivery Delay: **153 raw features → 6,953 encoded features**
- Profitability Risk: **152 raw features → 6,952 encoded features**
- Both models contain **178,396 training rows** and **2,123 future-test rows**.
- Train/test feature dimensions are fully aligned.
- Training and test row counts are preserved.
- Both preprocessors were fitted using **training data only**, maintaining the temporal leakage boundary.
- The increase from approximately 150 raw features to approximately 6,950 encoded features is primarily driven by **categorical feature expansion**.

<h2 style="color: #7F6000; font-weight: 700;">Modeling Implication</h2>

The encoded feature space is high-dimensional but expected due to categorical expansion. Algorithm selection should therefore consider computational efficiency, memory usage, and suitability for high-dimensional encoded data.

A further preprocessing consideration is required because the transformed outputs are currently **dense matrices** despite `OneHotEncoder` being configured for sparse output. With approximately **6,950 features × 178,396 training observations**, this may create unnecessary memory overhead.

<h2 style="color: #548235; font-weight: 700;">Status</h2>

**✅ PREPROCESSING VALIDATED — TRAIN/TEST ALIGNMENT AND LEAKAGE CONTROLS PASSED**

**⚠️ MEMORY REVIEW REQUIRED — DENSE ENCODED MATRICES SHOULD BE ADDRESSED BEFORE MODEL TRAINING**

<h1 style="color: #1F4E78; font-weight: 700;">8.5 — Final Preprocessing & Memory Validation</h1>

Before proceeding to model training, the final preprocessing outputs must be validated not only for correctness but also for computational feasibility.

The preprocessing pipeline has transformed the leakage-controlled raw feature sets into model-ready matrices containing numerical, categorical, and boolean features. Because categorical variables are one-hot encoded, the resulting feature space is substantially larger than the original feature space.

This section evaluates the actual representation and memory footprint of the encoded matrices to ensure that the selected dense or sparse representation is appropriate for the dataset size and subsequent model training.

<h2 style="color: #1F4E78; font-weight: 700;">Objectives</h2>

The objectives of this section are to:

1. Inspect the output configuration of the preprocessing transformers.
2. Determine whether the encoded matrices are dense or sparse.
3. Inspect the resulting data types and dimensions.
4. Estimate the memory footprint of the training and test matrices.
5. Assess whether the current representation is computationally practical.
6. Identify whether a sparse representation is required before model training.
7. Establish the final preprocessing representation that will be used consistently across all predictive models.

<h2 style="color: #1F4E78; font-weight: 700;">Why Memory Validation Is Important</h2>

One-hot encoding can significantly increase the dimensionality of the dataset, particularly when categorical features contain many unique values.

Therefore, the number of rows and encoded features alone is not sufficient to determine whether the model input is suitable for training. A dense matrix can require substantially more memory than a sparse matrix when most encoded values are zero.

The validation will therefore examine:

- matrix type
- matrix shape
- numerical data type
- memory consumption
- sparsity/density where applicable

<h2 style="color: #1F4E78; font-weight: 700;">Representation Decision</h2>

If the encoded matrices are confirmed to be computationally expensive in dense form, the preprocessing pipeline will be revised to produce sparse matrices while preserving the same leakage-safe preprocessing logic.

Any change to the representation must remain compatible with the downstream machine-learning algorithms and must be applied consistently to both training and future holdout datasets.

<h2 style="color: #1F4E78; font-weight: 700;">Validation Principle</h2>

No predictive model will be trained until the final feature representation has been validated for:

- dimensional consistency
- row preservation
- absence of unintended data leakage
- appropriate numerical representation
- computational feasibility

This ensures that the modeling stage begins with a stable, reproducible, and enterprise-ready analytical dataset.

<h1 style="color: #1F4E78; font-weight: 700;">Inspect Transformer Output Configuration</h1>

In [53]:
# ============================================================
# FINAL PREPROCESSING & MEMORY VALIDATION
# Inspect Output Configuration
# ============================================================

print("PREPROCESSOR OUTPUT CONFIGURATION")
print("=" * 70)

print("Delivery transformers:")
for name, transformer, columns in delivery_preprocessor.transformers_:
    print(f" - {name}: {type(transformer).__name__}")

print("\nProfitability transformers:")
for name, transformer, columns in profitability_preprocessor.transformers_:
    print(f" - {name}: {type(transformer).__name__}")

print("\nEncoded matrix types:")
print(
    f"Delivery      : {type(X_delivery_train_encoded).__name__}"
)

print(
    f"Profitability : {type(X_profit_train_encoded).__name__}"
)

PREPROCESSOR OUTPUT CONFIGURATION
Delivery transformers:
 - numeric: Pipeline
 - categorical: Pipeline
 - boolean: Pipeline

Profitability transformers:
 - numeric: Pipeline
 - categorical: Pipeline
 - boolean: Pipeline

Encoded matrix types:
Delivery      : ndarray
Profitability : ndarray


The final preprocessing configuration was inspected for both predictive models.

<h2 style="color: #548235; font-weight: 700;">Key Observations</h2>

- Both Delivery Delay and Profitability Risk preprocessors use dedicated **Pipeline** transformers for:
  - Numerical features
  - Categorical features
  - Boolean features
- The encoded outputs for both models are currently represented as **NumPy `ndarray` objects**.
- This confirms that the current `ColumnTransformer` configuration is producing **dense matrices**, consistent with the previous preprocessing validation.

<h2 style="color: #7F6000; font-weight: 700;">Modeling Implication</h2>

The dense representation should be evaluated for its actual memory footprint before model training. Given the approximately **6,950 encoded features** and **178,396 training observations**, a sparse representation may be more computationally efficient if the current dense matrices require excessive memory.

<h2 style="color: #548235; font-weight: 700;">Status</h2>

**⚠️ DENSE OUTPUT CONFIRMED — MEMORY FOOTPRINT VALIDATION REQUIRED BEFORE MODEL TRAINING**

<h1 style="color: #1F4E78; font-weight: 700;">Estimate Memory Footprint</h1>

In [54]:
# ============================================================
# Encoded Matrix Memory Audit
# ============================================================

def matrix_memory_mb(matrix):
    return matrix.nbytes / (1024 ** 2)


print("ENCODED MATRIX MEMORY AUDIT")
print("=" * 70)

print(
    f"Delivery training matrix memory      : "
    f"{matrix_memory_mb(X_delivery_train_encoded):,.2f} MB"
)

print(
    f"Delivery test matrix memory          : "
    f"{matrix_memory_mb(X_delivery_test_encoded):,.2f} MB"
)

print(
    f"Profitability training matrix memory : "
    f"{matrix_memory_mb(X_profit_train_encoded):,.2f} MB"
)

print(
    f"Profitability test matrix memory     : "
    f"{matrix_memory_mb(X_profit_test_encoded):,.2f} MB"
)

ENCODED MATRIX MEMORY AUDIT
Delivery training matrix memory      : 9,463.40 MB
Delivery test matrix memory          : 112.62 MB
Profitability training matrix memory : 9,462.04 MB
Profitability test matrix memory     : 112.60 MB


The memory footprint of the encoded matrices was measured to assess whether the current dense representation is computationally practical.

<h2 style="color: #548235; font-weight: 700;">Key Observations</h2>

- Delivery training matrix: **9,463.40 MB (~9.24 GB)**
- Delivery future-test matrix: **112.62 MB**
- Profitability training matrix: **9,462.04 MB (~9.24 GB)**
- Profitability future-test matrix: **112.60 MB**
- The training matrices require approximately **9.2 GB each** in dense form.
- The similar memory footprints across both models are expected because their encoded feature dimensions differ by only one feature.

<h2 style="color: #7F6000; font-weight: 700;">Modeling Implication</h2>

The dense representation is **memory-intensive** and is not ideal for repeated model training, experimentation, or production-scale workflows.

A **sparse matrix representation should therefore be evaluated and adopted if compatible with the selected downstream algorithms**, while preserving the existing leakage-safe preprocessing architecture.

<h2 style="color: #548235; font-weight: 700;">Status</h2>

**⚠️ DENSE REPRESENTATION NOT COMPUTATIONALLY EFFICIENT — SPARSE REPRESENTATION SHOULD BE EVALUATED BEFORE MODEL TRAINING**

<h2 style="color: #548235; font-weight: 700;">8.5.1 — Memory Validation Findings</h2>

The preprocessing audit confirms that both model inputs are currently represented as dense NumPy arrays (`ndarray`).

The encoded feature space contains approximately 6,953 features for the delivery model and 6,952 features for the profitability model. With 178,396 training observations, the resulting dense matrices require approximately:

- Delivery training matrix: 9.46 GB
- Profitability training matrix: 9.46 GB
- Delivery future-test matrix: 112.62 MB
- Profitability future-test matrix: 112.60 MB

The training matrices are therefore considerably larger than the original feature store and are not an efficient representation for a high-cardinality one-hot encoded dataset.

<h2 style="color: #548235; font-weight: 700;">Enterprise Design Decision</h2>

The current dense representation will **not** be used for model training.

The preprocessing architecture will be revised to produce a **sparse encoded representation**, which is more appropriate for one-hot encoded categorical features because the majority of encoded values are expected to be zero.

The revised preprocessing pipeline must preserve:

- the same leakage controls
- the same time-aware train/test split
- training-only fitting of preprocessing components
- median imputation for numerical variables
- categorical imputation
- unknown-category handling
- consistent feature dimensions between training and future-test data

The numerical scaling component will also be configured to remain compatible with sparse matrices.

<h2 style="color: #548235; font-weight: 700;">Validation Gate</h2>

**Current preprocessing representation: FAIL — computationally inefficient**

**Required action:** Rebuild the preprocessing pipeline using a sparse-compatible configuration and revalidate:

1. Training/test row counts
2. Encoded feature dimensions
3. Sparse matrix representation
4. Memory footprint
5. Train/test feature consistency

Model training will begin only after this validation gate passes.

<h1 style="color: #1F4E78; font-weight: 700;">8.5.2 — Sparse-Compatible Preprocessing Reconstruction</h1>

The memory audit demonstrated that the current dense encoded matrices are computationally inefficient for the high-dimensional feature space generated by one-hot encoding.

To resolve this issue, the preprocessing architecture will now be reconstructed using a **sparse-compatible representation**.

The revised pipeline preserves the previously established feature definitions, leakage controls, time-aware train/test split, and training-only fitting strategy. Only the computational representation of the transformed feature matrices is being optimized.

<h2 style="color: #1F4E78; font-weight: 700;">Sparse Preprocessing Strategy</h2>

The revised preprocessing pipeline will:

- retain median imputation for numerical features
- use `StandardScaler(with_mean=False)` for sparse compatibility
- retain most-frequent imputation for categorical features
- retain `OneHotEncoder(handle_unknown="ignore")`
- explicitly generate sparse one-hot encoded features
- configure the `ColumnTransformer` to preserve sparse output
- retain boolean features through the existing imputation strategy

The preprocessors will again be fitted **exclusively on the 2015–2017 training period** and subsequently applied to the January 2018 future holdout.

After reconstruction, the encoded matrices will be revalidated for matrix type, dimensional consistency, sparsity, and memory consumption before predictive modeling begins.

<h2 style="color: #1F4E78; font-weight: 700;">Release Dense Encoded Matrices</h2>

In [55]:
# ============================================================
# Release Dense Encoded Matrices
# ============================================================

import gc

dense_objects = [
    "X_delivery_train_encoded",
    "X_delivery_test_encoded",
    "X_profit_train_encoded",
    "X_profit_test_encoded"
]

for obj in dense_objects:
    if obj in globals():
        del globals()[obj]

gc.collect()

print("Dense encoded matrices released from memory.")

Dense encoded matrices released from memory.


The memory audit confirmed that the dense encoded matrices are computationally expensive, requiring approximately **9.2 GB per training matrix**. Therefore, the preprocessing architecture is being reconstructed using a sparse-compatible representation.

<h2 style="color: #548235; font-weight: 700;">Sparse Preprocessing Strategy</h2>

The revised preprocessing design preserves the existing feature definitions, leakage controls, and time-aware modeling boundary while optimizing only the matrix representation.

- Numerical features retain **median imputation** with `StandardScaler(with_mean=False)` for sparse compatibility.
- Categorical features retain **most-frequent imputation** and `OneHotEncoder(handle_unknown="ignore")`.
- One-hot encoded features are explicitly maintained in **sparse form**.
- `ColumnTransformer` will be configured to preserve sparse output.
- Boolean features retain the existing imputation strategy.
- Preprocessors will be fitted **only on 2015–2017 training data** and applied to the **January 2018 future holdout**.

<h2 style="color: #548235; font-weight: 700;">Memory Management</h2>

The previously generated dense encoded matrices were successfully released from memory using garbage collection before reconstructing the sparse-compatible preprocessing pipeline.

<h2 style="color: #7F6000; font-weight: 700;">Modeling Implication</h2>

The revised representation is intended to substantially reduce memory consumption while preserving the same encoded feature space and leakage-safe preprocessing logic.

The sparse outputs must be revalidated for **matrix type, dimensional consistency, sparsity, row preservation, and memory footprint** before model training.

<h2 style="color: #548235; font-weight: 700;">Status</h2>

**✅ DENSE MATRICES RELEASED — MEMORY CLEARED**

**🔄 SPARSE-COMPATIBLE PREPROCESSING RECONSTRUCTION IN PROGRESS**

<h2 style="color: #1F4E78; font-weight: 700;">Recover Final Feature-Type Lists</h2>

In [58]:
# ============================================================
# Recover Final Feature-Type Lists
# ============================================================

# Recreate the final feature-type lists from the current
# leakage-controlled modeling datasets.

# ============================================================
# DELIVERY MODEL
# ============================================================

delivery_numeric_features = X_delivery_train.select_dtypes(
    include=["number"]
).columns.tolist()

delivery_boolean_features = X_delivery_train.select_dtypes(
    include=["bool", "boolean"]
).columns.tolist()

delivery_categorical_features = X_delivery_train.select_dtypes(
    include=[
        "object",
        "string",
        "str",
        "category"
    ]
).columns.tolist()


# ============================================================
# PROFITABILITY MODEL
# ============================================================

profitability_numeric_features = X_profit_train.select_dtypes(
    include=["number"]
).columns.tolist()

profitability_boolean_features = X_profit_train.select_dtypes(
    include=["bool", "boolean"]
).columns.tolist()

profitability_categorical_features = X_profit_train.select_dtypes(
    include=[
        "object",
        "string",
        "str",
        "category"
    ]
).columns.tolist()


# ============================================================
# FEATURE COUNTS
# ============================================================

delivery_total = (
    len(delivery_numeric_features)
    + len(delivery_categorical_features)
    + len(delivery_boolean_features)
)

profitability_total = (
    len(profitability_numeric_features)
    + len(profitability_categorical_features)
    + len(profitability_boolean_features)
)


# ============================================================
# FINAL FEATURE-TYPE RECOVERY
# ============================================================

print("FINAL FEATURE-TYPE RECOVERY")
print("=" * 70)

print("\nDelivery:")
print("  Numeric    :", len(delivery_numeric_features))
print("  Categorical:", len(delivery_categorical_features))
print("  Boolean    :", len(delivery_boolean_features))
print("  Total      :", delivery_total)
print("  Matrix     :", X_delivery_train.shape[1])

if delivery_total == X_delivery_train.shape[1]:
    print("  ✓ All delivery features classified.")
else:
    print("  ⚠ Delivery feature count mismatch.")


print("\nProfitability:")
print(
    "  Numeric    :",
    len(profitability_numeric_features)
)
print(
    "  Categorical:",
    len(profitability_categorical_features)
)
print(
    "  Boolean    :",
    len(profitability_boolean_features)
)
print("  Total      :", profitability_total)
print("  Matrix     :", X_profit_train.shape[1])

if profitability_total == X_profit_train.shape[1]:
    print("  ✓ All profitability features classified.")
else:
    print("  ⚠ Profitability feature count mismatch.")


# ============================================================
# FINAL VALIDATION
# ============================================================

print("\nFINAL VALIDATION")
print("=" * 70)

if (
    delivery_total == X_delivery_train.shape[1]
    and
    profitability_total == X_profit_train.shape[1]
):
    print("✓ Delivery feature types successfully recovered.")
    print("✓ Profitability feature types successfully recovered.")
    print("✓ No unclassified feature types detected.")
    print("✓ Feature lists are ready for preprocessing.")
else:
    print("⚠ Feature-type validation requires review.")

FINAL FEATURE-TYPE RECOVERY

Delivery:
  Numeric    : 106
  Categorical: 35
  Boolean    : 12
  Total      : 153
  Matrix     : 153
  ✓ All delivery features classified.

Profitability:
  Numeric    : 105
  Categorical: 35
  Boolean    : 12
  Total      : 152
  Matrix     : 152
  ✓ All profitability features classified.

FINAL VALIDATION
✓ Delivery feature types successfully recovered.
✓ Profitability feature types successfully recovered.
✓ No unclassified feature types detected.
✓ Feature lists are ready for preprocessing.


<h2 style="color: #1F4E78; font-weight: 700;">Rebuild Sparse-Compatible Pipelines</h2>

In [59]:
# ============================================================
# Sparse-Compatible Preprocessing Pipelines
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Numerical pipeline
numeric_pipeline_sparse = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler(with_mean=False))
    ]
)

# Categorical pipeline
categorical_pipeline_sparse = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            )
        )
    ]
)

# Boolean pipeline
boolean_pipeline_sparse = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent"))
    ]
)

# Delivery preprocessing
delivery_preprocessor_sparse = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline_sparse, delivery_numeric_features),
        ("categorical", categorical_pipeline_sparse, delivery_categorical_features),
        ("boolean", boolean_pipeline_sparse, delivery_boolean_features)
    ],
    sparse_threshold=1.0
)

# Profitability preprocessing
profitability_preprocessor_sparse = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline_sparse, profitability_numeric_features),
        ("categorical", categorical_pipeline_sparse, profitability_categorical_features),
        ("boolean", boolean_pipeline_sparse, profitability_boolean_features)
    ],
    sparse_threshold=1.0
)

print("Sparse-compatible preprocessors created.")
print()
print("Delivery raw features     :", len(delivery_numeric_features)
      + len(delivery_categorical_features)
      + len(delivery_boolean_features))

print("Profitability raw features:", len(profitability_numeric_features)
      + len(profitability_categorical_features)
      + len(profitability_boolean_features))

Sparse-compatible preprocessors created.

Delivery raw features     : 153
Profitability raw features: 152


The preprocessing pipelines were successfully reconstructed using a **sparse-compatible architecture** to reduce the memory overhead associated with high-dimensional one-hot encoded features.

<h2 style="color: #548235; font-weight: 700;">Key Observations</h2>

- Numerical features use **median imputation** with `StandardScaler(with_mean=False)`, preserving sparse compatibility.
- Categorical features use **most-frequent imputation** followed by `OneHotEncoder(handle_unknown="ignore", sparse_output=True)`.
- Boolean features retain **most-frequent imputation**.
- Both `ColumnTransformer` preprocessors use `sparse_threshold=1.0` to preserve sparse output.
- Delivery preprocessing contains **153 raw estimator features**.
- Profitability preprocessing contains **152 raw estimator features**.
- The feature definitions and established leakage controls remain unchanged.

<h2 style="color: #7F6000; font-weight: 700;">Modeling Implication</h2>

The reconstructed preprocessors are designed to retain the high-dimensional encoded feature space while avoiding the unnecessary memory cost of dense matrices. They are now ready to be fitted exclusively on the **2015–2017 training data** and applied to the **January 2018 future holdout**.

<h2 style="color: #548235; font-weight: 700;">Status</h2>

**✅ SPARSE-COMPATIBLE PREPROCESSORS CREATED — DELIVERY: 153 FEATURES | PROFITABILITY: 152 FEATURES**

<h2 style="color: #1F4E78; font-weight: 700;">Fit Only on Training Data and Transform</h2>

In [61]:
# ============================================================
# Rebuild Sparse Preprocessors with Boolean Handling
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    FunctionTransformer
)

# ------------------------------------------------------------
# Boolean → Numeric Conversion
# ------------------------------------------------------------

boolean_to_numeric = FunctionTransformer(
    lambda X: X.astype("int8"),
    feature_names_out="one-to-one"
)

# ------------------------------------------------------------
# Numeric Pipeline
# ------------------------------------------------------------

numeric_pipeline_sparse = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

# ------------------------------------------------------------
# Boolean Pipeline
# ------------------------------------------------------------

boolean_pipeline_sparse = Pipeline(
    steps=[
        (
            "to_numeric",
            boolean_to_numeric
        ),
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        )
    ]
)

# ------------------------------------------------------------
# Categorical Pipeline
# ------------------------------------------------------------

categorical_pipeline_sparse = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            )
        )
    ]
)

# ------------------------------------------------------------
# Delivery Model Preprocessor
# ------------------------------------------------------------

delivery_preprocessor_sparse = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline_sparse,
            delivery_numeric_features
        ),
        (
            "categorical",
            categorical_pipeline_sparse,
            delivery_categorical_features
        ),
        (
            "boolean",
            boolean_pipeline_sparse,
            delivery_boolean_features
        )
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

# ------------------------------------------------------------
# Profitability Model Preprocessor
# ------------------------------------------------------------

profitability_preprocessor_sparse = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline_sparse,
            profitability_numeric_features
        ),
        (
            "categorical",
            categorical_pipeline_sparse,
            profitability_categorical_features
        ),
        (
            "boolean",
            boolean_pipeline_sparse,
            profitability_boolean_features
        )
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

print("Sparse preprocessors rebuilt successfully.")
print()
print("Delivery numeric features     :", len(delivery_numeric_features))
print("Delivery categorical features :", len(delivery_categorical_features))
print("Delivery boolean features     :", len(delivery_boolean_features))
print()
print("Profitability numeric features     :", len(profitability_numeric_features))
print("Profitability categorical features :", len(profitability_categorical_features))
print("Profitability boolean features     :", len(profitability_boolean_features))

Sparse preprocessors rebuilt successfully.

Delivery numeric features     : 106
Delivery categorical features : 35
Delivery boolean features     : 12

Profitability numeric features     : 105
Profitability categorical features : 35
Profitability boolean features     : 12


In [62]:
# ============================================================
# Fit Sparse Preprocessors on Training Data
# ============================================================

# ------------------------------------------------------------
# Fit ONLY on Historical Training Data
# ------------------------------------------------------------

delivery_preprocessor_sparse.fit(
    X_delivery_train,
    y_delivery_train
)

profitability_preprocessor_sparse.fit(
    X_profit_train,
    y_profit_train
)

# ------------------------------------------------------------
# Transform Training and Future Holdout Data
# ------------------------------------------------------------

X_delivery_train_sparse = delivery_preprocessor_sparse.transform(
    X_delivery_train
)

X_delivery_test_sparse = delivery_preprocessor_sparse.transform(
    X_delivery_test
)

X_profit_train_sparse = profitability_preprocessor_sparse.transform(
    X_profit_train
)

X_profit_test_sparse = profitability_preprocessor_sparse.transform(
    X_profit_test
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("Sparse preprocessing completed successfully.")
print()

print("Delivery train shape     :", X_delivery_train_sparse.shape)
print("Delivery test shape      :", X_delivery_test_sparse.shape)

print(
    "Profitability train shape:",
    X_profit_train_sparse.shape
)

print(
    "Profitability test shape :",
    X_profit_test_sparse.shape
)

print()
print("Delivery output type     :", type(X_delivery_train_sparse))
print("Profitability output type:", type(X_profit_train_sparse))

Sparse preprocessing completed successfully.

Delivery train shape     : (178396, 6953)
Delivery test shape      : (2123, 6953)
Profitability train shape: (178396, 6952)
Profitability test shape : (2123, 6952)

Delivery output type     : <class 'scipy.sparse._csr.csr_matrix'>
Profitability output type: <class 'scipy.sparse._csr.csr_matrix'>


The sparse-compatible preprocessors were successfully fitted using **historical training data only** and then applied to both the training and January 2018 future holdout datasets.

<h2 style="color: #548235; font-weight: 700;">Key Observations</h2>

- Delivery preprocessing:
  - **178,396 training rows × 6,953 encoded features**
  - **2,123 future-test rows × 6,953 encoded features**
- Profitability preprocessing:
  - **178,396 training rows × 6,952 encoded features**
  - **2,123 future-test rows × 6,952 encoded features**
- Train/test dimensions are aligned for both predictive models.
- Both transformed outputs are **SciPy CSR sparse matrices**.
- The sparse representation successfully replaces the previously memory-intensive dense `ndarray` outputs.
- Preprocessing remains fitted exclusively on the **2015–2017 training period**, preserving the established temporal leakage boundary.

<h2 style="color: #7F6000; font-weight: 700;">Modeling Implication</h2>

The sparse representation preserves the full encoded feature space while substantially improving memory efficiency. The transformed matrices are now suitable for further memory validation and downstream model training, subject to algorithm compatibility with sparse input.

<h2 style="color: #548235; font-weight: 700;">Status</h2>

**✅ SPARSE PREPROCESSING SUCCESSFULLY FITTED AND TRANSFORMED**

**✅ TRAIN/FUTURE-TEST DIMENSIONS ALIGNED**

**✅ CSR SPARSE REPRESENTATION CONFIRMED**

**✅ TRAINING-ONLY FITTING STRATEGY PRESERVED**

<h2 style="color: #1F4E78; font-weight: 700;">Sparse Representation Validation</h2>

In [63]:
# ============================================================
# Sparse Representation Validation
# ============================================================

from scipy import sparse

print("SPARSE REPRESENTATION VALIDATION")
print("=" * 70)

print("Delivery train sparse      :", sparse.issparse(X_delivery_train_sparse))
print("Delivery test sparse       :", sparse.issparse(X_delivery_test_sparse))
print("Profitability train sparse :", sparse.issparse(X_profit_train_sparse))
print("Profitability test sparse  :", sparse.issparse(X_profit_test_sparse))

print("\nMatrix types:")
print("Delivery      :", type(X_delivery_train_sparse).__name__)
print("Profitability :", type(X_profit_train_sparse).__name__)

print("\nDIMENSION VALIDATION")
print("=" * 70)

print(
    "Delivery train/test feature match:",
    X_delivery_train_sparse.shape[1] ==
    X_delivery_test_sparse.shape[1]
)

print(
    "Profitability train/test feature match:",
    X_profit_train_sparse.shape[1] ==
    X_profit_test_sparse.shape[1]
)

print(
    "Delivery training rows preserved:",
    X_delivery_train_sparse.shape[0] == len(y_delivery_train)
)

print(
    "Profitability training rows preserved:",
    X_profit_train_sparse.shape[0] == len(y_profit_train)
)

SPARSE REPRESENTATION VALIDATION
Delivery train sparse      : True
Delivery test sparse       : True
Profitability train sparse : True
Profitability test sparse  : True

Matrix types:
Delivery      : csr_matrix
Profitability : csr_matrix

DIMENSION VALIDATION
Delivery train/test feature match: True
Profitability train/test feature match: True
Delivery training rows preserved: True
Profitability training rows preserved: True


The sparse representation was validated for both predictive models to confirm that the transformed matrices are correctly stored and dimensionally consistent.

<h2 style="color: #548235; font-weight: 700;">Key Observations</h2>

- Delivery training and future-test matrices are confirmed as **sparse**.
- Profitability training and future-test matrices are confirmed as **sparse**.
- Both models use **CSR (`csr_matrix`) representation**.
- Delivery train/test feature dimensions match.
- Profitability train/test feature dimensions match.
- Training row counts are fully preserved for both models.
- All validation checks returned **True**.

<h2 style="color: #7F6000; font-weight: 700;">Modeling Implication</h2>

The sparse representation is functioning as intended and maintains consistent feature spaces between historical training data and the January 2018 future holdout. The matrices are now suitable for downstream algorithms that support sparse input.

<h2 style="color: #548235; font-weight: 700;">Status</h2>

**✅ SPARSE REPRESENTATION VALIDATED — CSR FORMAT, DIMENSIONS, AND ROW PRESERVATION PASSED**

<h2 style="color: #1F4E78; font-weight: 700;">Sparse Memory & Density Audit</h2>

In [64]:
# ============================================================
# Sparse Memory & Density Audit
# ============================================================

def sparse_memory_mb(matrix):
    return (
        matrix.data.nbytes
        + matrix.indices.nbytes
        + matrix.indptr.nbytes
    ) / (1024 ** 2)


def matrix_density(matrix):
    total_cells = matrix.shape[0] * matrix.shape[1]
    return matrix.nnz / total_cells


print("SPARSE MATRIX MEMORY AUDIT")
print("=" * 70)

print(
    f"Delivery training matrix      : "
    f"{sparse_memory_mb(X_delivery_train_sparse):,.2f} MB"
)

print(
    f"Delivery test matrix          : "
    f"{sparse_memory_mb(X_delivery_test_sparse):,.2f} MB"
)

print(
    f"Profitability training matrix : "
    f"{sparse_memory_mb(X_profit_train_sparse):,.2f} MB"
)

print(
    f"Profitability test matrix     : "
    f"{sparse_memory_mb(X_profit_test_sparse):,.2f} MB"
)

print("\nMATRIX DENSITY")
print("=" * 70)

print(
    f"Delivery training density      : "
    f"{matrix_density(X_delivery_train_sparse):.4%}"
)

print(
    f"Profitability training density : "
    f"{matrix_density(X_profit_train_sparse):.4%}"
)

print("\nNON-ZERO VALUES")
print("=" * 70)

print(
    f"Delivery      : {X_delivery_train_sparse.nnz:,}"
)

print(
    f"Profitability : {X_profit_train_sparse.nnz:,}"
)

SPARSE MATRIX MEMORY AUDIT
Delivery training matrix      : 293.93 MB
Delivery test matrix          : 3.46 MB
Profitability training matrix : 291.89 MB
Profitability test matrix     : 3.44 MB

MATRIX DENSITY
Delivery training density      : 2.0658%
Profitability training density : 2.0517%

NON-ZERO VALUES
Delivery      : 25,624,349
Profitability : 25,445,953


The sparse matrices were evaluated for memory consumption, density, and the number of stored non-zero values.

<h2 style="color: #548235; font-weight: 700;">Key Observations</h2>

- Delivery training matrix: **293.93 MB**
- Delivery future-test matrix: **3.46 MB**
- Profitability training matrix: **291.89 MB**
- Profitability future-test matrix: **3.44 MB**
- Delivery training density: **2.0658%**
- Profitability training density: **2.0517%**
- Delivery contains **25,624,349 non-zero values**.
- Profitability contains **25,445,953 non-zero values**.
- Approximately **98% of matrix cells are zero**, confirming a highly sparse feature representation.

<h2 style="color: #7F6000; font-weight: 700;">Modeling Implication</h2>

The sparse representation reduces the training matrix footprint from approximately **9.2 GB in dense form to below 300 MB**, making the encoded feature space substantially more computationally practical.

The low matrix density confirms that sparse storage is appropriate for the current high-dimensional one-hot encoded feature space.

<h2 style="color: #548235; font-weight: 700;">Status</h2>

**✅ SPARSE MEMORY VALIDATION PASSED — LOW DENSITY AND SUBSTANTIAL MEMORY REDUCTION CONFIRMED**

**✅ REPRESENTATION IS COMPUTATIONALLY PRACTICAL FOR DOWNSTREAM MODELING**

<h1 style="color: #1F4E78; font-weight: 700;">8.5.3 — Final Preprocessing Validation</h1>

The sparse-compatible preprocessing architecture has successfully transformed the leakage-controlled feature sets into computationally efficient model-ready matrices.

The final encoded datasets will now undergo a consolidated validation gate before Notebook 06A is considered complete.

The validation confirms that:

1. Both delivery and profitability matrices use sparse CSR representation.
2. Training and future-test matrices contain the expected number of observations.
3. Training and future-test matrices have identical encoded feature dimensions.
4. No observations were lost during preprocessing.
5. The sparse representation materially reduces memory consumption.
6. The same preprocessing structure is applied consistently to historical training data and the January 2018 future holdout.
7. The preprocessing stage remains leakage-safe because the transformers were fitted only on historical training data.

<h2 style="color: #1F4E78; font-weight: 700;">Final Modeling-Readiness Principle</h2>

The resulting matrices represent the final machine-learning input space for the current predictive analytics stage.

No additional feature transformation should be performed independently for individual models. All downstream models will consume these standardized, leakage-controlled representations to ensure fair and reproducible model comparison.

The final modeling stage can therefore proceed using the following artifacts:

- `X_delivery_train_sparse`
- `X_delivery_test_sparse`
- `y_delivery_train`
- `y_delivery_test`
- `X_profit_train_sparse`
- `X_profit_test_sparse`
- `y_profit_train`
- `y_profit_test`

The January 2018 dataset remains a strictly future holdout and will not be used for model fitting or preprocessing parameter estimation.

<h2 style="color: #1F4E78; font-weight: 700;">Final Validation Gate</h2>

In [65]:
# ============================================================
# Final Preprocessing Validation Gate
# ============================================================

from scipy import sparse

validation_results = {
    "Delivery sparse": sparse.issparse(X_delivery_train_sparse)
        and sparse.issparse(X_delivery_test_sparse),

    "Profitability sparse": sparse.issparse(X_profit_train_sparse)
        and sparse.issparse(X_profit_test_sparse),

    "Delivery dimensions match":
        X_delivery_train_sparse.shape[1] ==
        X_delivery_test_sparse.shape[1],

    "Profitability dimensions match":
        X_profit_train_sparse.shape[1] ==
        X_profit_test_sparse.shape[1],

    "Delivery training rows preserved":
        X_delivery_train_sparse.shape[0] ==
        len(y_delivery_train),

    "Delivery test rows preserved":
        X_delivery_test_sparse.shape[0] ==
        len(y_delivery_test),

    "Profitability training rows preserved":
        X_profit_train_sparse.shape[0] ==
        len(y_profit_train),

    "Profitability test rows preserved":
        X_profit_test_sparse.shape[0] ==
        len(y_profit_test),

    "Delivery encoded feature count":
        X_delivery_train_sparse.shape[1] == 6953,

    "Profitability encoded feature count":
        X_profit_train_sparse.shape[1] == 6952,
}

print("FINAL PREPROCESSING VALIDATION")
print("=" * 70)

for check, result in validation_results.items():
    print(f"{'✓' if result else '✗'} {check}: {result}")

all_passed = all(validation_results.values())

print("\n" + "=" * 70)

if all_passed:
    print("FINAL STATUS: PASS")
    print("Preprocessing is validated and ready for predictive modeling.")
else:
    print("FINAL STATUS: FAIL")
    print("Review failed validation checks before proceeding.")

FINAL PREPROCESSING VALIDATION
✓ Delivery sparse: True
✓ Profitability sparse: True
✓ Delivery dimensions match: True
✓ Profitability dimensions match: True
✓ Delivery training rows preserved: True
✓ Delivery test rows preserved: True
✓ Profitability training rows preserved: True
✓ Profitability test rows preserved: True
✓ Delivery encoded feature count: True
✓ Profitability encoded feature count: True

FINAL STATUS: PASS
Preprocessing is validated and ready for predictive modeling.


The final preprocessing validation gate confirms that both predictive modeling datasets are correctly transformed, dimensionally consistent, memory-efficient, and ready for downstream machine learning.

<h2 style="color: #548235; font-weight: 700;">Key Observations</h2>

- Delivery and Profitability matrices are confirmed as **sparse representations**.
- Delivery train/test encoded dimensions are aligned at **6,953 features**.
- Profitability train/test encoded dimensions are aligned at **6,952 features**.
- All training and future-test observations were preserved.
- Expected training and test row counts were maintained.
- No validation checks failed.
- The sparse representation substantially reduces memory requirements compared with the previous dense representation.
- Preprocessing was fitted only on the **2015–2017 historical training period**.
- The **January 2018 future holdout** remains isolated from preprocessing parameter estimation and model fitting.

<h2 style="color: #7F6000; font-weight: 700;">Modeling Readiness</h2>

The final machine-learning input artifacts are now established:

- `X_delivery_train_sparse`
- `X_delivery_test_sparse`
- `y_delivery_train`
- `y_delivery_test`
- `X_profit_train_sparse`
- `X_profit_test_sparse`
- `y_profit_train`
- `y_profit_test`

These standardized, leakage-controlled representations will be used consistently across downstream predictive models to support fair and reproducible model comparison.

<h2 style="color: #548235; font-weight: 700;">Final Validation Gate</h2>

**✅ FINAL STATUS: PASS**

**Preprocessing is validated and ready for predictive modeling.**

No additional independent feature transformations should be introduced at the individual model stage unless explicitly required and consistently validated.

<hr>

<h1 style="color: #1F4E78; font-weight: 700;">9. Export Modeling Artifacts</h1>

The modeling preparation stage has now been fully validated.

The final modeling artifacts will be exported so that subsequent predictive modeling notebooks can consume the exact same datasets without repeating feature engineering, leakage assessment, splitting, or preprocessing.

Because the encoded feature matrices are high-dimensional and sparse, they will be stored using sparse matrix-compatible `.npz` files rather than dense CSV files.

The following artifacts will be retained:

<h2 style="color: #1F4E78; font-weight: 700;">Delivery Delay Prediction</h2>

- Sparse training feature matrix
- Sparse future-test feature matrix
- Training target
- Future-test target
- Encoded feature names

<h2 style="color: #1F4E78; font-weight: 700;">Profitability Risk Prediction</h2>

- Sparse training feature matrix
- Sparse future-test feature matrix
- Training target
- Future-test target
- Encoded feature names

The chronological train/test split and preprocessing configuration established in this notebook remain unchanged.

These exported artifacts form the controlled modeling interface for Notebook 06B and subsequent predictive analytics notebooks.

<h2 style="color: #1F4E78; font-weight: 700;">Export Final Modeling Artifacts</h2>

In [68]:
# ============================================================
# Export Final Modeling Artifacts
# ============================================================

from pathlib import Path
import pandas as pd
from scipy import sparse

# ------------------------------------------------------------
# Output directory
# ------------------------------------------------------------

BASE_DIR = Path(
    r"C:\Users\chavi\Desktop\SupplySight AI – Enterprise Procurement Decision Intelligence Platform"
)

modeling_dir = (
    BASE_DIR
    / "Datasets"
    / "Modeling Artifacts"
)

modeling_dir.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# Export sparse matrices
# ------------------------------------------------------------

sparse.save_npz(
    modeling_dir / "delivery_train_sparse.npz",
    X_delivery_train_sparse
)

sparse.save_npz(
    modeling_dir / "delivery_test_sparse.npz",
    X_delivery_test_sparse
)

sparse.save_npz(
    modeling_dir / "profitability_train_sparse.npz",
    X_profit_train_sparse
)

sparse.save_npz(
    modeling_dir / "profitability_test_sparse.npz",
    X_profit_test_sparse
)

# ------------------------------------------------------------
# Export targets
# ------------------------------------------------------------

pd.DataFrame({
    "is_delayed": y_delivery_train
}).to_csv(
    modeling_dir / "delivery_train_target.csv",
    index=False
)

pd.DataFrame({
    "is_delayed": y_delivery_test
}).to_csv(
    modeling_dir / "delivery_test_target.csv",
    index=False
)

pd.DataFrame({
    "negative_profit_flag": y_profit_train
}).to_csv(
    modeling_dir / "profitability_train_target.csv",
    index=False
)

pd.DataFrame({
    "negative_profit_flag": y_profit_test
}).to_csv(
    modeling_dir / "profitability_test_target.csv",
    index=False
)

# ------------------------------------------------------------
# Export encoded feature names
# ------------------------------------------------------------

delivery_feature_names = (
    delivery_preprocessor_sparse
    .get_feature_names_out()
)

profitability_feature_names = (
    profitability_preprocessor_sparse
    .get_feature_names_out()
)

pd.DataFrame({
    "feature_index": range(len(delivery_feature_names)),
    "feature_name": delivery_feature_names
}).to_csv(
    modeling_dir / "delivery_encoded_feature_names.csv",
    index=False
)

pd.DataFrame({
    "feature_index": range(len(profitability_feature_names)),
    "feature_name": profitability_feature_names
}).to_csv(
    modeling_dir / "profitability_encoded_feature_names.csv",
    index=False
)

# ------------------------------------------------------------
# Export summary
# ------------------------------------------------------------

print("MODELING ARTIFACT EXPORT")
print("=" * 70)

print(f"Output directory: {modeling_dir.resolve()}")
print()

for file in sorted(modeling_dir.iterdir()):
    print(f"✓ {file.name}")

print("\nExport completed successfully.")

MODELING ARTIFACT EXPORT
Output directory: C:\Users\chavi\Desktop\SupplySight AI – Enterprise Procurement Decision Intelligence Platform\Datasets\Modeling Artifacts

✓ delivery_encoded_feature_names.csv
✓ delivery_test_sparse.npz
✓ delivery_test_target.csv
✓ delivery_train_sparse.npz
✓ delivery_train_target.csv
✓ profitability_encoded_feature_names.csv
✓ profitability_test_sparse.npz
✓ profitability_test_target.csv
✓ profitability_train_sparse.npz
✓ profitability_train_target.csv

Export completed successfully.


<h2 style="color: #1F4E78; font-weight: 700;">Export Validation</h2>

In [69]:
# ============================================================
# Export Validation
# ============================================================

expected_files = [
    "delivery_train_sparse.npz",
    "delivery_test_sparse.npz",
    "profitability_train_sparse.npz",
    "profitability_test_sparse.npz",
    "delivery_train_target.csv",
    "delivery_test_target.csv",
    "profitability_train_target.csv",
    "profitability_test_target.csv",
    "delivery_encoded_feature_names.csv",
    "profitability_encoded_feature_names.csv"
]

print("EXPORTED ARTIFACT VALIDATION")
print("=" * 70)

export_results = {}

for filename in expected_files:
    path = modeling_dir / filename
    export_results[filename] = path.exists()
    print(f"{'✓' if path.exists() else '✗'} {filename}")

print("\n" + "=" * 70)

if all(export_results.values()):
    print("EXPORT STATUS: PASS")
else:
    print("EXPORT STATUS: FAIL")

EXPORTED ARTIFACT VALIDATION
✓ delivery_train_sparse.npz
✓ delivery_test_sparse.npz
✓ profitability_train_sparse.npz
✓ profitability_test_sparse.npz
✓ delivery_train_target.csv
✓ delivery_test_target.csv
✓ profitability_train_target.csv
✓ profitability_test_target.csv
✓ delivery_encoded_feature_names.csv
✓ profitability_encoded_feature_names.csv

EXPORT STATUS: PASS


The modeling preparation stage has been fully validated, and the final modeling artifacts have been exported for reuse in subsequent predictive analytics notebooks.

<h2 style="color: #548235; font-weight: 700;">Key Observations</h2>

- Delivery Delay artifacts were successfully exported:
  - Sparse training feature matrix
  - Sparse future-test feature matrix
  - Training and future-test targets
  - Encoded feature names
- Profitability Risk artifacts were successfully exported:
  - Sparse training feature matrix
  - Sparse future-test feature matrix
  - Training and future-test targets
  - Encoded feature names
- High-dimensional feature matrices are stored as **sparse `.npz` files** to preserve memory efficiency.
- Target datasets and encoded feature names are stored separately as **CSV files**.
- All **10 expected modeling artifacts** were successfully detected in the modeling directory.
- The chronological train/test split and leakage-safe preprocessing configuration remain unchanged.

<h2 style="color: #7F6000; font-weight: 700;">Modeling Interface</h2>

The exported artifacts provide a controlled and reproducible interface for **Notebook 06B – Delivery Delay Prediction** and subsequent predictive modeling notebooks.

Downstream notebooks can consume these artifacts directly without repeating feature engineering, leakage assessment, temporal splitting, or preprocessing.

<h2 style="color: #548235; font-weight: 700;">Export Validation</h2>

**✅ EXPORT STATUS: PASS**

**10/10 expected modeling artifacts successfully exported and validated.**

<hr>

<h1 style="color: #1F4E78; font-weight: 700;">Notebook Conclusion — Advanced Analytics Preparation</h1>

Notebook 06A successfully transformed the validated enterprise feature store into a **leakage-controlled, time-aware, and computationally efficient foundation for predictive analytics**.

<h2 style="color: #548235; font-weight: 700;">Final Outcomes</h2>

**Dataset**
- **180,519** procurement transactions
- **190** feature-store columns
- Transaction-level analytical grain preserved
- Complete modeling dataset validated

**Prediction Targets**
- **Delivery Delay Prediction:** `is_delayed`
  - Delayed transactions: **103,400**
  - Delay rate: **57.28%**
- **Profitability Risk Prediction:** `negative_profit_flag`
  - Negative-profit transactions: **33,784**
  - Negative-profit rate: **18.71%**

**Leakage Control**
- Outcome-derived variables were systematically excluded from predictor sets.
- Delivery models exclude realized shipping outcomes such as `actual_shipping_days`, `shipping_delay_days`, `shipping_performance`, and `shipping_datetime`.
- Profitability models exclude realized financial outcomes such as `net_profit`, `profit_margin`, `profitability_category`, and `profit_per_unit`.
- Additional conservative exclusions were applied to prevent post-outcome information from entering the predictors.

**Time-Aware Validation**
- Training period: **2015–2017**
- Future holdout: **January 2018**
- The future holdout remained isolated from preprocessing fitting and model development.

**Feature Engineering & Preprocessing**
- Delivery raw predictors: **153**
- Profitability raw predictors: **152**
- Delivery encoded features: **6,953**
- Profitability encoded features: **6,952**
- Categorical variables were one-hot encoded with unknown-category handling.
- Numerical variables were imputed and scaled using training data only.
- Final encoded matrices use **sparse CSR representation**.

<h2 style="color: #7F6000; font-weight: 700;">Computational Optimization</h2>

The initial dense representation required approximately **9.46 GB per training matrix**.

The preprocessing architecture was therefore redesigned using sparse CSR matrices.

Final training memory usage:
- Delivery: **293.93 MB**
- Profitability: **291.89 MB**

This substantially reduced memory consumption while preserving the complete encoded feature space.

<h2 style="color: #548235; font-weight: 700;">Final Validation Status</h2>

All major preprocessing and modeling-readiness checks passed:

- **Sparse representation:** PASS
- **Train/test feature dimensions:** PASS
- **Training row preservation:** PASS
- **Test row preservation:** PASS
- **Encoded feature counts:** PASS
- **Leakage-controlled feature construction:** PASS
- **Time-aware split:** PASS
- **Training-only preprocessing:** PASS
- **Modeling artifact export:** PASS

<h2 style="color: #7F6000; font-weight: 700;">Notebook Status</h2>

**✅ 06A — ADVANCED ANALYTICS PREPARATION & TARGET ENGINEERING: COMPLETE**

The predictive modeling foundation is now validated, reproducible, leakage-controlled, and ready for model development.

<h2 style="color: #1F4E78; font-weight: 700;">Transition to Notebook 06B</h2>

The next stage moves from **model preparation** to **predictive modeling**.

**Notebook 06B — Delivery Delay Prediction**

The core business question is:

> **“Given the information available at the time of prediction, can we identify orders that are likely to be delayed?”**

Notebook 06B will compare multiple predictive approaches, evaluate classification performance using appropriate metrics, analyze prediction errors, and assess whether **external intelligence provides incremental predictive value beyond internal procurement information**.